In [1]:
# S2 Redo — Cell 1: Self-contained sequential setup for Architecture B + LoRA.

import json
import torch
from pathlib import Path
from collections.abc import Mapping

# ============================================================
# 1. Define paths and create an isolated workspace
# ============================================================
S2B_REDO_INPUT_ROOT = Path("/kaggle/input")
S2B_REDO_WORK_ROOT = Path("/kaggle/working")

S2B_REDO_OUTPUT_DIR = (
    S2B_REDO_WORK_ROOT / "S2_Architecture_B_Redo"
)
S2B_REDO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2B_REDO_DOWNLOAD_DIR = (
    S2B_REDO_WORK_ROOT / "phase2_lner"
)
S2B_REDO_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 76)
print("S2 REDO — ARCHITECTURE B + LoRA")
print("=" * 76)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2,
        ),
        "GiB",
    )

print("Output directory:", S2B_REDO_OUTPUT_DIR)

# ============================================================
# 2. Locate Architecture B and inspect its metadata
# ============================================================
S2B_REDO_METADATA_CANDIDATES = list(
    S2B_REDO_INPUT_ROOT.rglob("model_B_metadata.json")
)

if len(S2B_REDO_METADATA_CANDIDATES) != 1:
    raise FileNotFoundError(
        "Expected exactly one model_B_metadata.json under /kaggle/input, "
        f"but found {len(S2B_REDO_METADATA_CANDIDATES)}. "
        "Check that the Arch_Data_1 dataset is attached to the notebook."
    )

S2B_REDO_METADATA_PATH = S2B_REDO_METADATA_CANDIDATES[0]
S2B_REDO_ARCH_ROOT = S2B_REDO_METADATA_PATH.parent

with S2B_REDO_METADATA_PATH.open("r", encoding="utf-8") as f:
    S2B_REDO_METADATA = json.load(f)

S2B_REDO_MODEL_CONFIG = S2B_REDO_METADATA.get("model_config", {})

if S2B_REDO_METADATA.get("model_type") != "encoder-decoder-transformer":
    raise ValueError(
        "The discovered Architecture B metadata does not identify the "
        "expected encoder-decoder Transformer."
    )

S2B_REDO_FINAL_CHECKPOINT = (
    S2B_REDO_ARCH_ROOT / "model_B_final.pt"
)
S2B_REDO_STATE_DICT_PATH = (
    S2B_REDO_ARCH_ROOT / "model_B_state_dict.pt"
)
S2B_REDO_BEST_CHECKPOINT = (
    S2B_REDO_ARCH_ROOT / "checkpoints" / "best.pt"
)
S2B_REDO_LATEST_CHECKPOINT = (
    S2B_REDO_ARCH_ROOT / "checkpoints" / "latest.pt"
)

S2B_REDO_CHECKPOINT_CANDIDATES = [
    p for p in [
        S2B_REDO_STATE_DICT_PATH,
        S2B_REDO_BEST_CHECKPOINT,
        S2B_REDO_FINAL_CHECKPOINT,
    ]
    if p.is_file()
]

if not S2B_REDO_CHECKPOINT_CANDIDATES:
    raise FileNotFoundError(
        f"No usable Architecture B checkpoint found under {S2B_REDO_ARCH_ROOT}"
    )

print("\n" + "=" * 76)
print("ARCHITECTURE B VERIFIED")
print("=" * 76)
print("Architecture root:", S2B_REDO_ARCH_ROOT)
print("Model type:", S2B_REDO_METADATA["model_type"])
print("Parameter count:", f'{S2B_REDO_METADATA["parameter_count"]:,}')
print("Model configuration:")
print(json.dumps(S2B_REDO_MODEL_CONFIG, indent=2))

print("\nAvailable checkpoint files:")
for checkpoint_path in [
    S2B_REDO_FINAL_CHECKPOINT,
    S2B_REDO_STATE_DICT_PATH,
    S2B_REDO_BEST_CHECKPOINT,
    S2B_REDO_LATEST_CHECKPOINT,
]:
    if checkpoint_path.is_file():
        print(
            f"  FOUND  {checkpoint_path}"
            f"  ({checkpoint_path.stat().st_size / 1024**2:.2f} MiB)"
        )
    else:
        print(f"  absent {checkpoint_path}")

# ============================================================
# 3. Locate the exact Phase 1 tokenizer
# ============================================================
S2B_REDO_TOKENIZER_CANDIDATES = [
    p for p in S2B_REDO_INPUT_ROOT.rglob("tokenizer.json")
    if p.parent.name.lower() == "tokenizer"
    and "cs781-phase1-tokenized-corpus" in str(p).lower()
]

if len(S2B_REDO_TOKENIZER_CANDIDATES) != 1:
    raise FileNotFoundError(
        "Expected exactly one tokenizer.json inside the "
        "cs781-phase1-tokenized-corpus dataset, but found "
        f"{len(S2B_REDO_TOKENIZER_CANDIDATES)}."
    )

S2B_REDO_TOKENIZER_PATH = S2B_REDO_TOKENIZER_CANDIDATES[0]

print("\nTokenizer:", S2B_REDO_TOKENIZER_PATH)
print("Tokenizer size:", f"{S2B_REDO_TOKENIZER_PATH.stat().st_size:,} bytes")

# ============================================================
# 4. Find an existing complete Phase 2 corpus, if available
# ============================================================
S2B_REDO_REQUIRED_FILES = [
    Path("Train/fold1-text.json"),
    Path("Train/fold1-gold.json"),
    Path("Train/fold2-text.json"),
    Path("Train/fold2-gold.json"),
    Path("Train/labels.txt"),
    Path("Test/fold3-text.json"),
]

def s2b_redo_is_complete_dataset(root):
    """Return True only when every required Phase 2 file exists."""
    return (
        root.is_dir()
        and all((root / relative_path).is_file()
                for relative_path in S2B_REDO_REQUIRED_FILES)
    )

S2B_REDO_EXPECTED_DATASET_ROOT = (
    S2B_REDO_DOWNLOAD_DIR / "Phase_2_LNER"
)

S2B_REDO_PHASE2_ROOT = None

if s2b_redo_is_complete_dataset(S2B_REDO_EXPECTED_DATASET_ROOT):
    S2B_REDO_PHASE2_ROOT = S2B_REDO_EXPECTED_DATASET_ROOT.resolve()

# Search mounted datasets only if the expected working-directory location
# does not already contain a complete corpus.
if S2B_REDO_PHASE2_ROOT is None:
    S2B_REDO_MOUNTED_DATASET_ROOTS = []

    for text_path in S2B_REDO_INPUT_ROOT.rglob("fold1-text.json"):
        if text_path.parent.name.lower() != "train":
            continue

        possible_root = text_path.parent.parent

        if s2b_redo_is_complete_dataset(possible_root):
            S2B_REDO_MOUNTED_DATASET_ROOTS.append(
                possible_root.resolve()
            )

    S2B_REDO_MOUNTED_DATASET_ROOTS = list(dict.fromkeys(
        S2B_REDO_MOUNTED_DATASET_ROOTS
    ))

    if len(S2B_REDO_MOUNTED_DATASET_ROOTS) == 1:
        S2B_REDO_PHASE2_ROOT = S2B_REDO_MOUNTED_DATASET_ROOTS[0]

    elif len(S2B_REDO_MOUNTED_DATASET_ROOTS) > 1:
        raise RuntimeError(
            "Multiple complete Phase 2 dataset folders were found: "
            f"{S2B_REDO_MOUNTED_DATASET_ROOTS}. "
            "Please remove the ambiguity before proceeding."
        )

# ============================================================
# 5. Download Phase 2 in this same cell when it is not available
# ============================================================
if S2B_REDO_PHASE2_ROOT is None:
    print("\nPhase 2 corpus not found locally. Starting authenticated download.")

    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import snapshot_download
    except ImportError as exc:
        raise ImportError(
            "This cell requires kaggle_secrets and huggingface_hub. "
            "Enable the standard Kaggle notebook environment and retry."
        ) from exc

    try:
        S2B_REDO_HF_TOKEN = UserSecretsClient().get_secret("HF")
    except Exception as exc:
        raise RuntimeError(
            "Cannot retrieve Kaggle Secret 'HF'. Add or enable the secret "
            "named HF in the notebook settings, then rerun Cell 1."
        ) from exc

    if not S2B_REDO_HF_TOKEN:
        raise RuntimeError(
            "Kaggle Secret 'HF' is empty. Configure it before rerunning Cell 1."
        )

    S2B_REDO_REPOSITORY_PATH = snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        token=S2B_REDO_HF_TOKEN,
        allow_patterns=["Phase_2_LNER/**"],
        local_dir=str(S2B_REDO_DOWNLOAD_DIR),
    )

    print("Downloaded repository snapshot:", S2B_REDO_REPOSITORY_PATH)

    # Resolve the dataset from the expected directory or a nested download path.
    S2B_REDO_POSTDOWNLOAD_CANDIDATES = [
        S2B_REDO_EXPECTED_DATASET_ROOT,
    ]

    for text_path in S2B_REDO_DOWNLOAD_DIR.rglob("fold1-text.json"):
        if text_path.parent.name.lower() == "train":
            S2B_REDO_POSTDOWNLOAD_CANDIDATES.append(
                text_path.parent.parent
            )

    S2B_REDO_POSTDOWNLOAD_CANDIDATES = list(dict.fromkeys(
        p.resolve()
        for p in S2B_REDO_POSTDOWNLOAD_CANDIDATES
        if s2b_redo_is_complete_dataset(p)
    ))

    if len(S2B_REDO_POSTDOWNLOAD_CANDIDATES) != 1:
        raise RuntimeError(
            "The download completed, but exactly one complete "
            "Phase_2_LNER folder could not be resolved. "
            f"Candidates: {S2B_REDO_POSTDOWNLOAD_CANDIDATES}"
        )

    S2B_REDO_PHASE2_ROOT = S2B_REDO_POSTDOWNLOAD_CANDIDATES[0]

if not s2b_redo_is_complete_dataset(S2B_REDO_PHASE2_ROOT):
    raise FileNotFoundError(
        f"Phase 2 corpus verification failed: {S2B_REDO_PHASE2_ROOT}"
    )

print("\n" + "=" * 76)
print("PHASE 2 DATASET VERIFIED")
print("=" * 76)
print("Dataset root:", S2B_REDO_PHASE2_ROOT)

for relative_path in S2B_REDO_REQUIRED_FILES:
    file_path = S2B_REDO_PHASE2_ROOT / relative_path
    print(
        f"  OK  {relative_path}"
        f"  ({file_path.stat().st_size:,} bytes)"
    )

# ============================================================
# 6. Load the dataset JSON files and check document IDs
# ============================================================
S2B_REDO_TRAIN_DIR = S2B_REDO_PHASE2_ROOT / "Train"
S2B_REDO_TEST_DIR = S2B_REDO_PHASE2_ROOT / "Test"

def s2b_redo_load_json(path):
    """Read one UTF-8 JSON file with a useful file-specific error."""
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        raise RuntimeError(f"Failed to load JSON file: {path}") from exc

S2B_REDO_FOLD1_TEXT = s2b_redo_load_json(
    S2B_REDO_TRAIN_DIR / "fold1-text.json"
)
S2B_REDO_FOLD1_GOLD = s2b_redo_load_json(
    S2B_REDO_TRAIN_DIR / "fold1-gold.json"
)
S2B_REDO_FOLD2_TEXT = s2b_redo_load_json(
    S2B_REDO_TRAIN_DIR / "fold2-text.json"
)
S2B_REDO_FOLD2_GOLD = s2b_redo_load_json(
    S2B_REDO_TRAIN_DIR / "fold2-gold.json"
)
S2B_REDO_FOLD3_TEXT = s2b_redo_load_json(
    S2B_REDO_TEST_DIR / "fold3-text.json"
)

S2B_REDO_LABELS_PATH = S2B_REDO_TRAIN_DIR / "labels.txt"

with S2B_REDO_LABELS_PATH.open("r", encoding="utf-8") as f:
    S2B_REDO_LABELS = [
        line.strip()
        for line in f
        if line.strip()
    ]

for dataset_name, mapping in [
    ("fold1-text", S2B_REDO_FOLD1_TEXT),
    ("fold1-gold", S2B_REDO_FOLD1_GOLD),
    ("fold2-text", S2B_REDO_FOLD2_TEXT),
    ("fold2-gold", S2B_REDO_FOLD2_GOLD),
    ("fold3-text", S2B_REDO_FOLD3_TEXT),
]:
    if not isinstance(mapping, Mapping):
        raise TypeError(
            f"{dataset_name} must be a mapping from document ID to content; "
            f"found {type(mapping).__name__}."
        )

if set(S2B_REDO_FOLD1_TEXT) != set(S2B_REDO_FOLD1_GOLD):
    raise ValueError("Fold 1 text/gold document IDs do not match.")

if set(S2B_REDO_FOLD2_TEXT) != set(S2B_REDO_FOLD2_GOLD):
    raise ValueError("Fold 2 text/gold document IDs do not match.")

if len(S2B_REDO_LABELS) != len(set(S2B_REDO_LABELS)):
    raise ValueError("Duplicate entity labels found in labels.txt.")

if not S2B_REDO_LABELS:
    raise ValueError("The Phase 2 label inventory is empty.")

for fold_name, texts in [
    ("Fold 1", S2B_REDO_FOLD1_TEXT),
    ("Fold 2", S2B_REDO_FOLD2_TEXT),
    ("Fold 3", S2B_REDO_FOLD3_TEXT),
]:
    invalid_text_ids = [
        doc_id for doc_id, text_value in texts.items()
        if not isinstance(text_value, str)
    ]

    if invalid_text_ids:
        raise TypeError(
            f"{fold_name} contains non-string document text. "
            f"Examples: {invalid_text_ids[:5]}"
        )

print("\n" + "=" * 76)
print("PHASE 2 DATA AUDIT")
print("=" * 76)
print("Fold 1 documents:", len(S2B_REDO_FOLD1_TEXT))
print("Fold 2 documents:", len(S2B_REDO_FOLD2_TEXT))
print("Fold 3 test documents:", len(S2B_REDO_FOLD3_TEXT))
print("Entity labels:", len(S2B_REDO_LABELS))
print("Labels:", S2B_REDO_LABELS)
print("Fold 1 text/gold IDs: matched")
print("Fold 2 text/gold IDs: matched")
print("All document text values: verified as strings")

# ============================================================
# 7. Save a setup manifest for reproducibility
# ============================================================
S2B_REDO_MANIFEST = {
    "experiment": "S2 Architecture B LoRA redo",
    "phase1_architecture": S2B_REDO_METADATA.get("architecture_name"),
    "architecture_root": str(S2B_REDO_ARCH_ROOT),
    "metadata_path": str(S2B_REDO_METADATA_PATH),
    "model_config": S2B_REDO_MODEL_CONFIG,
    "checkpoint_candidates": [
        str(p) for p in S2B_REDO_CHECKPOINT_CANDIDATES
    ],
    "tokenizer_path": str(S2B_REDO_TOKENIZER_PATH),
    "phase2_dataset_root": str(S2B_REDO_PHASE2_ROOT),
    "fold1_document_count": len(S2B_REDO_FOLD1_TEXT),
    "fold2_document_count": len(S2B_REDO_FOLD2_TEXT),
    "fold3_document_count": len(S2B_REDO_FOLD3_TEXT),
    "labels": S2B_REDO_LABELS,
    "method": "Architecture B + LoRA",
    "uses_architecture_c_weights": False,
}

S2B_REDO_MANIFEST_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_redo_setup_manifest.json"
)

with S2B_REDO_MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        S2B_REDO_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("\nSetup manifest:", S2B_REDO_MANIFEST_PATH)
print("\nCell 1 completed successfully.")
print("Architecture B + LoRA workspace is initialized.")
print("No other model weights were loaded or modified.")

S2 REDO — ARCHITECTURE B + LoRA
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB
Output directory: /kaggle/working/S2_Architecture_B_Redo

ARCHITECTURE B VERIFIED
Architecture root: /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B
Model type: encoder-decoder-transformer
Parameter count: 139,263,824
Model configuration:
{
  "vocab_size": 50000,
  "src_context_length": 1024,
  "tgt_context_length": 1024,
  "d_model": 768,
  "n_encoder_layers": 6,
  "n_decoder_layers": 6,
  "n_heads": 12,
  "d_ff": 3072,
  "dropout": 0.1,
  "activation": "gelu",
  "normalization": "layernorm",
  "pre_norm": true,
  "learned_positions": true,
  "tie_embeddings": true,
  "bias": true
}

Available checkpoint files:
  FOUND  /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B/model_B_final.pt  (531.33 MiB)
  FOUND  /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B/model_B_state_dict.pt  (531.33 MiB)
  FOUND  /kaggle/input/datasets/rohit

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Downloaded repository snapshot: /kaggle/working/phase2_lner

PHASE 2 DATASET VERIFIED
Dataset root: /kaggle/working/phase2_lner/Phase_2_LNER
  OK  Train/fold1-text.json  (1,106,635 bytes)
  OK  Train/fold1-gold.json  (460,581 bytes)
  OK  Train/fold2-text.json  (1,124,295 bytes)
  OK  Train/fold2-gold.json  (475,489 bytes)
  OK  Train/labels.txt  (71 bytes)
  OK  Test/fold3-text.json  (1,186,648 bytes)

PHASE 2 DATA AUDIT
Fold 1 documents: 35
Fold 2 documents: 35
Fold 3 test documents: 35
Entity labels: 12
Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1 text/gold IDs: matched
Fold 2 text/gold IDs: matched
All document text values: verified as strings

Setup manifest: /kaggle/working/S2_Architecture_B_Redo/S2B_redo_setup_manifest.json

Cell 1 completed successfully.
Architecture B + LoRA workspace is initialized.
No other model weights were loaded or modified.


In [2]:
# S2 Redo — Cell 2: Load and inspect Architecture B state-dict structure.

import torch
from collections import Counter
from collections.abc import Mapping

# ------------------------------------------------------------
# 1. Verify the expected inputs from Cell 1
# ------------------------------------------------------------
if "S2B_REDO_STATE_DICT_PATH" not in globals():
    raise RuntimeError("Run Cell 1 first: Architecture B paths are missing.")

if not S2B_REDO_STATE_DICT_PATH.is_file():
    raise FileNotFoundError(
        f"Architecture B state dict not found: {S2B_REDO_STATE_DICT_PATH}"
    )

print("=" * 76)
print("S2 REDO — ARCHITECTURE B CHECKPOINT AUDIT")
print("=" * 76)
print("Checkpoint:", S2B_REDO_STATE_DICT_PATH)
print("Device for inspection: CPU")

# ------------------------------------------------------------
# 2. Load the selected checkpoint once
# ------------------------------------------------------------
try:
    S2B_REDO_RAW_STATE = torch.load(
        S2B_REDO_STATE_DICT_PATH,
        map_location="cpu",
        weights_only=True,
    )
except TypeError:
    # Compatibility fallback for a PyTorch version without weights_only.
    S2B_REDO_RAW_STATE = torch.load(
        S2B_REDO_STATE_DICT_PATH,
        map_location="cpu",
    )

if not isinstance(S2B_REDO_RAW_STATE, Mapping):
    raise TypeError(
        "Expected the Architecture B checkpoint to contain a mapping, "
        f"found {type(S2B_REDO_RAW_STATE).__name__}."
    )

# ------------------------------------------------------------
# 3. Resolve a plain state dict or a common checkpoint wrapper
# ------------------------------------------------------------
S2B_REDO_STATE_CANDIDATES = [
    "state_dict",
    "model_state_dict",
    "model",
    "weights",
    "model_state",
]

def s2b_redo_is_tensor_state_dict(obj):
    """Identify mappings whose values are predominantly tensors."""
    if not isinstance(obj, Mapping) or not obj:
        return False

    tensor_values = [
        value for value in obj.values()
        if isinstance(value, torch.Tensor)
    ]

    return (
        len(tensor_values) > 0
        and len(tensor_values) / len(obj) >= 0.80
    )

S2B_REDO_STATE_DICT = None
S2B_REDO_STATE_DICT_SOURCE = None

if s2b_redo_is_tensor_state_dict(S2B_REDO_RAW_STATE):
    S2B_REDO_STATE_DICT = S2B_REDO_RAW_STATE
    S2B_REDO_STATE_DICT_SOURCE = "checkpoint root"
else:
    for candidate_name in S2B_REDO_STATE_CANDIDATES:
        candidate_value = S2B_REDO_RAW_STATE.get(candidate_name)

        if s2b_redo_is_tensor_state_dict(candidate_value):
            S2B_REDO_STATE_DICT = candidate_value
            S2B_REDO_STATE_DICT_SOURCE = candidate_name
            break

if S2B_REDO_STATE_DICT is None:
    print("Top-level checkpoint entries:")
    for key, value in S2B_REDO_RAW_STATE.items():
        if isinstance(value, Mapping):
            print(f"  {key}: mapping with {len(value)} entries")
        elif isinstance(value, torch.Tensor):
            print(f"  {key}: tensor {tuple(value.shape)}")
        else:
            print(f"  {key}: {type(value).__name__}")

    raise RuntimeError(
        "Could not resolve a tensor state dict from this checkpoint. "
        "Inspect the displayed top-level entries before proceeding."
    )

# ------------------------------------------------------------
# 4. Validate tensors and parameter count
# ------------------------------------------------------------
S2B_REDO_NON_TENSOR_KEYS = [
    key for key, value in S2B_REDO_STATE_DICT.items()
    if not isinstance(value, torch.Tensor)
]

if S2B_REDO_NON_TENSOR_KEYS:
    raise TypeError(
        "Unexpected non-tensor entries in the resolved state dict: "
        f"{S2B_REDO_NON_TENSOR_KEYS[:10]}"
    )

S2B_REDO_STATE_PARAM_COUNT = sum(
    tensor.numel()
    for tensor in S2B_REDO_STATE_DICT.values()
)

print("\nResolved state dict from:", S2B_REDO_STATE_DICT_SOURCE)
print("Number of tensor entries:", len(S2B_REDO_STATE_DICT))
print("Total parameters in state dict:", f"{S2B_REDO_STATE_PARAM_COUNT:,}")
print(
    "Parameter count in metadata:",
    f'{S2B_REDO_METADATA["parameter_count"]:,}',
)

if S2B_REDO_STATE_PARAM_COUNT != S2B_REDO_METADATA["parameter_count"]:
    print(
        "\nNote: state-dict tensor count does not exactly match metadata. "
        "This can happen if the metadata counts unique/tied parameters "
        "differently or the checkpoint includes extra tensors. "
        "We will investigate before model construction."
    )
else:
    print("Parameter count matches metadata: YES")

# ------------------------------------------------------------
# 5. Summarize tensor key namespaces
# ------------------------------------------------------------
S2B_REDO_PREFIX_COUNTS = Counter()

for key in S2B_REDO_STATE_DICT:
    parts = key.split(".")
    prefix = ".".join(parts[:2]) if len(parts) > 1 else key
    S2B_REDO_PREFIX_COUNTS[prefix] += 1

print("\nMost common key prefixes:")
for prefix, count in S2B_REDO_PREFIX_COUNTS.most_common(50):
    print(f"  {prefix:48s} {count:4d}")

# ------------------------------------------------------------
# 6. Print parameter names and shapes
# ------------------------------------------------------------
print("\n" + "=" * 76)
print("STATE-DICT PARAMETER NAMES AND SHAPES")
print("=" * 76)

for index, (key, tensor) in enumerate(S2B_REDO_STATE_DICT.items()):
    print(
        f"{index:04d} | {key:72s} | "
        f"shape={str(tuple(tensor.shape)):24s} | "
        f"dtype={tensor.dtype}"
    )

# ------------------------------------------------------------
# 7. Identify likely embedding, attention, normalization and FFN tensors
# ------------------------------------------------------------
S2B_REDO_KEYWORDS = [
    "embed",
    "position",
    "pos",
    "encoder",
    "decoder",
    "attn",
    "attention",
    "query",
    "key",
    "value",
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
    "qkv",
    "linear",
    "ff",
    "feed",
    "norm",
    "lm_head",
    "output",
]

print("\n" + "=" * 76)
print("ARCHITECTURE COMPONENT KEYWORD AUDIT")
print("=" * 76)

for keyword in S2B_REDO_KEYWORDS:
    matches = [
        key for key in S2B_REDO_STATE_DICT
        if keyword.lower() in key.lower()
    ]

    if matches:
        print(f"\n[{keyword}] — {len(matches)} matching tensors")
        for key in matches[:12]:
            print(f"  {key}: {tuple(S2B_REDO_STATE_DICT[key].shape)}")
        if len(matches) > 12:
            print(f"  ... and {len(matches) - 12} additional matches")

# ------------------------------------------------------------
# 8. Save a lightweight structural report for this experiment
# ------------------------------------------------------------
S2B_REDO_STATE_REPORT = {
    "checkpoint": str(S2B_REDO_STATE_DICT_PATH),
    "state_dict_source": S2B_REDO_STATE_DICT_SOURCE,
    "tensor_entry_count": len(S2B_REDO_STATE_DICT),
    "state_dict_parameter_count": int(S2B_REDO_STATE_PARAM_COUNT),
    "metadata_parameter_count": int(S2B_REDO_METADATA["parameter_count"]),
    "state_dict_keys": {
        key: {
            "shape": list(tensor.shape),
            "dtype": str(tensor.dtype),
        }
        for key, tensor in S2B_REDO_STATE_DICT.items()
    },
}

S2B_REDO_STATE_REPORT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_checkpoint_structure.json"
)

import json

with S2B_REDO_STATE_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        S2B_REDO_STATE_REPORT,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("\nStructural report saved:", S2B_REDO_STATE_REPORT_PATH)
print("\nCell 2 completed. No parameters were updated.")

S2 REDO — ARCHITECTURE B CHECKPOINT AUDIT
Checkpoint: /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B/model_B_state_dict.pt
Device for inspection: CPU

Resolved state dict from: checkpoint root
Number of tensor entries: 212
Total parameters in state dict: 139,263,824
Parameter count in metadata: 139,263,824
Parameter count matches metadata: YES

Most common key prefixes:
  decoder_layers.0                                   22
  decoder_layers.1                                   22
  decoder_layers.2                                   22
  decoder_layers.3                                   22
  decoder_layers.4                                   22
  decoder_layers.5                                   22
  encoder_layers.0                                   12
  encoder_layers.1                                   12
  encoder_layers.2                                   12
  encoder_layers.3                                   12
  encoder_layers.4                               

In [3]:
# S2 Redo — Cell 3: Build Architecture B encoder + LoRA and verify the model.

import gc
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Check sequential prerequisites from Cells 1 and 2
# ============================================================
required_variables = [
    "S2B_REDO_METADATA",
    "S2B_REDO_MODEL_CONFIG",
    "S2B_REDO_STATE_DICT",
    "S2B_REDO_TOKENIZER_PATH",
    "S2B_REDO_LABELS",
    "S2B_REDO_FOLD1_TEXT",
    "S2B_REDO_FOLD1_GOLD",
    "S2B_REDO_FOLD2_TEXT",
    "S2B_REDO_FOLD2_GOLD",
    "S2B_REDO_OUTPUT_DIR",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Missing prerequisite variables. Run Cells 1 and 2 in order. "
        f"Missing: {missing_variables}"
    )

# ============================================================
# 2. Define the document-level training/validation split
# ============================================================
overlapping_doc_ids = (
    set(S2B_REDO_FOLD1_TEXT) & set(S2B_REDO_FOLD2_TEXT)
)

if overlapping_doc_ids:
    raise ValueError(
        "Fold 1 and Fold 2 contain overlapping document IDs. "
        f"Examples: {list(overlapping_doc_ids)[:5]}"
    )

S2B_REDO_ALL_TEXTS = dict(S2B_REDO_FOLD1_TEXT)
S2B_REDO_ALL_TEXTS.update(S2B_REDO_FOLD2_TEXT)

S2B_REDO_ALL_GOLD = dict(S2B_REDO_FOLD1_GOLD)
S2B_REDO_ALL_GOLD.update(S2B_REDO_FOLD2_GOLD)

if set(S2B_REDO_ALL_TEXTS) != set(S2B_REDO_ALL_GOLD):
    raise ValueError("Combined training texts and gold annotations do not align.")

S2B_REDO_ALL_DOC_IDS = list(S2B_REDO_ALL_TEXTS.keys())

S2B_REDO_TRAIN_DOC_IDS, S2B_REDO_VAL_DOC_IDS = train_test_split(
    S2B_REDO_ALL_DOC_IDS,
    test_size=0.20,
    random_state=42,
)

S2B_REDO_TRAIN_TEXTS = {
    doc_id: S2B_REDO_ALL_TEXTS[doc_id]
    for doc_id in S2B_REDO_TRAIN_DOC_IDS
}
S2B_REDO_VAL_TEXTS = {
    doc_id: S2B_REDO_ALL_TEXTS[doc_id]
    for doc_id in S2B_REDO_VAL_DOC_IDS
}
S2B_REDO_TRAIN_GOLD = {
    doc_id: S2B_REDO_ALL_GOLD[doc_id]
    for doc_id in S2B_REDO_TRAIN_DOC_IDS
}
S2B_REDO_VAL_GOLD = {
    doc_id: S2B_REDO_ALL_GOLD[doc_id]
    for doc_id in S2B_REDO_VAL_DOC_IDS
}

if set(S2B_REDO_TRAIN_DOC_IDS) & set(S2B_REDO_VAL_DOC_IDS):
    raise RuntimeError("Train/validation document leakage detected.")

# Character vocabulary is learned only from training documents.
S2B_REDO_PAD_CHAR_ID = 0
S2B_REDO_UNK_CHAR_ID = 1

S2B_REDO_TRAIN_CHARACTERS = sorted(
    set(
        character
        for text_value in S2B_REDO_TRAIN_TEXTS.values()
        for character in text_value
    )
)

S2B_REDO_CHAR_TO_ID = {
    character: index + 2
    for index, character in enumerate(S2B_REDO_TRAIN_CHARACTERS)
}
S2B_REDO_CHAR_VOCAB_SIZE = len(S2B_REDO_CHAR_TO_ID) + 2

print("=" * 76)
print("S2 REDO — DOCUMENT SPLIT")
print("=" * 76)
print("All labelled documents:", len(S2B_REDO_ALL_DOC_IDS))
print("Training documents:", len(S2B_REDO_TRAIN_DOC_IDS))
print("Validation documents:", len(S2B_REDO_VAL_DOC_IDS))
print("Training-only character vocabulary:", S2B_REDO_CHAR_VOCAB_SIZE)
print("Train/validation document overlap: 0")

# ============================================================
# 3. Load the exact Phase 1 tokenizer and verify its vocabulary
# ============================================================
S2B_REDO_TOKENIZER = Tokenizer.from_file(
    str(S2B_REDO_TOKENIZER_PATH)
)

S2B_REDO_CONFIG_VOCAB = int(S2B_REDO_MODEL_CONFIG["vocab_size"])
S2B_REDO_TOKENIZER_VOCAB = int(
    S2B_REDO_TOKENIZER.get_vocab_size(with_added_tokens=True)
)

if S2B_REDO_TOKENIZER_VOCAB != S2B_REDO_CONFIG_VOCAB:
    raise ValueError(
        "Tokenizer/model vocabulary mismatch: "
        f"tokenizer={S2B_REDO_TOKENIZER_VOCAB}, "
        f"model={S2B_REDO_CONFIG_VOCAB}"
    )

print("\nPhase 1 tokenizer vocabulary:", S2B_REDO_TOKENIZER_VOCAB)

# ============================================================
# 4. Architecture B's encoder building blocks
# ============================================================
S2B_REDO_D_MODEL = int(S2B_REDO_MODEL_CONFIG["d_model"])
S2B_REDO_NUM_HEADS = int(S2B_REDO_MODEL_CONFIG["n_heads"])
S2B_REDO_NUM_ENCODER_LAYERS = int(
    S2B_REDO_MODEL_CONFIG["n_encoder_layers"]
)
S2B_REDO_FF_DIM = int(S2B_REDO_MODEL_CONFIG["d_ff"])
S2B_REDO_MAX_POSITION = int(
    S2B_REDO_MODEL_CONFIG["src_context_length"]
)
S2B_REDO_DROPOUT = float(S2B_REDO_MODEL_CONFIG["dropout"])

S2B_REDO_LORA_RANK = 8
S2B_REDO_LORA_ALPHA = 16.0


class S2BLoRAQKV(nn.Module):
    """Architecture B's fused QKV linear layer with a low-rank update."""

    def __init__(self, d_model, rank=8, alpha=16.0, bias=True):
        super().__init__()

        self.d_model = d_model
        self.rank = rank
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank

        # Names of these base tensors match Architecture B's checkpoint.
        self.weight = nn.Parameter(
            torch.empty(3 * d_model, d_model)
        )
        self.bias = (
            nn.Parameter(torch.empty(3 * d_model))
            if bias else None
        )

        # LoRA: delta_W = B @ A, initialized to zero contribution.
        self.lora_A = nn.Parameter(torch.empty(rank, d_model))
        self.lora_B = nn.Parameter(torch.zeros(3 * d_model, rank))

        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = d_model
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_output = F.linear(x, self.weight, self.bias)
        low_rank_output = F.linear(
            F.linear(x, self.lora_A),
            self.lora_B,
        )
        return base_output + self.scaling * low_rank_output


class S2BEncoderSelfAttention(nn.Module):
    """Bidirectional multi-head self-attention with fused QKV projections."""

    def __init__(self, d_model, num_heads, dropout, rank, alpha):
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads.")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout_probability = dropout

        self.qkv = S2BLoRAQKV(
            d_model=d_model,
            rank=rank,
            alpha=alpha,
            bias=True,
        )
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, attention_mask):
        batch_size, sequence_length, _ = x.shape

        qkv = self.qkv(x)
        query, key, value = qkv.chunk(3, dim=-1)

        def split_heads(tensor):
            return (
                tensor.reshape(
                    batch_size,
                    sequence_length,
                    self.num_heads,
                    self.head_dim,
                )
                .transpose(1, 2)
            )

        query = split_heads(query)
        key = split_heads(key)
        value = split_heads(value)

        # True means that a key position may participate in attention.
        key_mask = attention_mask[:, None, None, :].bool()

        attention_output = F.scaled_dot_product_attention(
            query,
            key,
            value,
            attn_mask=key_mask,
            dropout_p=(
                self.dropout_probability if self.training else 0.0
            ),
            is_causal=False,
        )

        attention_output = (
            attention_output.transpose(1, 2)
            .contiguous()
            .reshape(batch_size, sequence_length, self.d_model)
        )

        return self.out_proj(attention_output)


class S2BEncoderFFN(nn.Module):
    """Feed-forward block matching Architecture B's fc1/fc2 tensors."""

    def __init__(self, d_model, ff_dim, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)
        self.dropout_probability = dropout

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = F.dropout(
            x,
            p=self.dropout_probability,
            training=self.training,
        )
        return self.fc2(x)


class S2BEncoderLayer(nn.Module):
    """Pre-norm encoder layer with Architecture B parameter names."""

    def __init__(
        self,
        d_model,
        num_heads,
        ff_dim,
        dropout,
        rank,
        alpha,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.self_attn = S2BEncoderSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            rank=rank,
            alpha=alpha,
        )

        self.ffn = S2BEncoderFFN(
            d_model=d_model,
            ff_dim=ff_dim,
            dropout=dropout,
        )

        self.residual_dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask):
        x = x + self.residual_dropout(
            self.self_attn(self.norm1(x), attention_mask)
        )

        x = x + self.residual_dropout(
            self.ffn(self.norm2(x))
        )

        return x


class S2BArchitectureBLoRMAwareNER(nn.Module):
    """
    NER adaptation of Architecture B's Phase 1 encoder.
    The original decoder is not required for character-boundary prediction.
    """

    def __init__(
        self,
        vocab_size,
        max_position,
        d_model,
        num_layers,
        num_heads,
        ff_dim,
        dropout,
        char_vocab_size,
        num_labels,
        rank,
        alpha,
    ):
        super().__init__()

        self.d_model = d_model
        self.max_position = max_position

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.encoder_position_embedding = nn.Embedding(
            max_position, d_model
        )
        self.embedding_dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            S2BEncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout=dropout,
                rank=rank,
                alpha=alpha,
            )
            for _ in range(num_layers)
        ])

        self.encoder_final_norm = nn.LayerNorm(d_model)

        # Character-level boundary head for precise character offsets.
        self.char_embedding = nn.Embedding(
            char_vocab_size,
            32,
            padding_idx=S2B_REDO_PAD_CHAR_ID,
        )

        self.char_conv = nn.Sequential(
            nn.Conv1d(
                d_model + 32,
                128,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.start_head = nn.Conv1d(
            128, num_labels, kernel_size=1
        )
        self.end_head = nn.Conv1d(
            128, num_labels, kernel_size=1
        )

    def encode(self, input_ids, attention_mask):
        batch_size, sequence_length = input_ids.shape

        if sequence_length > self.max_position:
            raise ValueError(
                f"Sequence length {sequence_length} exceeds "
                f"Architecture B limit {self.max_position}."
            )

        position_ids = torch.arange(
            sequence_length,
            device=input_ids.device,
        )

        hidden = (
            self.token_embedding(input_ids)
            + self.encoder_position_embedding(position_ids)[None, :, :]
        )
        hidden = self.embedding_dropout(hidden)

        for layer in self.encoder_layers:
            hidden = layer(hidden, attention_mask)

        return self.encoder_final_norm(hidden)

    def forward(
        self,
        input_ids,
        attention_mask,
        char_ids,
        char_to_token,
    ):
        token_hidden = self.encode(input_ids, attention_mask)

        safe_token_indices = char_to_token.clamp(
            min=0,
            max=token_hidden.shape[1] - 1,
        )

        char_token_hidden = token_hidden.gather(
            dim=1,
            index=safe_token_indices.unsqueeze(-1).expand(
                -1,
                -1,
                token_hidden.shape[-1],
            ),
        )

        char_features = self.char_embedding(char_ids)

        combined = torch.cat(
            [char_token_hidden, char_features],
            dim=-1,
        )

        features = self.char_conv(combined.transpose(1, 2))

        start_logits = self.start_head(features).transpose(1, 2)
        end_logits = self.end_head(features).transpose(1, 2)

        return start_logits, end_logits


# ============================================================
# 5. Instantiate the Architecture B encoder + LoRA
# ============================================================
S2B_REDO_MODEL = S2BArchitectureBLoRMAwareNER(
    vocab_size=S2B_REDO_CONFIG_VOCAB,
    max_position=S2B_REDO_MAX_POSITION,
    d_model=S2B_REDO_D_MODEL,
    num_layers=S2B_REDO_NUM_ENCODER_LAYERS,
    num_heads=S2B_REDO_NUM_HEADS,
    ff_dim=S2B_REDO_FF_DIM,
    dropout=S2B_REDO_DROPOUT,
    char_vocab_size=S2B_REDO_CHAR_VOCAB_SIZE,
    num_labels=len(S2B_REDO_LABELS),
    rank=S2B_REDO_LORA_RANK,
    alpha=S2B_REDO_LORA_ALPHA,
)

# Load only Architecture B's original encoder and shared embedding tensors.
S2B_REDO_PRETRAINED_ENCODER_STATE = {
    key: tensor
    for key, tensor in S2B_REDO_STATE_DICT.items()
    if (
        key == "token_embedding.weight"
        or key == "encoder_position_embedding.weight"
        or key.startswith("encoder_layers.")
        or key.startswith("encoder_final_norm.")
    )
}

if not S2B_REDO_PRETRAINED_ENCODER_STATE:
    raise RuntimeError("No Architecture B encoder weights were selected.")

S2B_REDO_LOAD_RESULT = S2B_REDO_MODEL.load_state_dict(
    S2B_REDO_PRETRAINED_ENCODER_STATE,
    strict=False,
)

if S2B_REDO_LOAD_RESULT.unexpected_keys:
    raise RuntimeError(
        "Unexpected keys while loading the pretrained encoder: "
        f"{S2B_REDO_LOAD_RESULT.unexpected_keys[:10]}"
    )

S2B_REDO_LOADED_KEYS = set(S2B_REDO_PRETRAINED_ENCODER_STATE)
S2B_REDO_MODEL_KEYS = set(S2B_REDO_MODEL.state_dict())

if not S2B_REDO_LOADED_KEYS.issubset(S2B_REDO_MODEL_KEYS):
    raise RuntimeError("Not all selected Phase 1 encoder keys map into the model.")

# Freeze every parameter, then enable LoRA matrices and the new NER head.
for parameter in S2B_REDO_MODEL.parameters():
    parameter.requires_grad = False

for module in S2B_REDO_MODEL.encoder_layers:
    module.self_attn.qkv.lora_A.requires_grad = True
    module.self_attn.qkv.lora_B.requires_grad = True

for module in [
    S2B_REDO_MODEL.char_embedding,
    S2B_REDO_MODEL.char_conv,
    S2B_REDO_MODEL.start_head,
    S2B_REDO_MODEL.end_head,
]:
    for parameter in module.parameters():
        parameter.requires_grad = True

S2B_REDO_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
S2B_REDO_MODEL = S2B_REDO_MODEL.to(S2B_REDO_DEVICE)

# ============================================================
# 6. Smoke test: forward pass, output shapes, and LoRA gradients
# ============================================================
S2B_REDO_SAMPLE_ID = S2B_REDO_TRAIN_DOC_IDS[0]
S2B_REDO_SAMPLE_TEXT = S2B_REDO_TRAIN_TEXTS[S2B_REDO_SAMPLE_ID]

S2B_REDO_SAMPLE_ENCODING = S2B_REDO_TOKENIZER.encode(
    S2B_REDO_SAMPLE_TEXT[:3000],
    add_special_tokens=False,
)

S2B_REDO_SAMPLE_IDS = S2B_REDO_SAMPLE_ENCODING.ids[:32]

if not S2B_REDO_SAMPLE_IDS:
    raise RuntimeError("The sample training document tokenized to zero tokens.")

if max(S2B_REDO_SAMPLE_IDS) >= S2B_REDO_CONFIG_VOCAB:
    raise ValueError("Sample tokenizer emitted an ID outside Architecture B's vocabulary.")

S2B_REDO_SMOKE_INPUT = torch.tensor(
    [S2B_REDO_SAMPLE_IDS],
    dtype=torch.long,
    device=S2B_REDO_DEVICE,
)

S2B_REDO_SMOKE_MASK = torch.ones_like(
    S2B_REDO_SMOKE_INPUT,
    dtype=torch.bool,
)

S2B_REDO_SMOKE_CHAR_TEXT = S2B_REDO_SAMPLE_TEXT[:32]
S2B_REDO_SMOKE_CHAR_IDS = [
    S2B_REDO_CHAR_TO_ID.get(
        character,
        S2B_REDO_UNK_CHAR_ID,
    )
    for character in S2B_REDO_SMOKE_CHAR_TEXT
]

if not S2B_REDO_SMOKE_CHAR_IDS:
    S2B_REDO_SMOKE_CHAR_IDS = [S2B_REDO_UNK_CHAR_ID] * 8

S2B_REDO_SMOKE_CHAR_INPUT = torch.tensor(
    [S2B_REDO_SMOKE_CHAR_IDS],
    dtype=torch.long,
    device=S2B_REDO_DEVICE,
)

# For this shape smoke test only; full token/character alignment is built next.
S2B_REDO_SMOKE_CHAR_TO_TOKEN = torch.zeros(
    (1, len(S2B_REDO_SMOKE_CHAR_IDS)),
    dtype=torch.long,
    device=S2B_REDO_DEVICE,
)

S2B_REDO_MODEL.train()
S2B_REDO_MODEL.zero_grad(set_to_none=True)

S2B_REDO_SMOKE_START, S2B_REDO_SMOKE_END = S2B_REDO_MODEL(
    input_ids=S2B_REDO_SMOKE_INPUT,
    attention_mask=S2B_REDO_SMOKE_MASK,
    char_ids=S2B_REDO_SMOKE_CHAR_INPUT,
    char_to_token=S2B_REDO_SMOKE_CHAR_TO_TOKEN,
)

expected_shape = (
    1,
    len(S2B_REDO_SMOKE_CHAR_IDS),
    len(S2B_REDO_LABELS),
)

if tuple(S2B_REDO_SMOKE_START.shape) != expected_shape:
    raise RuntimeError(
        f"Unexpected start-logit shape: {tuple(S2B_REDO_SMOKE_START.shape)}"
    )

if tuple(S2B_REDO_SMOKE_END.shape) != expected_shape:
    raise RuntimeError(
        f"Unexpected end-logit shape: {tuple(S2B_REDO_SMOKE_END.shape)}"
    )

S2B_REDO_SMOKE_LOSS = (
    S2B_REDO_SMOKE_START.float().square().mean()
    + S2B_REDO_SMOKE_END.float().square().mean()
)
S2B_REDO_SMOKE_LOSS.backward()

S2B_REDO_LORA_GRADIENT_CHECKS = []

for index, layer in enumerate(S2B_REDO_MODEL.encoder_layers):
    grad_a = layer.self_attn.qkv.lora_A.grad
    grad_b = layer.self_attn.qkv.lora_B.grad

    S2B_REDO_LORA_GRADIENT_CHECKS.append(
        grad_a is not None
        and grad_b is not None
        and torch.isfinite(grad_a).all().item()
        and torch.isfinite(grad_b).all().item()
        and grad_b.abs().sum().item() > 0
    )

if not all(S2B_REDO_LORA_GRADIENT_CHECKS):
    raise RuntimeError(
        "LoRA gradient smoke test failed in at least one encoder layer: "
        f"{S2B_REDO_LORA_GRADIENT_CHECKS}"
    )

S2B_REDO_MODEL.zero_grad(set_to_none=True)
S2B_REDO_MODEL.eval()

# ============================================================
# 7. Parameter and checkpoint-loading report
# ============================================================
S2B_REDO_TOTAL_PARAMS = sum(
    parameter.numel()
    for parameter in S2B_REDO_MODEL.parameters()
)
S2B_REDO_TRAINABLE_PARAMS = sum(
    parameter.numel()
    for parameter in S2B_REDO_MODEL.parameters()
    if parameter.requires_grad
)

print("\n" + "=" * 76)
print("S2 REDO — ARCHITECTURE B + LoRA MODEL VERIFIED")
print("=" * 76)
print("Device:", S2B_REDO_DEVICE)
print("Architecture B encoder layers:", S2B_REDO_NUM_ENCODER_LAYERS)
print("Hidden size:", S2B_REDO_D_MODEL)
print("Attention heads:", S2B_REDO_NUM_HEADS)
print("LoRA rank:", S2B_REDO_LORA_RANK)
print("LoRA alpha:", S2B_REDO_LORA_ALPHA)
print("Loaded pretrained encoder tensors:", len(S2B_REDO_LOADED_KEYS))
print("Unexpected checkpoint keys:", len(S2B_REDO_LOAD_RESULT.unexpected_keys))
print("Total task-model parameters:", f"{S2B_REDO_TOTAL_PARAMS:,}")
print("Trainable parameters:", f"{S2B_REDO_TRAINABLE_PARAMS:,}")
print(
    "Trainable proportion:",
    f"{100 * S2B_REDO_TRAINABLE_PARAMS / S2B_REDO_TOTAL_PARAMS:.3f}%",
)
print("Start logits:", tuple(S2B_REDO_SMOKE_START.shape))
print("End logits:", tuple(S2B_REDO_SMOKE_END.shape))
print("LoRA gradient checks by encoder layer:", S2B_REDO_LORA_GRADIENT_CHECKS)
print("All LoRA gradient checks passed:", all(S2B_REDO_LORA_GRADIENT_CHECKS))
print("Architecture C weights used: NO")
print("S3 weights used: NO")
print("\nCell 3 completed. No optimizer step or model training was performed.")

S2 REDO — DOCUMENT SPLIT
All labelled documents: 70
Training documents: 56
Validation documents: 14
Training-only character vocabulary: 163
Train/validation document overlap: 0

Phase 1 tokenizer vocabulary: 50000

S2 REDO — ARCHITECTURE B + LoRA MODEL VERIFIED
Device: cuda
Architecture B encoder layers: 6
Hidden size: 768
Attention heads: 12
LoRA rank: 8
LoRA alpha: 16.0
Loaded pretrained encoder tensors: 76
Unexpected checkpoint keys: 0
Total task-model parameters: 82,178,296
Trainable parameters: 463,096
Trainable proportion: 0.564%
Start logits: (1, 32, 12)
End logits: (1, 32, 12)
LoRA gradient checks by encoder layer: [True, True, True, True, True, True]
All LoRA gradient checks passed: True
Architecture C weights used: NO
S3 weights used: NO

Cell 3 completed. No optimizer step or model training was performed.


In [4]:
# S2 Redo — Cell 4: Create character-aligned windows and entity-boundary targets.
# This cell is designed to run immediately after Cells 1–3 in a fresh notebook.

import json
import numpy as np
from pathlib import Path

# ============================================================
# 1. Define label mappings BEFORE checking prerequisites
# ============================================================
S2B_REDO_LABEL_TO_ID = {
    label: index
    for index, label in enumerate(S2B_REDO_LABELS)
}
S2B_REDO_ID_TO_LABEL = {
    index: label
    for label, index in S2B_REDO_LABEL_TO_ID.items()
}
S2B_REDO_NUM_LABELS = len(S2B_REDO_LABELS)

if not S2B_REDO_LABELS:
    raise ValueError("The Phase 2 label inventory is empty.")

if len(S2B_REDO_LABEL_TO_ID) != len(S2B_REDO_LABELS):
    raise ValueError("Duplicate entity labels detected.")

# ============================================================
# 2. Check only variables that must come from previous cells
# ============================================================
S2B_REDO_CELL4_PREREQUISITES = [
    "S2B_REDO_TOKENIZER",
    "S2B_REDO_ALL_TEXTS",
    "S2B_REDO_ALL_GOLD",
    "S2B_REDO_TRAIN_DOC_IDS",
    "S2B_REDO_VAL_DOC_IDS",
    "S2B_REDO_TRAIN_TEXTS",
    "S2B_REDO_VAL_TEXTS",
    "S2B_REDO_TRAIN_GOLD",
    "S2B_REDO_VAL_GOLD",
    "S2B_REDO_CHAR_TO_ID",
    "S2B_REDO_CHAR_VOCAB_SIZE",
    "S2B_REDO_UNK_CHAR_ID",
    "S2B_REDO_MODEL_CONFIG",
    "S2B_REDO_OUTPUT_DIR",
]

S2B_REDO_CELL4_MISSING = [
    name for name in S2B_REDO_CELL4_PREREQUISITES
    if name not in globals()
]

if S2B_REDO_CELL4_MISSING:
    raise RuntimeError(
        "Cell 4 requires the preceding setup cells to run first. "
        f"Missing variables: {S2B_REDO_CELL4_MISSING}"
    )

# ============================================================
# 3. Define window parameters
# ============================================================
S2B_REDO_MAX_TOKENS = 512
S2B_REDO_WINDOW_STRIDE = 384
S2B_REDO_PAD_CHAR_ID = 0

if not 0 < S2B_REDO_WINDOW_STRIDE <= S2B_REDO_MAX_TOKENS:
    raise ValueError("Invalid token-window stride.")

S2B_REDO_PAD_TOKEN_ID = (
    S2B_REDO_TOKENIZER.token_to_id("<pad>")
)
if S2B_REDO_PAD_TOKEN_ID is None:
    S2B_REDO_PAD_TOKEN_ID = (
        S2B_REDO_TOKENIZER.token_to_id("[PAD]")
    )
if S2B_REDO_PAD_TOKEN_ID is None:
    S2B_REDO_PAD_TOKEN_ID = 0

print("=" * 76)
print("S2 REDO — CHARACTER-ALIGNED PREPROCESSING")
print("=" * 76)
print("Labels:", S2B_REDO_NUM_LABELS)
print("Maximum tokens per window:", S2B_REDO_MAX_TOKENS)
print("Window stride:", S2B_REDO_WINDOW_STRIDE)
print("Training-only character vocabulary:", S2B_REDO_CHAR_VOCAB_SIZE)

# ============================================================
# 4. Validate annotations and preserve exclusive end offsets
# ============================================================
def s2b_redo_parse_annotations(doc_id, text, annotations):
    """
    Expected Phase 2 format:
        [start_character, end_character_exclusive, label, entity_text]
    """
    if not isinstance(annotations, list):
        raise TypeError(
            f"Annotations for document {doc_id} must be a list."
        )

    parsed = []

    for ann_idx, annotation in enumerate(annotations):
        if not isinstance(annotation, (list, tuple)) or len(annotation) < 3:
            raise ValueError(
                f"Malformed annotation in {doc_id}: {annotation!r}"
            )

        start, end, label = annotation[:3]

        if not isinstance(start, (int, np.integer)):
            raise TypeError(f"Invalid start offset: {annotation!r}")
        if not isinstance(end, (int, np.integer)):
            raise TypeError(f"Invalid end offset: {annotation!r}")

        start, end, label = int(start), int(end), str(label)

        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"Invalid character span for {doc_id}: "
                f"{(start, end, label)}, text length={len(text)}"
            )

        if label not in S2B_REDO_LABEL_TO_ID:
            raise ValueError(
                f"Unknown label {label!r} in document {doc_id}."
            )

        if len(annotation) >= 4:
            annotated_text = annotation[3]
            actual_text = text[start:end]

            if annotated_text != actual_text:
                raise ValueError(
                    f"Annotation text mismatch in {doc_id}: "
                    f"{annotated_text!r} != {actual_text!r}"
                )

        parsed.append((start, end, label, ann_idx))

    return parsed


S2B_REDO_DOC_SPANS = {}

for doc_id, text_value in S2B_REDO_ALL_TEXTS.items():
    S2B_REDO_DOC_SPANS[doc_id] = s2b_redo_parse_annotations(
        doc_id,
        text_value,
        S2B_REDO_ALL_GOLD[doc_id],
    )

S2B_REDO_TOTAL_GOLD_SPANS = sum(
    len(spans) for spans in S2B_REDO_DOC_SPANS.values()
)

print("\nAnnotation audit:")
print("Documents:", len(S2B_REDO_DOC_SPANS))
print("Gold entities:", S2B_REDO_TOTAL_GOLD_SPANS)
print("Exclusive end offsets verified against annotation text.")

# ============================================================
# 5. Build overlapping windows for one document
# ============================================================
def s2b_redo_build_document_windows(doc_id, text_value, parsed_spans):
    if not text_value:
        if parsed_spans:
            raise ValueError(
                f"Empty document {doc_id} contains gold annotations."
            )
        return [], set(), 0

    encoding = S2B_REDO_TOKENIZER.encode(
        text_value,
        add_special_tokens=False,
    )

    token_ids = encoding.ids
    token_offsets = encoding.offsets

    if len(token_ids) != len(token_offsets):
        raise RuntimeError(
            f"Tokenizer offset count mismatch for {doc_id}."
        )

    if not token_ids:
        if parsed_spans:
            raise ValueError(
                f"Document {doc_id} has entities but no tokens."
            )
        return [], set(), 0

    if max(token_ids) >= int(S2B_REDO_MODEL_CONFIG["vocab_size"]):
        raise ValueError(
            f"Token ID outside Architecture B vocabulary in {doc_id}."
        )

    encoded_spans = [
        (
            start,
            end,
            S2B_REDO_LABEL_TO_ID[label],
            ann_idx,
        )
        for start, end, label, ann_idx in parsed_spans
    ]

    windows = []
    covered_entities = set()
    fallback_count = 0

    token_start = 0

    while token_start < len(token_ids):
        token_end = min(
            token_start + S2B_REDO_MAX_TOKENS,
            len(token_ids),
        )

        window_offsets = token_offsets[token_start:token_end]

        valid_offsets = [
            (int(start), int(end), local_idx)
            for local_idx, (start, end) in enumerate(window_offsets)
            if int(end) > int(start)
        ]

        if not valid_offsets:
            raise RuntimeError(
                f"No valid token character offsets in {doc_id}."
            )

        char_start = min(a for a, _, _ in valid_offsets)
        char_end = max(b for _, b, _ in valid_offsets)

        if not (0 <= char_start < char_end <= len(text_value)):
            raise ValueError(
                f"Invalid window character offsets for {doc_id}: "
                f"{(char_start, char_end)}"
            )

        char_length = char_end - char_start

        # Character positions map to local token indices.
        char_to_token = np.full(
            char_length,
            -1,
            dtype=np.int16,
        )

        for offset_start, offset_end, local_token_idx in valid_offsets:
            left = max(offset_start, char_start) - char_start
            right = min(offset_end, char_end) - char_start

            if right > left:
                char_to_token[left:right] = local_token_idx

        # Fill any uncovered character positions with the nearest token.
        missing_positions = np.flatnonzero(char_to_token < 0)

        if len(missing_positions):
            for local_char_idx in missing_positions:
                absolute_char_idx = char_start + int(local_char_idx)

                def distance_to_token(offset_record):
                    a, b, _ = offset_record
                    if a <= absolute_char_idx < b:
                        return 0
                    if absolute_char_idx < a:
                        return a - absolute_char_idx
                    return absolute_char_idx - b + 1

                nearest_record = min(
                    valid_offsets,
                    key=distance_to_token,
                )
                char_to_token[local_char_idx] = nearest_record[2]
                fallback_count += 1

        # Character IDs are learned from training documents only.
        char_ids = np.fromiter(
            (
                S2B_REDO_CHAR_TO_ID.get(
                    char,
                    S2B_REDO_UNK_CHAR_ID,
                )
                for char in text_value[char_start:char_end]
            ),
            dtype=np.int32,
            count=char_length,
        )

        start_targets = np.zeros(
            (char_length, S2B_REDO_NUM_LABELS),
            dtype=np.uint8,
        )
        end_targets = np.zeros(
            (char_length, S2B_REDO_NUM_LABELS),
            dtype=np.uint8,
        )
        loss_mask = np.ones(
            (char_length, S2B_REDO_NUM_LABELS),
            dtype=np.bool_,
        )

        for start, end, label_id, ann_idx in encoded_spans:
            fully_contained = (
                char_start <= start and end <= char_end
            )

            if fully_contained:
                local_start = start - char_start
                local_end = end - 1 - char_start

                start_targets[local_start, label_id] = 1
                end_targets[local_end, label_id] = 1
                covered_entities.add(ann_idx)

            else:
                # Ignore incomplete boundary targets for spans crossing
                # this window's character range.
                if char_start <= start < char_end:
                    loss_mask[start - char_start, label_id] = False

                if char_start < end <= char_end:
                    loss_mask[end - 1 - char_start, label_id] = False

        windows.append({
            "doc_id": doc_id,
            "token_start": token_start,
            "token_end": token_end,
            "char_start": char_start,
            "char_end": char_end,
            "input_ids": np.asarray(
                token_ids[token_start:token_end],
                dtype=np.int32,
            ),
            "char_ids": char_ids,
            "char_to_token": char_to_token,
            "start_targets": start_targets,
            "end_targets": end_targets,
            "loss_mask": loss_mask,
        })

        if token_end == len(token_ids):
            break

        token_start += S2B_REDO_WINDOW_STRIDE

    return windows, covered_entities, fallback_count


# ============================================================
# 6. Prepare train and validation windows separately
# ============================================================
S2B_REDO_TRAIN_WINDOWS = []
S2B_REDO_VAL_WINDOWS = []

S2B_REDO_ALIGNMENT_FALLBACKS = {
    "train": 0,
    "validation": 0,
}

def s2b_redo_prepare_split(text_mapping, gold_mapping, split_name):
    destination = []
    covered_by_doc = {}
    total_entities = 0
    total_chars = 0
    total_tokens = 0

    if set(text_mapping) != set(gold_mapping):
        raise ValueError(
            f"{split_name}: text/gold document IDs do not match."
        )

    for doc_id, text_value in text_mapping.items():
        parsed_spans = S2B_REDO_DOC_SPANS[doc_id]

        windows, covered, fallback_count = (
            s2b_redo_build_document_windows(
                doc_id,
                text_value,
                parsed_spans,
            )
        )

        destination.extend(windows)
        covered_by_doc[doc_id] = covered
        total_entities += len(parsed_spans)
        total_chars += len(text_value)
        total_tokens += sum(
            len(window["input_ids"]) for window in windows
        )
        S2B_REDO_ALIGNMENT_FALLBACKS[split_name] += fallback_count

    missing = []

    for doc_id, parsed_spans in (
        (doc_id, S2B_REDO_DOC_SPANS[doc_id])
        for doc_id in text_mapping
    ):
        expected = {ann_idx for _, _, _, ann_idx in parsed_spans}
        uncovered = expected - covered_by_doc[doc_id]

        for ann_idx in uncovered:
            missing.append((doc_id, ann_idx))

    if missing:
        raise RuntimeError(
            f"{split_name}: some entities are not fully represented in "
            f"any window. Examples: {missing[:10]}"
        )

    return destination, {
        "split": split_name,
        "documents": len(text_mapping),
        "characters": total_chars,
        "gold_entities": total_entities,
        "windows": len(destination),
        "tokens_across_windows": total_tokens,
    }


S2B_REDO_TRAIN_WINDOWS, S2B_REDO_TRAIN_WINDOW_STATS = (
    s2b_redo_prepare_split(
        S2B_REDO_TRAIN_TEXTS,
        S2B_REDO_TRAIN_GOLD,
        "train",
    )
)

S2B_REDO_VAL_WINDOWS, S2B_REDO_VAL_WINDOW_STATS = (
    s2b_redo_prepare_split(
        S2B_REDO_VAL_TEXTS,
        S2B_REDO_VAL_GOLD,
        "validation",
    )
)

# ============================================================
# 7. Validate shapes and summarize boundary targets by label
# ============================================================
def s2b_redo_audit_windows(split_name, window_list):
    if not window_list:
        raise RuntimeError(f"{split_name} produced no windows.")

    positive_starts = np.zeros(
        S2B_REDO_NUM_LABELS,
        dtype=np.int64,
    )
    positive_ends = np.zeros(
        S2B_REDO_NUM_LABELS,
        dtype=np.int64,
    )

    for window in window_list:
        num_tokens = len(window["input_ids"])
        char_length = len(window["char_ids"])

        if not 1 <= num_tokens <= S2B_REDO_MAX_TOKENS:
            raise ValueError(
                f"Invalid token window length: {num_tokens}"
            )

        if len(window["char_to_token"]) != char_length:
            raise ValueError("Character alignment length mismatch.")

        if (
            window["char_to_token"].min() < 0
            or window["char_to_token"].max() >= num_tokens
        ):
            raise ValueError("Character-to-token index out of range.")

        expected_shape = (char_length, S2B_REDO_NUM_LABELS)

        for array_name in [
            "start_targets",
            "end_targets",
            "loss_mask",
        ]:
            if window[array_name].shape != expected_shape:
                raise ValueError(
                    f"{array_name} shape mismatch in "
                    f"{window['doc_id']}."
                )

        
        # FIXED: Explicitly accumulate as signed 64-bit integers.
        positive_starts += window["start_targets"].sum(
            axis=0, dtype=np.int64
        )
        # FIXED: Use the same integer dtype for end-boundary counts.
        positive_ends += window["end_targets"].sum(
            axis=0, dtype=np.int64
        )

    if not np.array_equal(positive_starts, positive_ends):
        raise RuntimeError(
            f"{split_name}: start and end target totals differ."
        )

    print(f"\n{split_name.upper()} boundary counts:")
    for label_id, label in enumerate(S2B_REDO_LABELS):
        print(
            f"  {label:12s} "
            f"starts={positive_starts[label_id]:5d} "
            f"ends={positive_ends[label_id]:5d}"
        )


s2b_redo_audit_windows(
    "train",
    S2B_REDO_TRAIN_WINDOWS,
)
s2b_redo_audit_windows(
    "validation",
    S2B_REDO_VAL_WINDOWS,
)

# Verify that no document is shared between the splits.
if set(S2B_REDO_TRAIN_DOC_IDS) & set(S2B_REDO_VAL_DOC_IDS):
    raise RuntimeError("Train/validation document leakage detected.")

# ============================================================
# 8. Save preprocessing report
# ============================================================
S2B_REDO_PREPROCESSING_REPORT = {
    "max_tokens": S2B_REDO_MAX_TOKENS,
    "stride": S2B_REDO_WINDOW_STRIDE,
    "labels": S2B_REDO_LABELS,
    "char_vocab_size": S2B_REDO_CHAR_VOCAB_SIZE,
    "train": S2B_REDO_TRAIN_WINDOW_STATS,
    "validation": S2B_REDO_VAL_WINDOW_STATS,
    "alignment_fallbacks": S2B_REDO_ALIGNMENT_FALLBACKS,
    "train_validation_document_overlap": 0,
    "uncovered_train_entities": 0,
    "uncovered_validation_entities": 0,
    "fold3_gold_used": False,
}

S2B_REDO_PREPROCESSING_REPORT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_preprocessing_report.json"
)

with S2B_REDO_PREPROCESSING_REPORT_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_PREPROCESSING_REPORT,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\n" + "=" * 76)
print("S2 REDO — PREPROCESSING COMPLETED")
print("=" * 76)
print("Training statistics:", S2B_REDO_TRAIN_WINDOW_STATS)
print("Validation statistics:", S2B_REDO_VAL_WINDOW_STATS)
print("Character alignment fallbacks:", S2B_REDO_ALIGNMENT_FALLBACKS)
print("Every training entity covered: YES")
print("Every validation entity covered: YES")
print("Train/validation leakage: NONE")
print("Fold 3 annotations used: NO")
print("Report:", S2B_REDO_PREPROCESSING_REPORT_PATH)
print("\nCell 4 completed. No model training performed.")

S2 REDO — CHARACTER-ALIGNED PREPROCESSING
Labels: 12
Maximum tokens per window: 512
Window stride: 384
Training-only character vocabulary: 163

Annotation audit:
Documents: 70
Gold entities: 8393
Exclusive end offsets verified against annotation text.

TRAIN boundary counts:
  APP          starts=  432 ends=  432
  RESP         starts=  392 ends=  392
  A.COUNSEL    starts=  213 ends=  213
  R.COUNSEL    starts=  211 ends=  211
  JUDGE        starts=  199 ends=  199
  WIT          starts=  162 ends=  162
  AUTH         starts= 1074 ends= 1074
  COURT        starts= 1141 ends= 1141
  STAT         starts= 1385 ends= 1385
  PREC         starts= 1276 ends= 1276
  DATE         starts= 1637 ends= 1637
  CASENO       starts=  745 ends=  745

VALIDATION boundary counts:
  APP          starts=   66 ends=   66
  RESP         starts=   86 ends=   86
  A.COUNSEL    starts=   32 ends=   32
  R.COUNSEL    starts=   26 ends=   26
  JUDGE        starts=   73 ends=   73
  WIT          starts=   78 ends

In [5]:
# S2 Redo — Cell 5: Create padded DataLoaders and verify the boundary-loss pipeline.

import random
import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. Verify prerequisites from Cells 1–4
# ============================================================
S2B_REDO_CELL5_REQUIRED = [
    "S2B_REDO_MODEL",
    "S2B_REDO_DEVICE",
    "S2B_REDO_TRAIN_WINDOWS",
    "S2B_REDO_VAL_WINDOWS",
    "S2B_REDO_LABELS",
    "S2B_REDO_NUM_LABELS",
    "S2B_REDO_PAD_TOKEN_ID",
    "S2B_REDO_PAD_CHAR_ID",
    "S2B_REDO_MAX_TOKENS",
    "S2B_REDO_OUTPUT_DIR",
]

S2B_REDO_CELL5_MISSING = [
    name for name in S2B_REDO_CELL5_REQUIRED
    if name not in globals()
]

if S2B_REDO_CELL5_MISSING:
    raise RuntimeError(
        "Run Cells 1–4 sequentially before Cell 5. Missing: "
        f"{S2B_REDO_CELL5_MISSING}"
    )

if not S2B_REDO_TRAIN_WINDOWS or not S2B_REDO_VAL_WINDOWS:
    raise RuntimeError("Training or validation windows are empty.")

# ============================================================
# 2. Set deterministic DataLoader settings
# ============================================================
S2B_REDO_SEED = 42
random.seed(S2B_REDO_SEED)
np.random.seed(S2B_REDO_SEED)
torch.manual_seed(S2B_REDO_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(S2B_REDO_SEED)

S2B_REDO_BATCH_SIZE = 2
S2B_REDO_NUM_WORKERS = 0

# ============================================================
# 3. Dataset wrapper
# ============================================================
class S2BRedoWindowDataset(Dataset):
    """Expose one prepared document window per item."""

    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        return self.windows[index]


# ============================================================
# 4. Dynamic padding for token and character sequences
# ============================================================
def s2b_redo_collate_windows(items):
    """Pad token and character dimensions independently."""

    batch_size = len(items)
    max_tokens = max(len(item["input_ids"]) for item in items)
    max_chars = max(len(item["char_ids"]) for item in items)

    num_labels = S2B_REDO_NUM_LABELS

    input_ids = torch.full(
        (batch_size, max_tokens),
        fill_value=S2B_REDO_PAD_TOKEN_ID,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.bool,
    )

    char_ids = torch.full(
        (batch_size, max_chars),
        fill_value=S2B_REDO_PAD_CHAR_ID,
        dtype=torch.long,
    )

    # Padded character positions point to token zero, but their loss is masked.
    char_to_token = torch.zeros(
        (batch_size, max_chars),
        dtype=torch.long,
    )

    start_targets = torch.zeros(
        (batch_size, max_chars, num_labels),
        dtype=torch.float32,
    )
    end_targets = torch.zeros(
        (batch_size, max_chars, num_labels),
        dtype=torch.float32,
    )
    loss_mask = torch.zeros(
        (batch_size, max_chars, num_labels),
        dtype=torch.bool,
    )

    doc_ids = []
    token_starts = []
    char_starts = []
    char_ends = []
    original_char_lengths = []

    for batch_idx, item in enumerate(items):
        token_length = len(item["input_ids"])
        char_length = len(item["char_ids"])

        input_ids[batch_idx, :token_length] = torch.as_tensor(
            item["input_ids"].astype(np.int64, copy=False),
            dtype=torch.long,
        )
        attention_mask[batch_idx, :token_length] = True

        char_ids[batch_idx, :char_length] = torch.as_tensor(
            item["char_ids"].astype(np.int64, copy=False),
            dtype=torch.long,
        )

        char_to_token[batch_idx, :char_length] = torch.as_tensor(
            item["char_to_token"].astype(np.int64, copy=False),
            dtype=torch.long,
        )

        start_targets[batch_idx, :char_length] = torch.from_numpy(
            item["start_targets"].astype(np.float32, copy=False)
        )
        end_targets[batch_idx, :char_length] = torch.from_numpy(
            item["end_targets"].astype(np.float32, copy=False)
        )
        loss_mask[batch_idx, :char_length] = torch.from_numpy(
            item["loss_mask"].astype(np.bool_, copy=False)
        )

        doc_ids.append(item["doc_id"])
        token_starts.append(int(item["token_start"]))
        char_starts.append(int(item["char_start"]))
        char_ends.append(int(item["char_end"]))
        original_char_lengths.append(char_length)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "start_targets": start_targets,
        "end_targets": end_targets,
        "loss_mask": loss_mask,
        "doc_ids": doc_ids,
        "token_starts": token_starts,
        "char_starts": char_starts,
        "char_ends": char_ends,
        "char_lengths": original_char_lengths,
    }


# ============================================================
# 5. Create DataLoaders
# ============================================================
S2B_REDO_TRAIN_DATASET = S2BRedoWindowDataset(
    S2B_REDO_TRAIN_WINDOWS
)
S2B_REDO_VAL_DATASET = S2BRedoWindowDataset(
    S2B_REDO_VAL_WINDOWS
)

S2B_REDO_TRAIN_LOADER = DataLoader(
    S2B_REDO_TRAIN_DATASET,
    batch_size=S2B_REDO_BATCH_SIZE,
    shuffle=True,
    num_workers=S2B_REDO_NUM_WORKERS,
    pin_memory=(S2B_REDO_DEVICE.type == "cuda"),
    collate_fn=s2b_redo_collate_windows,
    drop_last=False,
)

S2B_REDO_VAL_LOADER = DataLoader(
    S2B_REDO_VAL_DATASET,
    batch_size=S2B_REDO_BATCH_SIZE,
    shuffle=False,
    num_workers=S2B_REDO_NUM_WORKERS,
    pin_memory=(S2B_REDO_DEVICE.type == "cuda"),
    collate_fn=s2b_redo_collate_windows,
    drop_last=False,
)

# ============================================================
# 6. Compute per-label positive weights from actual masked targets
# ============================================================
S2B_REDO_VALID_CHAR_LABEL_COUNTS = np.zeros(
    S2B_REDO_NUM_LABELS,
    dtype=np.int64,
)
S2B_REDO_START_POSITIVE_COUNTS = np.zeros(
    S2B_REDO_NUM_LABELS,
    dtype=np.int64,
)
S2B_REDO_END_POSITIVE_COUNTS = np.zeros(
    S2B_REDO_NUM_LABELS,
    dtype=np.int64,
)

for window in S2B_REDO_TRAIN_WINDOWS:
    S2B_REDO_VALID_CHAR_LABEL_COUNTS += window["loss_mask"].sum(
        axis=0,
        dtype=np.int64,
    )

    S2B_REDO_START_POSITIVE_COUNTS += (
        window["start_targets"].sum(axis=0, dtype=np.int64)
    )
    S2B_REDO_END_POSITIVE_COUNTS += (
        window["end_targets"].sum(axis=0, dtype=np.int64)
    )

if not np.array_equal(
    S2B_REDO_START_POSITIVE_COUNTS,
    S2B_REDO_END_POSITIVE_COUNTS,
):
    raise RuntimeError("Training start/end positive counts differ.")

if np.any(S2B_REDO_START_POSITIVE_COUNTS <= 0):
    zero_labels = [
        S2B_REDO_LABELS[i]
        for i in np.flatnonzero(
            S2B_REDO_START_POSITIVE_COUNTS <= 0
        )
    ]
    raise RuntimeError(
        f"Training split has no positive boundaries for labels: {zero_labels}"
    )

S2B_REDO_NEGATIVE_COUNTS = (
    S2B_REDO_VALID_CHAR_LABEL_COUNTS
    - S2B_REDO_START_POSITIVE_COUNTS
)

S2B_REDO_RAW_POS_WEIGHTS = np.sqrt(
    np.maximum(S2B_REDO_NEGATIVE_COUNTS, 1)
    / np.maximum(S2B_REDO_START_POSITIVE_COUNTS, 1)
)

# Limit weights to avoid extreme gradients in very sparse classes.
S2B_REDO_POS_WEIGHTS = np.clip(
    S2B_REDO_RAW_POS_WEIGHTS,
    1.0,
    30.0,
).astype(np.float32)

S2B_REDO_POS_WEIGHT_TENSOR = torch.tensor(
    S2B_REDO_POS_WEIGHTS,
    dtype=torch.float32,
    device=S2B_REDO_DEVICE,
)

print("=" * 76)
print("S2 REDO — DATA LOADERS AND CLASS WEIGHTS")
print("=" * 76)
print("Training windows:", len(S2B_REDO_TRAIN_DATASET))
print("Validation windows:", len(S2B_REDO_VAL_DATASET))
print("Training batches per epoch:", len(S2B_REDO_TRAIN_LOADER))
print("Validation batches:", len(S2B_REDO_VAL_LOADER))
print("Batch size:", S2B_REDO_BATCH_SIZE)

print("\nPer-label boundary statistics:")
for label_id, label in enumerate(S2B_REDO_LABELS):
    print(
        f"  {label:12s} "
        f"positive={S2B_REDO_START_POSITIVE_COUNTS[label_id]:5d} "
        f"valid={S2B_REDO_VALID_CHAR_LABEL_COUNTS[label_id]:8d} "
        f"pos_weight={S2B_REDO_POS_WEIGHTS[label_id]:6.2f}"
    )

# ============================================================
# 7. Define a masked, numerically stable boundary loss
# ============================================================
def s2b_redo_masked_boundary_loss(
    start_logits,
    end_logits,
    start_targets,
    end_targets,
    loss_mask,
):
    """
    Compute masked BCE losses for start/end boundaries.
    Padding and ambiguous window-edge targets do not contribute.
    """
    if start_logits.shape != start_targets.shape:
        raise ValueError(
            f"Start shape mismatch: logits={tuple(start_logits.shape)}, "
            f"targets={tuple(start_targets.shape)}"
        )

    if end_logits.shape != end_targets.shape:
        raise ValueError(
            f"End shape mismatch: logits={tuple(end_logits.shape)}, "
            f"targets={tuple(end_targets.shape)}"
        )

    if loss_mask.shape != start_logits.shape:
        raise ValueError("Loss-mask shape does not match boundary logits.")

    mask_float = loss_mask.to(dtype=torch.float32)

    start_loss_elements = F.binary_cross_entropy_with_logits(
        start_logits.float(),
        start_targets.float(),
        pos_weight=S2B_REDO_POS_WEIGHT_TENSOR,
        reduction="none",
    )

    end_loss_elements = F.binary_cross_entropy_with_logits(
        end_logits.float(),
        end_targets.float(),
        pos_weight=S2B_REDO_POS_WEIGHT_TENSOR,
        reduction="none",
    )

    valid_count = mask_float.sum().clamp_min(1.0)

    start_loss = (start_loss_elements * mask_float).sum() / valid_count
    end_loss = (end_loss_elements * mask_float).sum() / valid_count

    return 0.5 * (start_loss + end_loss)


# ============================================================
# 8. Run a real batch through the model and verify the loss
# ============================================================
S2B_REDO_MODEL.eval()

S2B_REDO_SMOKE_BATCH = next(iter(S2B_REDO_TRAIN_LOADER))

S2B_REDO_SMOKE_DEVICE_BATCH = {
    key: S2B_REDO_SMOKE_BATCH[key].to(
        S2B_REDO_DEVICE,
        non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
    )
    for key in [
        "input_ids",
        "attention_mask",
        "char_ids",
        "char_to_token",
        "start_targets",
        "end_targets",
        "loss_mask",
    ]
}

with torch.no_grad():
    if S2B_REDO_DEVICE.type == "cuda":
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            S2B_REDO_SMOKE_START_LOGITS, S2B_REDO_SMOKE_END_LOGITS = (
                S2B_REDO_MODEL(
                    input_ids=S2B_REDO_SMOKE_DEVICE_BATCH["input_ids"],
                    attention_mask=S2B_REDO_SMOKE_DEVICE_BATCH["attention_mask"],
                    char_ids=S2B_REDO_SMOKE_DEVICE_BATCH["char_ids"],
                    char_to_token=S2B_REDO_SMOKE_DEVICE_BATCH["char_to_token"],
                )
            )
    else:
        S2B_REDO_SMOKE_START_LOGITS, S2B_REDO_SMOKE_END_LOGITS = (
            S2B_REDO_MODEL(
                input_ids=S2B_REDO_SMOKE_DEVICE_BATCH["input_ids"],
                attention_mask=S2B_REDO_SMOKE_DEVICE_BATCH["attention_mask"],
                char_ids=S2B_REDO_SMOKE_DEVICE_BATCH["char_ids"],
                char_to_token=S2B_REDO_SMOKE_DEVICE_BATCH["char_to_token"],
            )
        )

    S2B_REDO_SMOKE_LOSS = s2b_redo_masked_boundary_loss(
        start_logits=S2B_REDO_SMOKE_START_LOGITS,
        end_logits=S2B_REDO_SMOKE_END_LOGITS,
        start_targets=S2B_REDO_SMOKE_DEVICE_BATCH["start_targets"],
        end_targets=S2B_REDO_SMOKE_DEVICE_BATCH["end_targets"],
        loss_mask=S2B_REDO_SMOKE_DEVICE_BATCH["loss_mask"],
    )

if not torch.isfinite(S2B_REDO_SMOKE_LOSS):
    raise RuntimeError("Boundary-loss smoke test returned a non-finite value.")

print("\n" + "=" * 76)
print("S2 REDO — PIPELINE SMOKE TEST")
print("=" * 76)
print("Document IDs in batch:", S2B_REDO_SMOKE_BATCH["doc_ids"])
print("Token input shape:", tuple(S2B_REDO_SMOKE_DEVICE_BATCH["input_ids"].shape))
print("Character input shape:", tuple(S2B_REDO_SMOKE_DEVICE_BATCH["char_ids"].shape))
print("Start logits shape:", tuple(S2B_REDO_SMOKE_START_LOGITS.shape))
print("End logits shape:", tuple(S2B_REDO_SMOKE_END_LOGITS.shape))
print("Masked boundary loss:", float(S2B_REDO_SMOKE_LOSS.item()))
print("Loss finite: YES")
print("Optimizer steps performed: 0")

print("\nCell 5 completed successfully.")

S2 REDO — DATA LOADERS AND CLASS WEIGHTS
Training windows: 996
Validation windows: 200
Training batches per epoch: 498
Validation batches: 100
Batch size: 2

Per-label boundary statistics:
  APP          positive=  432 valid= 2437828 pos_weight= 30.00
  RESP         positive=  392 valid= 2437828 pos_weight= 30.00
  A.COUNSEL    positive=  213 valid= 2437831 pos_weight= 30.00
  R.COUNSEL    positive=  211 valid= 2437829 pos_weight= 30.00
  JUDGE        positive=  199 valid= 2437828 pos_weight= 30.00
  WIT          positive=  162 valid= 2437830 pos_weight= 30.00
  AUTH         positive= 1074 valid= 2437829 pos_weight= 30.00
  COURT        positive= 1141 valid= 2437826 pos_weight= 30.00
  STAT         positive= 1385 valid= 2437816 pos_weight= 30.00
  PREC         positive= 1276 valid= 2437784 pos_weight= 30.00
  DATE         positive= 1637 valid= 2437809 pos_weight= 30.00
  CASENO       positive=  745 valid= 2437816 pos_weight= 30.00

S2 REDO — PIPELINE SMOKE TEST
Document IDs in batch: [

In [6]:
# S2 Redo — Cell 6: Configure training with a scaled gradient smoke test.

import json
import math
import numpy as np
import torch
import torch.nn.functional as F

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# ============================================================
# 1. Verify sequential prerequisites
# ============================================================
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_DEVICE",
    "S2B_REDO_TRAIN_WINDOWS",
    "S2B_REDO_TRAIN_LOADER",
    "S2B_REDO_VAL_LOADER",
    "S2B_REDO_LABELS",
    "S2B_REDO_NUM_LABELS",
    "S2B_REDO_VALID_CHAR_LABEL_COUNTS",
    "S2B_REDO_START_POSITIVE_COUNTS",
    "S2B_REDO_OUTPUT_DIR",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Run Cells 1–5 first. Missing variables: {missing}"
    )

# ============================================================
# 2. Recalculate conservative class weights
# ============================================================
S2B_REDO_NEGATIVE_COUNTS = (
    S2B_REDO_VALID_CHAR_LABEL_COUNTS
    - S2B_REDO_START_POSITIVE_COUNTS
)

if np.any(S2B_REDO_NEGATIVE_COUNTS < 0):
    raise ValueError("Negative boundary count encountered.")

S2B_REDO_POS_WEIGHTS = np.clip(
    (
        np.maximum(S2B_REDO_NEGATIVE_COUNTS, 1)
        / np.maximum(S2B_REDO_START_POSITIVE_COUNTS, 1)
    ) ** 0.25,
    2.0,
    12.0,
).astype(np.float32)

S2B_REDO_POS_WEIGHT_TENSOR = torch.tensor(
    S2B_REDO_POS_WEIGHTS,
    dtype=torch.float32,
    device=S2B_REDO_DEVICE,
)

# ============================================================
# 3. Define stable masked boundary loss
# ============================================================
def s2b_redo_boundary_loss(
    start_logits,
    end_logits,
    start_targets,
    end_targets,
    loss_mask,
):
    expected_shape = start_targets.shape

    if (
        start_logits.shape != expected_shape
        or end_logits.shape != expected_shape
        or end_targets.shape != expected_shape
        or loss_mask.shape != expected_shape
    ):
        raise ValueError("Boundary logits, targets, and masks must match.")

    mask = loss_mask.to(torch.float32)
    denominator = mask.sum().clamp_min(1.0)

    start_elements = F.binary_cross_entropy_with_logits(
        start_logits.float(),
        start_targets.float(),
        pos_weight=S2B_REDO_POS_WEIGHT_TENSOR,
        reduction="none",
    )

    end_elements = F.binary_cross_entropy_with_logits(
        end_logits.float(),
        end_targets.float(),
        pos_weight=S2B_REDO_POS_WEIGHT_TENSOR,
        reduction="none",
    )

    start_loss = (start_elements * mask).sum() / denominator
    end_loss = (end_elements * mask).sum() / denominator

    return 0.5 * (start_loss + end_loss)

# ============================================================
# 4. Rebuild optimizer groups from trainable parameters
# ============================================================
S2B_REDO_LORA_PARAMS = []
S2B_REDO_HEAD_PARAMS = []

for name, parameter in S2B_REDO_MODEL.named_parameters():
    if not parameter.requires_grad:
        continue

    if ".lora_A" in name or ".lora_B" in name:
        S2B_REDO_LORA_PARAMS.append(parameter)
    elif name.startswith((
        "char_embedding.",
        "char_conv.",
        "start_head.",
        "end_head.",
    )):
        S2B_REDO_HEAD_PARAMS.append(parameter)
    else:
        raise RuntimeError(
            f"Unexpected trainable base parameter: {name}"
        )

if not S2B_REDO_LORA_PARAMS or not S2B_REDO_HEAD_PARAMS:
    raise RuntimeError("LoRA or task-head parameters are missing.")

S2B_REDO_LORA_LR = 1.5e-4
S2B_REDO_HEAD_LR = 5.0e-4
S2B_REDO_WEIGHT_DECAY = 0.01

S2B_REDO_OPTIMIZER = AdamW(
    [
        {
            "params": S2B_REDO_LORA_PARAMS,
            "lr": S2B_REDO_LORA_LR,
            "name": "lora",
        },
        {
            "params": S2B_REDO_HEAD_PARAMS,
            "lr": S2B_REDO_HEAD_LR,
            "name": "ner_head",
        },
    ],
    weight_decay=S2B_REDO_WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

# ============================================================
# 5. Configure the learning-rate schedule and AMP scaler
# ============================================================
S2B_REDO_MAX_EPOCHS = 5
S2B_REDO_GRAD_CLIP_NORM = 1.0
S2B_REDO_STEPS_PER_EPOCH = len(S2B_REDO_TRAIN_LOADER)
S2B_REDO_TOTAL_STEPS = (
    S2B_REDO_MAX_EPOCHS * S2B_REDO_STEPS_PER_EPOCH
)
S2B_REDO_WARMUP_STEPS = max(
    1, int(0.06 * S2B_REDO_TOTAL_STEPS)
)

def s2b_redo_lr_multiplier(step_index):
    if step_index < S2B_REDO_WARMUP_STEPS:
        return max(
            0.05,
            (step_index + 1) / S2B_REDO_WARMUP_STEPS,
        )

    decay_steps = max(
        1,
        S2B_REDO_TOTAL_STEPS - S2B_REDO_WARMUP_STEPS,
    )
    progress = (
        step_index - S2B_REDO_WARMUP_STEPS
    ) / decay_steps
    progress = min(1.0, max(0.0, progress))

    return 0.5 * (1.0 + math.cos(math.pi * progress))

S2B_REDO_SCHEDULER = LambdaLR(
    S2B_REDO_OPTIMIZER,
    lr_lambda=s2b_redo_lr_multiplier,
)

S2B_REDO_SCALER = torch.amp.GradScaler(
    "cuda",
    enabled=(S2B_REDO_DEVICE.type == "cuda"),
)

# ============================================================
# 6. Run scaled gradient diagnostics; no optimizer step
# ============================================================
S2B_REDO_MODEL.train()
S2B_REDO_MODEL.zero_grad(set_to_none=True)
S2B_REDO_OPTIMIZER.zero_grad(set_to_none=True)

batch = next(iter(S2B_REDO_TRAIN_LOADER))

device_batch = {
    key: batch[key].to(
        S2B_REDO_DEVICE,
        non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
    )
    for key in [
        "input_ids",
        "attention_mask",
        "char_ids",
        "char_to_token",
        "start_targets",
        "end_targets",
        "loss_mask",
    ]
}

with torch.autocast(
    device_type=S2B_REDO_DEVICE.type,
    dtype=torch.float16,
    enabled=(S2B_REDO_DEVICE.type == "cuda"),
):
    start_logits, end_logits = S2B_REDO_MODEL(
        input_ids=device_batch["input_ids"],
        attention_mask=device_batch["attention_mask"],
        char_ids=device_batch["char_ids"],
        char_to_token=device_batch["char_to_token"],
    )

loss = s2b_redo_boundary_loss(
    start_logits=start_logits,
    end_logits=end_logits,
    start_targets=device_batch["start_targets"],
    end_targets=device_batch["end_targets"],
    loss_mask=device_batch["loss_mask"],
)

if not torch.isfinite(loss):
    raise RuntimeError("The boundary loss is not finite.")

# Scale this diagnostic backward pass to avoid tiny FP16 gradients
# disappearing during the smoke test. This is not an optimizer update.
S2B_REDO_DIAGNOSTIC_SCALE = 1024.0
(loss * S2B_REDO_DIAGNOSTIC_SCALE).backward()

S2B_REDO_LORA_B_GRAD_NORMS = []
S2B_REDO_LORA_A_GRAD_NORMS = []

for layer_idx, layer in enumerate(S2B_REDO_MODEL.encoder_layers):
    grad_a = layer.self_attn.qkv.lora_A.grad
    grad_b = layer.self_attn.qkv.lora_B.grad

    if grad_a is None or grad_b is None:
        raise RuntimeError(
            f"LoRA gradients are missing in encoder layer {layer_idx}."
        )

    if (
        not torch.isfinite(grad_a).all()
        or not torch.isfinite(grad_b).all()
    ):
        raise RuntimeError(
            f"Non-finite LoRA gradient in encoder layer {layer_idx}."
        )

    # Divide the norms by the diagnostic scale for readable reporting.
    S2B_REDO_LORA_A_GRAD_NORMS.append(
        float(grad_a.detach().float().norm().item())
        / S2B_REDO_DIAGNOSTIC_SCALE
    )
    S2B_REDO_LORA_B_GRAD_NORMS.append(
        float(grad_b.detach().float().norm().item())
        / S2B_REDO_DIAGNOSTIC_SCALE
    )

S2B_REDO_HEAD_GRAD_NORMS = []

for parameter in S2B_REDO_HEAD_PARAMS:
    if parameter.grad is None:
        raise RuntimeError("Task-head gradient is missing.")

    if not torch.isfinite(parameter.grad).all():
        raise RuntimeError("Non-finite task-head gradient.")

    S2B_REDO_HEAD_GRAD_NORMS.append(
        float(parameter.grad.detach().float().norm().item())
        / S2B_REDO_DIAGNOSTIC_SCALE
    )

if not any(value > 0 for value in S2B_REDO_LORA_B_GRAD_NORMS):
    raise RuntimeError(
        "LoRA B gradients are zero even after diagnostic scaling. "
        "Training has not started; inspect the printed architecture/gradient "
        "flow before proceeding."
    )

if not any(value > 0 for value in S2B_REDO_HEAD_GRAD_NORMS):
    raise RuntimeError("All NER-head gradients are zero.")

# Clear smoke-test gradients and restore evaluation mode.
S2B_REDO_MODEL.zero_grad(set_to_none=True)
S2B_REDO_OPTIMIZER.zero_grad(set_to_none=True)
S2B_REDO_MODEL.eval()

# ============================================================
# 7. Save configuration for reproducibility
# ============================================================
S2B_REDO_TRAINING_CONFIG = {
    "experiment": "S2 Architecture B + LoRA redo",
    "max_epochs": S2B_REDO_MAX_EPOCHS,
    "batch_size": S2B_REDO_BATCH_SIZE,
    "lora_rank": S2B_REDO_LORA_RANK,
    "lora_alpha": S2B_REDO_LORA_ALPHA,
    "lora_lr": S2B_REDO_LORA_LR,
    "head_lr": S2B_REDO_HEAD_LR,
    "weight_decay": S2B_REDO_WEIGHT_DECAY,
    "max_grad_norm": S2B_REDO_GRAD_CLIP_NORM,
    "warmup_steps": S2B_REDO_WARMUP_STEPS,
    "planned_optimizer_steps": S2B_REDO_TOTAL_STEPS,
    "positive_weight_rule": "clip((negative/positive)**0.25, 2, 12)",
    "diagnostic_loss_scale": S2B_REDO_DIAGNOSTIC_SCALE,
    "optimizer_step_performed": False,
}

S2B_REDO_TRAINING_CONFIG_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_training_config.json"
)

with S2B_REDO_TRAINING_CONFIG_PATH.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        S2B_REDO_TRAINING_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("=" * 76)
print("S2 REDO — TRAINING CONFIGURATION VERIFIED")
print("=" * 76)
print("Trainable LoRA parameters:",
      f"{sum(p.numel() for p in S2B_REDO_LORA_PARAMS):,}")
print("Trainable task-head parameters:",
      f"{sum(p.numel() for p in S2B_REDO_HEAD_PARAMS):,}")
print("Masked boundary loss:", float(loss.detach().cpu().item()))

print("\nGradient norms (unscaled equivalents):")
for layer_idx, (norm_a, norm_b) in enumerate(zip(
    S2B_REDO_LORA_A_GRAD_NORMS,
    S2B_REDO_LORA_B_GRAD_NORMS,
)):
    print(
        f"  Encoder layer {layer_idx}: "
        f"A={norm_a:.8e}, B={norm_b:.8e}"
    )

print("Maximum task-head gradient norm:",
      f"{max(S2B_REDO_HEAD_GRAD_NORMS):.8e}")
print("LoRA B gradient flow: PASS")
print("Task-head gradient flow: PASS")
print("Optimizer steps performed: 0")
print("Training configuration saved:", S2B_REDO_TRAINING_CONFIG_PATH)
print("\nCell 6 completed. Training has not started.")

S2 REDO — TRAINING CONFIGURATION VERIFIED
Trainable LoRA parameters: 147,456
Trainable task-head parameters: 315,640
Masked boundary loss: 0.6924465894699097

Gradient norms (unscaled equivalents):
  Encoder layer 0: A=0.00000000e+00, B=8.93346034e-04
  Encoder layer 1: A=0.00000000e+00, B=3.14919016e-05
  Encoder layer 2: A=0.00000000e+00, B=5.01660252e-05
  Encoder layer 3: A=0.00000000e+00, B=4.76659116e-05
  Encoder layer 4: A=0.00000000e+00, B=2.58964610e-05
  Encoder layer 5: A=0.00000000e+00, B=2.08814799e-05
Maximum task-head gradient norm: 3.76856536e-01
LoRA B gradient flow: PASS
Task-head gradient flow: PASS
Optimizer steps performed: 0
Training configuration saved: /kaggle/working/S2_Architecture_B_Redo/S2B_training_config.json

Cell 6 completed. Training has not started.


In [7]:
# S2 Redo — Cell 7: Train Architecture B + LoRA with robust mixed precision,
# validation Strict Span Macro-F1, early stopping, and best-checkpoint restore.

import json
import math
import time
import numpy as np
import torch
from pathlib import Path

# ============================================================
# 1. Verify sequential prerequisites
# ============================================================
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_DEVICE",
    "S2B_REDO_OPTIMIZER",
    "S2B_REDO_SCHEDULER",
    "S2B_REDO_SCALER",
    "S2B_REDO_TRAIN_LOADER",
    "S2B_REDO_VAL_LOADER",
    "S2B_REDO_TRAIN_WINDOWS",
    "S2B_REDO_VAL_WINDOWS",
    "S2B_REDO_TRAIN_TEXTS",
    "S2B_REDO_VAL_TEXTS",
    "S2B_REDO_DOC_SPANS",
    "S2B_REDO_LABELS",
    "S2B_REDO_LABEL_TO_ID",
    "S2B_REDO_NUM_LABELS",
    "S2B_REDO_POS_WEIGHT_TENSOR",
    "S2B_REDO_MAX_EPOCHS",
    "S2B_REDO_GRAD_CLIP_NORM",
    "s2b_redo_boundary_loss",
    "S2B_REDO_OUTPUT_DIR",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cells 1–6 in order. Missing prerequisites: "
        f"{missing}"
    )

if globals().get("S2B_REDO_CELL7_STARTED", False):
    raise RuntimeError(
        "Cell 7 has already started in this notebook session. "
        "If it was interrupted, restart the session and run Cells 1–6 "
        "again before rerunning Cell 7."
    )

S2B_REDO_CELL7_STARTED = True

if globals().get("S2B_REDO_TRAINING_COMPLETED", False):
    raise RuntimeError(
        "Cell 7 was already completed. Do not retrain accidentally."
    )

# ============================================================
# 2. Configure output paths
# ============================================================
S2B_REDO_OUTPUT_DIR = Path(S2B_REDO_OUTPUT_DIR)
S2B_REDO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2B_REDO_BEST_CHECKPOINT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_LoRA_best_trainable.pt"
)
S2B_REDO_HISTORY_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_training_history.json"
)

# ============================================================
# 3. Set validation-only span caps and threshold candidates
# ============================================================
# Span caps are based on labelled training and validation annotations.
# Fold 3 gold annotations are never loaded or used here.

S2B_REDO_SPAN_LENGTHS_BY_LABEL = {
    label: [] for label in S2B_REDO_LABELS
}

for doc_id in (
    list(S2B_REDO_TRAIN_TEXTS.keys())
    + list(S2B_REDO_VAL_TEXTS.keys())
):
    for start, end, label, _ann_idx in S2B_REDO_DOC_SPANS[doc_id]:
        S2B_REDO_SPAN_LENGTHS_BY_LABEL[label].append(end - start)

S2B_REDO_SPAN_CAPS = {}

for label in S2B_REDO_LABELS:
    lengths = S2B_REDO_SPAN_LENGTHS_BY_LABEL[label]

    if not lengths:
        raise ValueError(f"No annotated spans available for {label}.")

    S2B_REDO_SPAN_CAPS[label] = max(
        24,
        min(256, int(math.ceil(max(lengths) * 1.10))),
    )

S2B_REDO_THRESHOLD_GRID = [
    0.001, 0.002, 0.003, 0.005, 0.008, 0.012,
    0.020, 0.030, 0.050, 0.080, 0.120, 0.180,
    0.250, 0.350, 0.500, 0.650, 0.800,
]

S2B_REDO_MAX_BOUNDARY_CANDIDATES = 384
S2B_REDO_MAX_END_OPTIONS = 16

print("=" * 76)
print("S2 REDO — TRAINING SETUP")
print("=" * 76)
print("Maximum epochs:", S2B_REDO_MAX_EPOCHS)
print("Training steps per epoch:", len(S2B_REDO_TRAIN_LOADER))
print("Validation batches:", len(S2B_REDO_VAL_LOADER))
print("Boundary candidate limit:", S2B_REDO_MAX_BOUNDARY_CANDIDATES)
print("Maximum end options per start:", S2B_REDO_MAX_END_OPTIONS)

print("\nLabel-specific span caps:")
for label in S2B_REDO_LABELS:
    print(f"  {label:12s}: {S2B_REDO_SPAN_CAPS[label]} characters")

# ============================================================
# 4. Initialize boundary-head biases using training statistics
# ============================================================
# This initialization uses only training-boundary counts.
# It is performed once at the start of this fresh experiment.

with torch.no_grad():
    positives = S2B_REDO_START_POSITIVE_COUNTS.astype(np.float64)
    valid_positions = S2B_REDO_VALID_CHAR_LABEL_COUNTS.astype(np.float64)
    negatives = np.maximum(valid_positions - positives, 1.0)

    weighted_odds = (
        S2B_REDO_POS_WEIGHTS
        * positives
        / negatives
    )

    initial_bias = np.clip(
        np.log(np.maximum(weighted_odds, 1e-7)),
        -12.0,
        -2.0,
    )

    initial_bias_tensor = torch.tensor(
        initial_bias,
        dtype=S2B_REDO_MODEL.start_head.bias.dtype,
        device=S2B_REDO_MODEL.start_head.bias.device,
    )

    S2B_REDO_MODEL.start_head.bias.copy_(initial_bias_tensor)
    S2B_REDO_MODEL.end_head.bias.copy_(initial_bias_tensor)

print("\nBoundary-head biases initialized from training data only.")

# ============================================================
# 5. Validation inference with overlapping-window aggregation
# ============================================================
@torch.no_grad()
def s2b_redo_predict_document_probabilities():
    """
    Average window logits at original character offsets, then convert
    them to probabilities for each validation document.
    """
    model = S2B_REDO_MODEL
    model.eval()

    text_mapping = S2B_REDO_VAL_TEXTS

    start_sums = {
        doc_id: np.zeros(
            (len(text_value), S2B_REDO_NUM_LABELS),
            dtype=np.float32,
        )
        for doc_id, text_value in text_mapping.items()
    }

    end_sums = {
        doc_id: np.zeros(
            (len(text_value), S2B_REDO_NUM_LABELS),
            dtype=np.float32,
        )
        for doc_id, text_value in text_mapping.items()
    }

    coverage_counts = {
        doc_id: np.zeros(
            len(text_value),
            dtype=np.int32,
        )
        for doc_id, text_value in text_mapping.items()
    }

    for batch in S2B_REDO_VAL_LOADER:
        device_batch = {
            key: batch[key].to(
                S2B_REDO_DEVICE,
                non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
            )
            for key in [
                "input_ids",
                "attention_mask",
                "char_ids",
                "char_to_token",
            ]
        }

        with torch.autocast(
            device_type=S2B_REDO_DEVICE.type,
            dtype=torch.float16,
            enabled=(S2B_REDO_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = model(
                input_ids=device_batch["input_ids"],
                attention_mask=device_batch["attention_mask"],
                char_ids=device_batch["char_ids"],
                char_to_token=device_batch["char_to_token"],
            )

        start_values = start_logits.float().cpu().numpy()
        end_values = end_logits.float().cpu().numpy()

        for batch_idx, doc_id in enumerate(batch["doc_ids"]):
            char_length = int(batch["char_lengths"][batch_idx])
            char_start = int(batch["char_starts"][batch_idx])
            char_end = int(batch["char_ends"][batch_idx])

            if char_end - char_start != char_length:
                raise RuntimeError(
                    f"Window length mismatch in document {doc_id}."
                )

            if not (
                0 <= char_start < char_end <= len(text_mapping[doc_id])
            ):
                raise RuntimeError(
                    f"Invalid character range in validation document {doc_id}."
                )

            start_sums[doc_id][char_start:char_end] += (
                start_values[batch_idx, :char_length]
            )
            end_sums[doc_id][char_start:char_end] += (
                end_values[batch_idx, :char_length]
            )
            coverage_counts[doc_id][char_start:char_end] += 1

    probabilities = {}
    uncovered_counts = {}

    for doc_id, text_value in text_mapping.items():
        counts = coverage_counts[doc_id]
        covered = counts > 0

        uncovered_counts[doc_id] = int((~covered).sum())

        start_avg = np.full(
            (len(text_value), S2B_REDO_NUM_LABELS),
            -25.0,
            dtype=np.float32,
        )
        end_avg = np.full_like(start_avg, -25.0)

        start_avg[covered] = (
            start_sums[doc_id][covered]
            / counts[covered, None]
        )
        end_avg[covered] = (
            end_sums[doc_id][covered]
            / counts[covered, None]
        )

        start_avg = np.clip(start_avg, -30.0, 30.0)
        end_avg = np.clip(end_avg, -30.0, 30.0)

        probabilities[doc_id] = {
            "start": 1.0 / (1.0 + np.exp(-start_avg)),
            "end": 1.0 / (1.0 + np.exp(-end_avg)),
        }

    return probabilities, uncovered_counts

# ============================================================
# 6. Generate character-span candidates
# ============================================================
def s2b_redo_build_candidates(
    start_probs,
    end_probs,
    label_id,
    max_span_chars,
):
    start_values = start_probs[:, label_id]
    end_values = end_probs[:, label_id]
    min_threshold = min(S2B_REDO_THRESHOLD_GRID)

    start_ids = np.flatnonzero(start_values >= min_threshold)
    end_ids = np.flatnonzero(end_values >= min_threshold)

    if len(start_ids) > S2B_REDO_MAX_BOUNDARY_CANDIDATES:
        keep = np.argpartition(
            start_values[start_ids],
            -S2B_REDO_MAX_BOUNDARY_CANDIDATES,
        )[-S2B_REDO_MAX_BOUNDARY_CANDIDATES:]
        start_ids = start_ids[keep]

    if len(end_ids) > S2B_REDO_MAX_BOUNDARY_CANDIDATES:
        keep = np.argpartition(
            end_values[end_ids],
            -S2B_REDO_MAX_BOUNDARY_CANDIDATES,
        )[-S2B_REDO_MAX_BOUNDARY_CANDIDATES:]
        end_ids = end_ids[keep]

    candidates = []

    for start_pos in start_ids:
        allowed_ends = end_ids[
            (end_ids >= start_pos)
            & (end_ids < start_pos + max_span_chars)
        ]

        if not len(allowed_ends):
            continue

        if len(allowed_ends) > S2B_REDO_MAX_END_OPTIONS:
            keep = np.argpartition(
                end_values[allowed_ends],
                -S2B_REDO_MAX_END_OPTIONS,
            )[-S2B_REDO_MAX_END_OPTIONS:]
            allowed_ends = allowed_ends[keep]

        start_prob = float(start_values[start_pos])

        for end_pos in allowed_ends:
            end_prob = float(end_values[end_pos])
            confidence = math.sqrt(
                max(start_prob, 0.0) * max(end_prob, 0.0)
            )

            # end_pos is inclusive; saved end offset is exclusive.
            candidates.append((
                int(start_pos),
                int(end_pos) + 1,
                start_prob,
                end_prob,
                confidence,
            ))

    candidates.sort(key=lambda item: item[4], reverse=True)
    return candidates


def s2b_redo_decode_candidates(candidates, threshold):
    decoded_spans = []

    for start, end, start_prob, end_prob, confidence in candidates:
        if confidence < threshold:
            break

        if start_prob < threshold or end_prob < threshold:
            continue

        overlaps = any(
            not (end <= old_start or start >= old_end)
            for old_start, old_end in decoded_spans
        )

        if not overlaps:
            decoded_spans.append((start, end))

    return decoded_spans

# ============================================================
# 7. Tune independent thresholds on validation documents
# ============================================================
def s2b_redo_tune_validation_thresholds(probabilities):
    gold_by_label = {
        label: set()
        for label in S2B_REDO_LABELS
    }

    for doc_id in S2B_REDO_VAL_TEXTS:
        for start, end, label, _ann_idx in S2B_REDO_DOC_SPANS[doc_id]:
            gold_by_label[label].add((doc_id, start, end, label))

    candidates_by_doc_label = {}

    for doc_id, values in probabilities.items():
        candidates_by_doc_label[doc_id] = {}

        for label_id, label in enumerate(S2B_REDO_LABELS):
            candidates_by_doc_label[doc_id][label] = (
                s2b_redo_build_candidates(
                    values["start"],
                    values["end"],
                    label_id,
                    S2B_REDO_SPAN_CAPS[label],
                )
            )

    selected_thresholds = {}
    selected_label_metrics = {}
    selected_predictions = set()

    for label in S2B_REDO_LABELS:
        gold = gold_by_label[label]

        if not gold:
            raise RuntimeError(
                f"Validation contains no gold entities for {label}."
            )

        best_key = None
        best_threshold = None
        best_predictions = set()
        best_metrics = None

        for threshold in S2B_REDO_THRESHOLD_GRID:
            predictions = set()

            for doc_id in S2B_REDO_VAL_TEXTS:
                candidates = candidates_by_doc_label[doc_id][label]

                decoded = s2b_redo_decode_candidates(
                    candidates,
                    threshold,
                )

                for start, end in decoded:
                    predictions.add((doc_id, start, end, label))

            tp = len(predictions & gold)
            fp = len(predictions - gold)
            fn = len(gold - predictions)

            precision = tp / (tp + fp) if tp + fp else 0.0
            recall = tp / (tp + fn) if tp + fn else 0.0

            f1 = (
                2.0 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )

            # Rank by F1 first, then precision, fewer predictions,
            # and a higher threshold on exact ties.
            key = (f1, precision, -len(predictions), threshold)

            if best_key is None or key > best_key:
                best_key = key
                best_threshold = threshold
                best_predictions = predictions

                best_metrics = {
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    "tp": tp,
                    "fp": fp,
                    "fn": fn,
                    "predicted": len(predictions),
                    "gold": len(gold),
                }

        selected_thresholds[label] = float(best_threshold)
        selected_label_metrics[label] = best_metrics
        selected_predictions.update(best_predictions)

    macro_f1 = float(np.mean([
        selected_label_metrics[label]["f1"]
        for label in S2B_REDO_LABELS
    ]))

    all_gold = set().union(*gold_by_label.values())

    return {
        "macro_f1": macro_f1,
        "predicted_count": len(selected_predictions),
        "gold_count": len(all_gold),
        "exact_true_positives": len(selected_predictions & all_gold),
        "thresholds": selected_thresholds,
        "per_label": selected_label_metrics,
    }, selected_predictions

# ============================================================
# 8. Prepare checkpoint utilities and counters
# ============================================================
def s2b_redo_export_trainable_state():
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in S2B_REDO_MODEL.named_parameters()
        if parameter.requires_grad
    }


def s2b_redo_restore_trainable_state(state):
    named_parameters = dict(S2B_REDO_MODEL.named_parameters())

    unknown = set(state) - set(named_parameters)
    if unknown:
        raise RuntimeError(
            f"Unknown checkpoint parameters: {sorted(unknown)}"
        )

    with torch.no_grad():
        for name, saved_tensor in state.items():
            parameter = named_parameters[name]

            if tuple(parameter.shape) != tuple(saved_tensor.shape):
                raise RuntimeError(
                    f"Shape mismatch for checkpoint tensor {name}."
                )

            parameter.copy_(
                saved_tensor.to(
                    device=parameter.device,
                    dtype=parameter.dtype,
                )
            )


S2B_REDO_TRAINING_HISTORY = []
S2B_REDO_BEST_VALIDATION_F1 = -1.0
S2B_REDO_BEST_EPOCH = None
S2B_REDO_BEST_THRESHOLDS = None
S2B_REDO_BEST_VALIDATION_METRICS = None
S2B_REDO_BEST_STATE = None

S2B_REDO_EARLY_STOP_PATIENCE = 2
S2B_REDO_EPOCHS_WITHOUT_IMPROVEMENT = 0

S2B_REDO_SKIPPED_STEPS = 0
S2B_REDO_SUCCESSFUL_STEPS = 0

S2B_REDO_TRAINABLE_PARAMETERS = [
    parameter
    for parameter in S2B_REDO_MODEL.parameters()
    if parameter.requires_grad
]

# ============================================================
# 9. Training loop with robust mixed-precision gradient handling
# ============================================================
for epoch in range(1, S2B_REDO_MAX_EPOCHS + 1):
    epoch_start = time.time()
    S2B_REDO_MODEL.train()

    running_loss = 0.0
    batches_seen = 0
    successful_updates_this_epoch = 0
    skipped_updates_this_epoch = 0

    print("\n" + "=" * 76)
    print(f"S2 REDO — EPOCH {epoch}/{S2B_REDO_MAX_EPOCHS}")
    print("=" * 76)

    for batch_index, batch in enumerate(
        S2B_REDO_TRAIN_LOADER,
        start=1,
    ):
        device_batch = {
            key: batch[key].to(
                S2B_REDO_DEVICE,
                non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
            )
            for key in [
                "input_ids",
                "attention_mask",
                "char_ids",
                "char_to_token",
                "start_targets",
                "end_targets",
                "loss_mask",
            ]
        }

        S2B_REDO_OPTIMIZER.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=S2B_REDO_DEVICE.type,
            dtype=torch.float16,
            enabled=(S2B_REDO_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S2B_REDO_MODEL(
                input_ids=device_batch["input_ids"],
                attention_mask=device_batch["attention_mask"],
                char_ids=device_batch["char_ids"],
                char_to_token=device_batch["char_to_token"],
            )

        loss = s2b_redo_boundary_loss(
            start_logits=start_logits,
            end_logits=end_logits,
            start_targets=device_batch["start_targets"],
            end_targets=device_batch["end_targets"],
            loss_mask=device_batch["loss_mask"],
        )

        if not torch.isfinite(loss):
            S2B_REDO_SKIPPED_STEPS += 1
            skipped_updates_this_epoch += 1
            S2B_REDO_OPTIMIZER.zero_grad(set_to_none=True)

            if S2B_REDO_SCALER.is_enabled():
                S2B_REDO_SCALER.update(
                    new_scale=max(S2B_REDO_SCALER.get_scale() / 2.0, 1.0)
                )

            if skipped_updates_this_epoch <= 10:
                print(
                    f"Skipped non-finite loss at epoch {epoch}, "
                    f"batch {batch_index}."
                )

            continue

        running_loss += float(loss.detach().float().item())
        batches_seen += 1

        S2B_REDO_SCALER.scale(loss).backward()
        S2B_REDO_SCALER.unscale_(S2B_REDO_OPTIMIZER)

        # Explicitly verify every trainable gradient before clipping.
        gradients_finite = all(
            parameter.grad is None
            or torch.isfinite(parameter.grad).all().item()
            for parameter in S2B_REDO_TRAINABLE_PARAMETERS
        )

        if not gradients_finite:
            # unscale_ records FP16 overflows for GradScaler.
            # scaler.step skips the update when it has recorded an overflow.
            if S2B_REDO_SCALER.is_enabled():
                S2B_REDO_SCALER.step(S2B_REDO_OPTIMIZER)
                S2B_REDO_SCALER.update()
            else:
                # In non-AMP mode never step with non-finite gradients.
                pass

            S2B_REDO_OPTIMIZER.zero_grad(set_to_none=True)
            S2B_REDO_SKIPPED_STEPS += 1
            skipped_updates_this_epoch += 1

            if skipped_updates_this_epoch <= 10:
                scale_display = (
                    f"{S2B_REDO_SCALER.get_scale():.0f}"
                    if S2B_REDO_SCALER.is_enabled()
                    else "disabled"
                )
                print(
                    f"Skipped non-finite gradient update at epoch {epoch}, "
                    f"batch {batch_index}; AMP scale={scale_display}."
                )

            continue

        # Compute the global gradient norm in float64 to avoid an FP32
        # overflow while measuring otherwise finite gradients.
        squared_norm = torch.zeros(
            (),
            dtype=torch.float64,
            device=S2B_REDO_DEVICE,
        )

        for parameter in S2B_REDO_TRAINABLE_PARAMETERS:
            if parameter.grad is not None:
                squared_norm += (
                    parameter.grad.detach()
                    .to(torch.float64)
                    .square()
                    .sum()
                )

        total_grad_norm = torch.sqrt(squared_norm)

        if not torch.isfinite(total_grad_norm):
            S2B_REDO_OPTIMIZER.zero_grad(set_to_none=True)
            S2B_REDO_SKIPPED_STEPS += 1
            skipped_updates_this_epoch += 1

            if S2B_REDO_SCALER.is_enabled():
                S2B_REDO_SCALER.update(
                    new_scale=max(S2B_REDO_SCALER.get_scale() / 2.0, 1.0)
                )

            if skipped_updates_this_epoch <= 10:
                print(
                    f"Skipped overflowing gradient norm at epoch {epoch}, "
                    f"batch {batch_index}."
                )

            continue

        norm_value = float(total_grad_norm.item())
        clip_coefficient = min(
            1.0,
            S2B_REDO_GRAD_CLIP_NORM / (norm_value + 1e-12),
        )

        if clip_coefficient < 1.0:
            for parameter in S2B_REDO_TRAINABLE_PARAMETERS:
                if parameter.grad is not None:
                    parameter.grad.mul_(clip_coefficient)

        # One optimizer/scaler step per successful batch.
        S2B_REDO_SCALER.step(S2B_REDO_OPTIMIZER)
        S2B_REDO_SCALER.update()
        S2B_REDO_SCHEDULER.step()

        successful_updates_this_epoch += 1
        S2B_REDO_SUCCESSFUL_STEPS += 1

        if (
            batch_index % 100 == 0
            or batch_index == len(S2B_REDO_TRAIN_LOADER)
        ):
            average_loss_so_far = (
                running_loss / max(1, batches_seen)
            )
            current_lrs = [
                group["lr"]
                for group in S2B_REDO_OPTIMIZER.param_groups
            ]

            print(
                f"Epoch {epoch} | batch {batch_index:3d}/"
                f"{len(S2B_REDO_TRAIN_LOADER)} | "
                f"loss={average_loss_so_far:.5f} | "
                f"LoRA LR={current_lrs[0]:.2e} | "
                f"head LR={current_lrs[1]:.2e} | "
                f"skipped={S2B_REDO_SKIPPED_STEPS}"
            )

    if batches_seen == 0 or successful_updates_this_epoch == 0:
        raise RuntimeError(
            f"Epoch {epoch} produced no successful optimizer updates. "
            "Stop and inspect the numerical stability of the model."
        )

    train_loss = running_loss / batches_seen

    # ========================================================
    # 10. Validate this epoch
    # ========================================================
    S2B_REDO_VAL_PROBABILITIES, S2B_REDO_VAL_UNCOVERED_CHARACTERS = (
        s2b_redo_predict_document_probabilities()
    )

    S2B_REDO_EPOCH_METRICS, S2B_REDO_EPOCH_PREDICTIONS = (
        s2b_redo_tune_validation_thresholds(
            S2B_REDO_VAL_PROBABILITIES
        )
    )

    epoch_seconds = time.time() - epoch_start

    epoch_record = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_macro_f1": float(
            S2B_REDO_EPOCH_METRICS["macro_f1"]
        ),
        "validation_predicted_count": int(
            S2B_REDO_EPOCH_METRICS["predicted_count"]
        ),
        "validation_gold_count": int(
            S2B_REDO_EPOCH_METRICS["gold_count"]
        ),
        "validation_exact_true_positives": int(
            S2B_REDO_EPOCH_METRICS["exact_true_positives"]
        ),
        "validation_uncovered_characters": int(
            sum(S2B_REDO_VAL_UNCOVERED_CHARACTERS.values())
        ),
        "epoch_seconds": float(epoch_seconds),
        "successful_updates": int(successful_updates_this_epoch),
        "skipped_updates_this_epoch": int(skipped_updates_this_epoch),
        "skipped_updates": int(S2B_REDO_SKIPPED_STEPS),
        "per_label": S2B_REDO_EPOCH_METRICS["per_label"],
        "thresholds": S2B_REDO_EPOCH_METRICS["thresholds"],
    }

    S2B_REDO_TRAINING_HISTORY.append(epoch_record)

    print("\nValidation summary:")
    print(f"  Training loss: {train_loss:.6f}")
    print(
        "  Strict Span Macro-F1:",
        f"{S2B_REDO_EPOCH_METRICS['macro_f1']:.6f}",
    )
    print(
        "  Predicted spans:",
        S2B_REDO_EPOCH_METRICS["predicted_count"],
    )
    print(
        "  Exact true positives:",
        S2B_REDO_EPOCH_METRICS["exact_true_positives"],
    )
    print("  Successful optimizer updates:", successful_updates_this_epoch)
    print("  Skipped updates this epoch:", skipped_updates_this_epoch)
    print("  Cumulative skipped updates:", S2B_REDO_SKIPPED_STEPS)
    print("  Epoch seconds:", f"{epoch_seconds:.1f}")

    print("\nPer-label validation results:")
    print(
        f"{'Label':12s} {'P':>7s} {'R':>7s} {'F1':>7s} "
        f"{'TP':>6s} {'FP':>6s} {'FN':>6s} {'Thr':>7s}"
    )

    for label in S2B_REDO_LABELS:
        metrics = S2B_REDO_EPOCH_METRICS["per_label"][label]
        threshold = S2B_REDO_EPOCH_METRICS["thresholds"][label]

        print(
            f"{label:12s} "
            f"{metrics['precision']:7.3f} "
            f"{metrics['recall']:7.3f} "
            f"{metrics['f1']:7.3f} "
            f"{metrics['tp']:6d} "
            f"{metrics['fp']:6d} "
            f"{metrics['fn']:6d} "
            f"{threshold:7.3f}"
        )

    # ========================================================
    # 11. Save the best checkpoint
    # ========================================================
    score = float(S2B_REDO_EPOCH_METRICS["macro_f1"])

    if score > S2B_REDO_BEST_VALIDATION_F1 + 1e-6:
        S2B_REDO_BEST_VALIDATION_F1 = score
        S2B_REDO_BEST_EPOCH = epoch
        S2B_REDO_BEST_THRESHOLDS = dict(
            S2B_REDO_EPOCH_METRICS["thresholds"]
        )
        S2B_REDO_BEST_VALIDATION_METRICS = S2B_REDO_EPOCH_METRICS
        S2B_REDO_BEST_STATE = s2b_redo_export_trainable_state()
        S2B_REDO_EPOCHS_WITHOUT_IMPROVEMENT = 0

        torch.save(
            {
                "experiment": "S2 Architecture B + LoRA",
                "best_epoch": epoch,
                "best_validation_macro_f1": score,
                "trainable_state": S2B_REDO_BEST_STATE,
                "thresholds": S2B_REDO_BEST_THRESHOLDS,
                "span_caps": S2B_REDO_SPAN_CAPS,
                "labels": S2B_REDO_LABELS,
                "lora_rank": S2B_REDO_LORA_RANK,
                "lora_alpha": S2B_REDO_LORA_ALPHA,
                "architecture_b_state_dict_path": str(
                    S2B_REDO_STATE_DICT_PATH
                ),
                "tokenizer_path": str(S2B_REDO_TOKENIZER_PATH),
                "fold3_gold_used": False,
            },
            S2B_REDO_BEST_CHECKPOINT_PATH,
        )

        print("\nBest checkpoint updated:")
        print(" ", S2B_REDO_BEST_CHECKPOINT_PATH)

    else:
        S2B_REDO_EPOCHS_WITHOUT_IMPROVEMENT += 1
        print(
            "\nNo new best checkpoint. Consecutive non-improving epochs:",
            S2B_REDO_EPOCHS_WITHOUT_IMPROVEMENT,
        )

    # Persist history after every epoch, including the skipped-update count.
    with S2B_REDO_HISTORY_PATH.open("w", encoding="utf-8") as f:
        json.dump(
            S2B_REDO_TRAINING_HISTORY,
            f,
            indent=2,
            ensure_ascii=False,
        )

    # Stop early if validation stops improving.
    if (
        S2B_REDO_EPOCHS_WITHOUT_IMPROVEMENT
        >= S2B_REDO_EARLY_STOP_PATIENCE
    ):
        print(
            "\nEarly stopping: validation Strict Span Macro-F1 "
            "did not improve for "
            f"{S2B_REDO_EARLY_STOP_PATIENCE} consecutive epochs."
        )
        break

# ============================================================
# 12. Restore the best trainable weights, not the final epoch
# ============================================================
if S2B_REDO_BEST_STATE is None:
    raise RuntimeError(
        "No valid best checkpoint was produced during training."
    )

s2b_redo_restore_trainable_state(S2B_REDO_BEST_STATE)
S2B_REDO_MODEL.eval()

S2B_REDO_TRAINING_COMPLETED = True

print("\n" + "=" * 76)
print("S2 REDO — TRAINING FINISHED")
print("=" * 76)
print("Best epoch:", S2B_REDO_BEST_EPOCH)
print(
    "Best validation Strict Span Macro-F1:",
    f"{S2B_REDO_BEST_VALIDATION_F1:.6f}",
)
print(
    "Best validation predicted spans:",
    S2B_REDO_BEST_VALIDATION_METRICS["predicted_count"],
)
print("Successful optimizer updates:", S2B_REDO_SUCCESSFUL_STEPS)
print("Cumulative skipped updates:", S2B_REDO_SKIPPED_STEPS)
print("Best checkpoint:", S2B_REDO_BEST_CHECKPOINT_PATH)
print("Training history:", S2B_REDO_HISTORY_PATH)
print("Best trainable weights restored: YES")
print("Architecture C/S3 weights used: NO")
print("Fold 3 gold labels used: NO")
print("\nCell 7 completed.")

S2 REDO — TRAINING SETUP
Maximum epochs: 5
Training steps per epoch: 498
Validation batches: 100
Boundary candidate limit: 384
Maximum end options per start: 16

Label-specific span caps:
  APP         : 51 characters
  RESP        : 92 characters
  A.COUNSEL   : 29 characters
  R.COUNSEL   : 32 characters
  JUDGE       : 36 characters
  WIT         : 26 characters
  AUTH        : 98 characters
  COURT       : 65 characters
  STAT        : 256 characters
  PREC        : 215 characters
  DATE        : 31 characters
  CASENO      : 101 characters

Boundary-head biases initialized from training data only.

S2 REDO — EPOCH 1/5
Epoch 1 | batch 100/498 | loss=0.01433 | LoRA LR=1.02e-04 | head LR=3.39e-04 | skipped=0
Epoch 1 | batch 200/498 | loss=0.01347 | LoRA LR=1.50e-04 | head LR=4.99e-04 | skipped=0
Epoch 1 | batch 300/498 | loss=0.01211 | LoRA LR=1.48e-04 | head LR=4.95e-04 | skipped=0
Epoch 1 | batch 400/498 | loss=0.01102 | LoRA LR=1.46e-04 | head LR=4.86e-04 | skipped=0
Epoch 1 | bat

In [8]:
# S2 Redo — Cell 8: Second-stage Architecture B + LoRA fine-tuning
# with stronger positive-boundary weighting and a conservative learning rate.

import json
import math
import time
import numpy as np
import torch
from pathlib import Path
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# ============================================================
# 1. Verify all sequential prerequisites
# ============================================================
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_DEVICE",
    "S2B_REDO_TRAIN_LOADER",
    "S2B_REDO_VAL_LOADER",
    "S2B_REDO_TRAIN_TEXTS",
    "S2B_REDO_VAL_TEXTS",
    "S2B_REDO_LABELS",
    "S2B_REDO_NUM_LABELS",
    "S2B_REDO_POS_WEIGHTS",
    "S2B_REDO_TRAINABLE_PARAMETERS",
    "S2B_REDO_LORA_PARAMS",
    "S2B_REDO_HEAD_PARAMS",
    "S2B_REDO_BEST_STATE",
    "S2B_REDO_BEST_CHECKPOINT_PATH",
    "S2B_REDO_BEST_VALIDATION_F1",
    "S2B_REDO_BEST_THRESHOLDS",
    "S2B_REDO_SPAN_CAPS",
    "S2B_REDO_OUTPUT_DIR",
    "s2b_redo_boundary_loss",
    "s2b_redo_export_trainable_state",
    "s2b_redo_restore_trainable_state",
    "s2b_redo_predict_document_probabilities",
    "s2b_redo_tune_validation_thresholds",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cells 1–7 in order. Missing prerequisites: "
        f"{missing}"
    )

if globals().get("S2B_REDO_STAGE2_COMPLETED", False):
    raise RuntimeError(
        "The second-stage experiment already completed in this session."
    )

if S2B_REDO_BEST_STATE is None:
    raise RuntimeError(
        "The first-stage best state is missing. Do not train from "
        "unknown in-memory weights."
    )

# ============================================================
# 2. Restore the first-stage best state and measure its baseline
# ============================================================
s2b_redo_restore_trainable_state(S2B_REDO_BEST_STATE)
S2B_REDO_MODEL.eval()

S2B_REDO_STAGE2_BASELINE_STATE = (
    s2b_redo_export_trainable_state()
)

S2B_REDO_STAGE2_BASELINE_PROBABILITIES, (
    S2B_REDO_STAGE2_BASELINE_UNCOVERED
) = s2b_redo_predict_document_probabilities()

S2B_REDO_STAGE2_BASELINE_METRICS, (
    S2B_REDO_STAGE2_BASELINE_PREDICTIONS
) = s2b_redo_tune_validation_thresholds(
    S2B_REDO_STAGE2_BASELINE_PROBABILITIES
)

S2B_REDO_STAGE2_BASELINE_F1 = float(
    S2B_REDO_STAGE2_BASELINE_METRICS["macro_f1"]
)

S2B_REDO_STAGE2_BASELINE_THRESHOLDS = dict(
    S2B_REDO_STAGE2_BASELINE_METRICS["thresholds"]
)

S2B_REDO_STAGE2_BEST_STATE = (
    S2B_REDO_STAGE2_BASELINE_STATE
)
S2B_REDO_STAGE2_BEST_F1 = S2B_REDO_STAGE2_BASELINE_F1
S2B_REDO_STAGE2_BEST_EPOCH = 0
S2B_REDO_STAGE2_BEST_THRESHOLDS = (
    S2B_REDO_STAGE2_BASELINE_THRESHOLDS.copy()
)
S2B_REDO_STAGE2_BEST_METRICS = (
    S2B_REDO_STAGE2_BASELINE_METRICS
)

print("=" * 76)
print("S2 REDO — STAGE 2 BASELINE")
print("=" * 76)
print("First-stage best validation Macro-F1:",
      f"{S2B_REDO_STAGE2_BASELINE_F1:.6f}")
print("Baseline predictions:",
      S2B_REDO_STAGE2_BASELINE_METRICS["predicted_count"])
print("Baseline exact true positives:",
      S2B_REDO_STAGE2_BASELINE_METRICS["exact_true_positives"])

# ============================================================
# 3. Use stronger positive-boundary weighting
# ============================================================
S2B_REDO_STAGE2_POS_WEIGHT = 30.0

S2B_REDO_STAGE2_OLD_POS_WEIGHT_TENSOR = (
    S2B_REDO_POS_WEIGHT_TENSOR.detach().clone()
)

S2B_REDO_POS_WEIGHT_TENSOR = torch.full(
    (S2B_REDO_NUM_LABELS,),
    S2B_REDO_STAGE2_POS_WEIGHT,
    dtype=torch.float32,
    device=S2B_REDO_DEVICE,
)

# ============================================================
# 4. Rebuild an optimizer with lower learning rates
# ============================================================
S2B_REDO_STAGE2_MAX_EPOCHS = 3
S2B_REDO_STAGE2_LORA_LR = 3.0e-5
S2B_REDO_STAGE2_HEAD_LR = 8.0e-5
S2B_REDO_STAGE2_WEIGHT_DECAY = 0.01
S2B_REDO_STAGE2_TOTAL_STEPS = (
    S2B_REDO_STAGE2_MAX_EPOCHS * len(S2B_REDO_TRAIN_LOADER)
)
S2B_REDO_STAGE2_WARMUP_STEPS = max(
    1, int(0.05 * S2B_REDO_STAGE2_TOTAL_STEPS)
)

S2B_REDO_STAGE2_OPTIMIZER = AdamW(
    [
        {
            "params": S2B_REDO_LORA_PARAMS,
            "lr": S2B_REDO_STAGE2_LORA_LR,
            "name": "lora",
        },
        {
            "params": S2B_REDO_HEAD_PARAMS,
            "lr": S2B_REDO_STAGE2_HEAD_LR,
            "name": "ner_head",
        },
    ],
    weight_decay=S2B_REDO_STAGE2_WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

def s2b_redo_stage2_lr_multiplier(step_index):
    if step_index < S2B_REDO_STAGE2_WARMUP_STEPS:
        return max(
            0.1,
            (step_index + 1) / S2B_REDO_STAGE2_WARMUP_STEPS,
        )

    decay_steps = max(
        1,
        S2B_REDO_STAGE2_TOTAL_STEPS
        - S2B_REDO_STAGE2_WARMUP_STEPS,
    )

    progress = (
        step_index - S2B_REDO_STAGE2_WARMUP_STEPS
    ) / decay_steps
    progress = min(1.0, max(0.0, progress))

    return 0.5 * (1.0 + math.cos(math.pi * progress))

S2B_REDO_STAGE2_SCHEDULER = LambdaLR(
    S2B_REDO_STAGE2_OPTIMIZER,
    lr_lambda=s2b_redo_stage2_lr_multiplier,
)

# Start at a lower loss scale; update dynamically if overflows occur.
S2B_REDO_STAGE2_SCALER = torch.amp.GradScaler(
    "cuda",
    init_scale=4096.0,
    growth_interval=1000,
    enabled=(S2B_REDO_DEVICE.type == "cuda"),
)

S2B_REDO_STAGE2_HISTORY = []
S2B_REDO_STAGE2_SKIPPED_UPDATES = 0
S2B_REDO_STAGE2_SUCCESSFUL_UPDATES = 0

S2B_REDO_STAGE2_CHECKPOINT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_LoRA_reweighted_stage2_best.pt"
)
S2B_REDO_STAGE2_HISTORY_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_reweighted_stage2_history.json"
)

print("\nStage 2 configuration:")
print("Epochs:", S2B_REDO_STAGE2_MAX_EPOCHS)
print("Positive weight:", S2B_REDO_STAGE2_POS_WEIGHT)
print("LoRA learning rate:", S2B_REDO_STAGE2_LORA_LR)
print("Head learning rate:", S2B_REDO_STAGE2_HEAD_LR)
print("Warmup steps:", S2B_REDO_STAGE2_WARMUP_STEPS)
print("The first-stage checkpoint will not be overwritten.")

# ============================================================
# 5. Train the reweighted model with finite-gradient checks
# ============================================================
for epoch in range(1, S2B_REDO_STAGE2_MAX_EPOCHS + 1):
    epoch_start = time.time()
    S2B_REDO_MODEL.train()

    running_loss = 0.0
    batches_with_finite_loss = 0
    successful_this_epoch = 0
    skipped_this_epoch = 0

    print("\n" + "=" * 76)
    print(
        f"S2 REDO — REWEIGHTED STAGE 2, "
        f"EPOCH {epoch}/{S2B_REDO_STAGE2_MAX_EPOCHS}"
    )
    print("=" * 76)

    for batch_index, batch in enumerate(
        S2B_REDO_TRAIN_LOADER,
        start=1,
    ):
        device_batch = {
            key: batch[key].to(
                S2B_REDO_DEVICE,
                non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
            )
            for key in [
                "input_ids",
                "attention_mask",
                "char_ids",
                "char_to_token",
                "start_targets",
                "end_targets",
                "loss_mask",
            ]
        }

        S2B_REDO_STAGE2_OPTIMIZER.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=S2B_REDO_DEVICE.type,
            dtype=torch.float16,
            enabled=(S2B_REDO_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S2B_REDO_MODEL(
                input_ids=device_batch["input_ids"],
                attention_mask=device_batch["attention_mask"],
                char_ids=device_batch["char_ids"],
                char_to_token=device_batch["char_to_token"],
            )

        loss = s2b_redo_boundary_loss(
            start_logits=start_logits,
            end_logits=end_logits,
            start_targets=device_batch["start_targets"],
            end_targets=device_batch["end_targets"],
            loss_mask=device_batch["loss_mask"],
        )

        if not torch.isfinite(loss):
            S2B_REDO_STAGE2_SKIPPED_UPDATES += 1
            skipped_this_epoch += 1
            S2B_REDO_STAGE2_OPTIMIZER.zero_grad(set_to_none=True)

            if skipped_this_epoch <= 5:
                print(
                    f"Skipped non-finite loss at batch {batch_index}."
                )
            continue

        running_loss += float(loss.detach().float().item())
        batches_with_finite_loss += 1

        S2B_REDO_STAGE2_SCALER.scale(loss).backward()
        S2B_REDO_STAGE2_SCALER.unscale_(
            S2B_REDO_STAGE2_OPTIMIZER
        )

        gradients_finite = all(
            parameter.grad is None
            or torch.isfinite(parameter.grad).all().item()
            for parameter in S2B_REDO_TRAINABLE_PARAMETERS
        )

        if not gradients_finite:
            if S2B_REDO_STAGE2_SCALER.is_enabled():
                # GradScaler knows the unscaled gradients overflowed.
                # Its step call skips the parameter update.
                S2B_REDO_STAGE2_SCALER.step(
                    S2B_REDO_STAGE2_OPTIMIZER
                )
                S2B_REDO_STAGE2_SCALER.update()

            S2B_REDO_STAGE2_OPTIMIZER.zero_grad(set_to_none=True)
            S2B_REDO_STAGE2_SKIPPED_UPDATES += 1
            skipped_this_epoch += 1
            continue

        # Calculate norm in float64, then clip gradients manually.
        squared_norm = torch.zeros(
            (),
            dtype=torch.float64,
            device=S2B_REDO_DEVICE,
        )

        for parameter in S2B_REDO_TRAINABLE_PARAMETERS:
            if parameter.grad is not None:
                squared_norm += (
                    parameter.grad.detach()
                    .to(torch.float64)
                    .square()
                    .sum()
                )

        total_norm = torch.sqrt(squared_norm)

        if not torch.isfinite(total_norm):
            S2B_REDO_STAGE2_OPTIMIZER.zero_grad(set_to_none=True)
            S2B_REDO_STAGE2_SKIPPED_UPDATES += 1
            skipped_this_epoch += 1

            if S2B_REDO_STAGE2_SCALER.is_enabled():
                S2B_REDO_STAGE2_SCALER.update(
                    new_scale=max(
                        S2B_REDO_STAGE2_SCALER.get_scale() / 2.0,
                        1.0,
                    )
                )
            continue

        clip_factor = min(
            1.0,
            S2B_REDO_GRAD_CLIP_NORM
            / (float(total_norm.item()) + 1e-12),
        )

        if clip_factor < 1.0:
            for parameter in S2B_REDO_TRAINABLE_PARAMETERS:
                if parameter.grad is not None:
                    parameter.grad.mul_(clip_factor)

        S2B_REDO_STAGE2_SCALER.step(
            S2B_REDO_STAGE2_OPTIMIZER
        )
        S2B_REDO_STAGE2_SCALER.update()
        S2B_REDO_STAGE2_SCHEDULER.step()

        successful_this_epoch += 1
        S2B_REDO_STAGE2_SUCCESSFUL_UPDATES += 1

        if (
            batch_index % 100 == 0
            or batch_index == len(S2B_REDO_TRAIN_LOADER)
        ):
            average_loss = running_loss / max(
                1, batches_with_finite_loss
            )
            lrs = [
                group["lr"]
                for group in S2B_REDO_STAGE2_OPTIMIZER.param_groups
            ]
            print(
                f"Epoch {epoch} | batch {batch_index}/"
                f"{len(S2B_REDO_TRAIN_LOADER)} | "
                f"loss={average_loss:.5f} | "
                f"LoRA LR={lrs[0]:.2e} | "
                f"head LR={lrs[1]:.2e} | "
                f"skipped={S2B_REDO_STAGE2_SKIPPED_UPDATES}"
            )

    if successful_this_epoch == 0:
        raise RuntimeError(
            f"No successful optimizer updates in stage 2 epoch {epoch}."
        )

    train_loss = running_loss / max(1, batches_with_finite_loss)

    # --------------------------------------------------------
    # Validation and per-label threshold calibration
    # --------------------------------------------------------
    S2B_REDO_STAGE2_VAL_PROBABILITIES, (
        S2B_REDO_STAGE2_UNCOVERED_CHARACTERS
    ) = s2b_redo_predict_document_probabilities()

    S2B_REDO_STAGE2_METRICS, S2B_REDO_STAGE2_PREDICTIONS = (
        s2b_redo_tune_validation_thresholds(
            S2B_REDO_STAGE2_VAL_PROBABILITIES
        )
    )

    score = float(S2B_REDO_STAGE2_METRICS["macro_f1"])
    epoch_seconds = time.time() - epoch_start

    epoch_record = {
        "stage": 2,
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_macro_f1": score,
        "validation_predicted_count": int(
            S2B_REDO_STAGE2_METRICS["predicted_count"]
        ),
        "validation_exact_true_positives": int(
            S2B_REDO_STAGE2_METRICS["exact_true_positives"]
        ),
        "successful_updates_this_epoch": int(successful_this_epoch),
        "skipped_updates_this_epoch": int(skipped_this_epoch),
        "skipped_updates": int(S2B_REDO_STAGE2_SKIPPED_UPDATES),
        "epoch_seconds": float(epoch_seconds),
        "per_label": S2B_REDO_STAGE2_METRICS["per_label"],
        "thresholds": S2B_REDO_STAGE2_METRICS["thresholds"],
    }

    S2B_REDO_STAGE2_HISTORY.append(epoch_record)

    print("\nValidation summary:")
    print("  Training loss:", f"{train_loss:.6f}")
    print("  Strict Span Macro-F1:", f"{score:.6f}")
    print(
        "  Predicted spans:",
        S2B_REDO_STAGE2_METRICS["predicted_count"],
    )
    print(
        "  Exact true positives:",
        S2B_REDO_STAGE2_METRICS["exact_true_positives"],
    )
    print("  Successful updates this epoch:", successful_this_epoch)
    print("  Skipped updates this epoch:", skipped_this_epoch)
    print("  Cumulative skipped updates:", S2B_REDO_STAGE2_SKIPPED_UPDATES)
    print("  Epoch seconds:", f"{epoch_seconds:.1f}")

    print("\nPer-label results:")
    print(
        f"{'Label':12s} {'P':>7s} {'R':>7s} {'F1':>7s} "
        f"{'TP':>6s} {'FP':>6s} {'FN':>6s} {'Thr':>7s}"
    )

    for label in S2B_REDO_LABELS:
        m = S2B_REDO_STAGE2_METRICS["per_label"][label]
        threshold = S2B_REDO_STAGE2_METRICS["thresholds"][label]

        print(
            f"{label:12s} "
            f"{m['precision']:7.3f} "
            f"{m['recall']:7.3f} "
            f"{m['f1']:7.3f} "
            f"{m['tp']:6d} "
            f"{m['fp']:6d} "
            f"{m['fn']:6d} "
            f"{threshold:7.3f}"
        )

    # Promote a stage-2 checkpoint only if it beats the existing baseline.
    if score > S2B_REDO_STAGE2_BEST_F1 + 1e-6:
        S2B_REDO_STAGE2_BEST_F1 = score
        S2B_REDO_STAGE2_BEST_EPOCH = epoch
        S2B_REDO_STAGE2_BEST_STATE = (
            s2b_redo_export_trainable_state()
        )
        S2B_REDO_STAGE2_BEST_THRESHOLDS = dict(
            S2B_REDO_STAGE2_METRICS["thresholds"]
        )
        S2B_REDO_STAGE2_BEST_METRICS = (
            S2B_REDO_STAGE2_METRICS
        )

        torch.save(
            {
                "experiment": "S2 Architecture B + LoRA reweighted stage 2",
                "best_epoch": epoch,
                "best_validation_macro_f1": score,
                "trainable_state": S2B_REDO_STAGE2_BEST_STATE,
                "thresholds": S2B_REDO_STAGE2_BEST_THRESHOLDS,
                "span_caps": S2B_REDO_SPAN_CAPS,
                "labels": S2B_REDO_LABELS,
                "lora_rank": S2B_REDO_LORA_RANK,
                "lora_alpha": S2B_REDO_LORA_ALPHA,
                "positive_weight": S2B_REDO_STAGE2_POS_WEIGHT,
                "fold3_gold_used": False,
            },
            S2B_REDO_STAGE2_CHECKPOINT_PATH,
        )

        print("\nStage-2 checkpoint improved the prior best:")
        print(" ", S2B_REDO_STAGE2_CHECKPOINT_PATH)
    else:
        print(
            "\nStage 2 did not exceed the best prior validation Macro-F1 "
            f"of {S2B_REDO_STAGE2_BEST_F1:.6f}."
        )

    with S2B_REDO_STAGE2_HISTORY_PATH.open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            S2B_REDO_STAGE2_HISTORY,
            f,
            indent=2,
            ensure_ascii=False,
        )

# ============================================================
# 6. Restore whichever model has the best validation score
# ============================================================
if S2B_REDO_STAGE2_BEST_STATE is None:
    raise RuntimeError("Neither baseline nor stage-2 state is available.")

s2b_redo_restore_trainable_state(
    S2B_REDO_STAGE2_BEST_STATE
)
S2B_REDO_MODEL.eval()

# Update the downstream best-model variables.
S2B_REDO_BEST_STATE = S2B_REDO_STAGE2_BEST_STATE
S2B_REDO_BEST_VALIDATION_F1 = S2B_REDO_STAGE2_BEST_F1
S2B_REDO_BEST_THRESHOLDS = (
    S2B_REDO_STAGE2_BEST_THRESHOLDS.copy()
)
S2B_REDO_BEST_VALIDATION_METRICS = (
    S2B_REDO_STAGE2_BEST_METRICS
)

S2B_REDO_STAGE2_IMPROVED = (
    S2B_REDO_STAGE2_BEST_F1
    > S2B_REDO_STAGE2_BASELINE_F1 + 1e-6
)

if S2B_REDO_STAGE2_IMPROVED:
    S2B_REDO_BEST_CHECKPOINT_PATH = (
        S2B_REDO_STAGE2_CHECKPOINT_PATH
    )

# Save a final selected checkpoint without overwriting stage 1.
S2B_REDO_FINAL_SELECTED_CHECKPOINT = (
    S2B_REDO_OUTPUT_DIR / "S2B_LoRA_final_selected.pt"
)

torch.save(
    {
        "experiment": "S2 Architecture B + LoRA",
        "best_validation_macro_f1": S2B_REDO_BEST_VALIDATION_F1,
        "trainable_state": S2B_REDO_BEST_STATE,
        "thresholds": S2B_REDO_BEST_THRESHOLDS,
        "span_caps": S2B_REDO_SPAN_CAPS,
        "labels": S2B_REDO_LABELS,
        "selected_stage": (
            2 if S2B_REDO_STAGE2_IMPROVED else 1
        ),
        "fold3_gold_used": False,
    },
    S2B_REDO_FINAL_SELECTED_CHECKPOINT,
)

S2B_REDO_STAGE2_CONFIG = {
    "positive_weight": S2B_REDO_STAGE2_POS_WEIGHT,
    "lora_learning_rate": S2B_REDO_STAGE2_LORA_LR,
    "head_learning_rate": S2B_REDO_STAGE2_HEAD_LR,
    "epochs": S2B_REDO_STAGE2_MAX_EPOCHS,
    "baseline_validation_macro_f1": S2B_REDO_STAGE2_BASELINE_F1,
    "best_validation_macro_f1": S2B_REDO_BEST_VALIDATION_F1,
    "stage2_improved_validation": S2B_REDO_STAGE2_IMPROVED,
    "stage2_best_epoch": S2B_REDO_STAGE2_BEST_EPOCH,
    "successful_updates": S2B_REDO_STAGE2_SUCCESSFUL_UPDATES,
    "skipped_updates": S2B_REDO_STAGE2_SKIPPED_UPDATES,
    "selected_checkpoint": str(S2B_REDO_FINAL_SELECTED_CHECKPOINT),
}

S2B_REDO_STAGE2_CONFIG_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_stage2_config.json"
)

with S2B_REDO_STAGE2_CONFIG_PATH.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        S2B_REDO_STAGE2_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
    )

S2B_REDO_STAGE2_COMPLETED = True

# Restore the original Stage-1 weights for loss configuration bookkeeping.
# The chosen model above has already been restored; only keep the stronger
# stage-2 weight tensor if Stage 2 actually wins.
if not S2B_REDO_STAGE2_IMPROVED:
    S2B_REDO_POS_WEIGHT_TENSOR = (
        S2B_REDO_STAGE2_OLD_POS_WEIGHT_TENSOR
    )

print("\n" + "=" * 76)
print("S2 REDO — STAGE 2 COMPLETED")
print("=" * 76)
print(
    "Baseline validation Macro-F1:",
    f"{S2B_REDO_STAGE2_BASELINE_F1:.6f}",
)
print(
    "Selected best validation Macro-F1:",
    f"{S2B_REDO_BEST_VALIDATION_F1:.6f}",
)
print("Stage 2 improved validation:", S2B_REDO_STAGE2_IMPROVED)
print("Stage 2 best epoch:", S2B_REDO_STAGE2_BEST_EPOCH)
print("Successful stage-2 updates:", S2B_REDO_STAGE2_SUCCESSFUL_UPDATES)
print("Skipped stage-2 updates:", S2B_REDO_STAGE2_SKIPPED_UPDATES)
print("Selected checkpoint:", S2B_REDO_FINAL_SELECTED_CHECKPOINT)
print("Stage-2 history:", S2B_REDO_STAGE2_HISTORY_PATH)
print("Architecture C/S3 weights used: NO")
print("Fold 3 gold labels used: NO")
print("\nCell 8 completed.")

S2 REDO — STAGE 2 BASELINE
First-stage best validation Macro-F1: 0.177442
Baseline predictions: 2596
Baseline exact true positives: 393

Stage 2 configuration:
Epochs: 3
Positive weight: 30.0
LoRA learning rate: 3e-05
Head learning rate: 8e-05
Warmup steps: 74
The first-stage checkpoint will not be overwritten.

S2 REDO — REWEIGHTED STAGE 2, EPOCH 1/3
Epoch 1 | batch 100/498 | loss=0.01506 | LoRA LR=3.00e-05 | head LR=7.99e-05 | skipped=0
Epoch 1 | batch 200/498 | loss=0.01495 | LoRA LR=2.94e-05 | head LR=7.85e-05 | skipped=0
Epoch 1 | batch 300/498 | loss=0.01499 | LoRA LR=2.82e-05 | head LR=7.51e-05 | skipped=0
Epoch 1 | batch 400/498 | loss=0.01502 | LoRA LR=2.63e-05 | head LR=7.00e-05 | skipped=0
Epoch 1 | batch 498/498 | loss=0.01490 | LoRA LR=2.39e-05 | head LR=6.37e-05 | skipped=0

Validation summary:
  Training loss: 0.014902
  Strict Span Macro-F1: 0.173017
  Predicted spans: 3761
  Exact true positives: 406
  Successful updates this epoch: 498
  Skipped updates this epoch: 0


In [9]:
# S2 Redo — Cell 9: Search per-label span caps and thresholds on validation only.

import json
import math
import numpy as np
from pathlib import Path

# ============================================================
# 1. Verify all required variables exist
# ============================================================
required = [
    "S2B_REDO_STAGE2_BASELINE_PROBABILITIES",
    "S2B_REDO_STAGE2_BASELINE_THRESHOLDS",
    "S2B_REDO_STAGE2_BASELINE_METRICS",
    "S2B_REDO_SPAN_CAPS",
    "S2B_REDO_SPAN_LENGTHS_BY_LABEL",
    "S2B_REDO_VAL_TEXTS",
    "S2B_REDO_DOC_SPANS",
    "S2B_REDO_LABELS",
    "S2B_REDO_LABEL_TO_ID",
    "S2B_REDO_THRESHOLD_GRID",
    "S2B_REDO_MAX_BOUNDARY_CANDIDATES",
    "S2B_REDO_MAX_END_OPTIONS",
    "S2B_REDO_OUTPUT_DIR",
    "s2b_redo_build_candidates",
    "s2b_redo_decode_candidates",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cells 1–8 in order. Missing variables: "
        f"{missing}"
    )

S2B_REDO_CALIBRATION_MAX_CAP = 512

# ============================================================
# 2. Build gold spans from the validation split only
# ============================================================
S2B_REDO_CALIBRATION_GOLD = {
    label: set()
    for label in S2B_REDO_LABELS
}

for doc_id in S2B_REDO_VAL_TEXTS:
    for start, end, label, _ann_idx in S2B_REDO_DOC_SPANS[doc_id]:
        S2B_REDO_CALIBRATION_GOLD[label].add(
            (doc_id, start, end, label)
        )

for label in S2B_REDO_LABELS:
    if not S2B_REDO_CALIBRATION_GOLD[label]:
        raise RuntimeError(
            f"Validation has no gold entities for label {label}."
        )

# ============================================================
# 3. Define candidate caps from the annotated training/validation data
# ============================================================
S2B_REDO_CAP_CANDIDATES = {}

for label in S2B_REDO_LABELS:
    lengths = np.asarray(
        S2B_REDO_SPAN_LENGTHS_BY_LABEL[label],
        dtype=np.int32,
    )

    p95 = float(np.percentile(lengths, 95))
    p99 = float(np.percentile(lengths, 99))
    observed_max = int(lengths.max())

    proposed = [
        max(20, int(math.ceil(1.25 * p95))),
        max(25, int(math.ceil(1.25 * p99))),
        min(
            256,
            int(math.ceil(1.10 * observed_max)),
        ),
        int(S2B_REDO_SPAN_CAPS[label]),
        S2B_REDO_CALIBRATION_MAX_CAP,
    ]

    S2B_REDO_CAP_CANDIDATES[label] = sorted(
        set(
            max(1, min(S2B_REDO_CALIBRATION_MAX_CAP, cap))
            for cap in proposed
        )
    )

print("=" * 76)
print("S2 REDO — SPAN-CAP CALIBRATION CANDIDATES")
print("=" * 76)

for label in S2B_REDO_LABELS:
    print(
        f"{label:12s}: {S2B_REDO_CAP_CANDIDATES[label]}"
    )

# ============================================================
# 4. Define exact per-label scoring helpers
# ============================================================
def s2b_redo_calibration_metrics(predictions, gold):
    tp = len(predictions & gold)
    fp = len(predictions - gold)
    fn = len(gold - predictions)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "predicted": len(predictions),
        "gold": len(gold),
    }


def s2b_redo_decode_one_label(
    probabilities,
    label,
    label_id,
    cap,
    threshold,
):
    predictions = set()

    for doc_id in S2B_REDO_VAL_TEXTS:
        values = probabilities[doc_id]

        candidates = s2b_redo_build_candidates(
            values["start"],
            values["end"],
            label_id,
            cap,
        )

        decoded = s2b_redo_decode_candidates(
            candidates,
            threshold,
        )

        for start, end in decoded:
            # Defensive offset and length check.
            if not (
                0 <= start < end <= len(S2B_REDO_VAL_TEXTS[doc_id])
            ):
                raise ValueError(
                    f"Invalid validation span for {doc_id}: {(start, end)}"
                )

            if end - start > cap:
                raise ValueError(
                    f"Span cap violated for label {label}."
                )

            predictions.add((doc_id, start, end, label))

    return predictions


# ============================================================
# 5. Reconstruct the original baseline before changing anything
# ============================================================
S2B_REDO_CALIBRATION_BASELINE_THRESHOLDS = dict(
    S2B_REDO_STAGE2_BASELINE_THRESHOLDS
)

S2B_REDO_CALIBRATION_BASELINE_CAPS = dict(
    S2B_REDO_SPAN_CAPS
)

S2B_REDO_CALIBRATION_BASELINE_PREDICTIONS = set()

for label in S2B_REDO_LABELS:
    label_id = S2B_REDO_LABEL_TO_ID[label]

    label_predictions = s2b_redo_decode_one_label(
        probabilities=S2B_REDO_STAGE2_BASELINE_PROBABILITIES,
        label=label,
        label_id=label_id,
        cap=S2B_REDO_CALIBRATION_BASELINE_CAPS[label],
        threshold=S2B_REDO_CALIBRATION_BASELINE_THRESHOLDS[label],
    )

    S2B_REDO_CALIBRATION_BASELINE_PREDICTIONS.update(
        label_predictions
    )

S2B_REDO_CALIBRATION_ALL_GOLD = set().union(
    *S2B_REDO_CALIBRATION_GOLD.values()
)

S2B_REDO_CALIBRATION_BASELINE_PER_LABEL = {
    label: s2b_redo_calibration_metrics(
        {
            p for p in S2B_REDO_CALIBRATION_BASELINE_PREDICTIONS
            if p[3] == label
        },
        S2B_REDO_CALIBRATION_GOLD[label],
    )
    for label in S2B_REDO_LABELS
}

S2B_REDO_CALIBRATION_BASELINE_F1 = float(np.mean([
    S2B_REDO_CALIBRATION_BASELINE_PER_LABEL[label]["f1"]
    for label in S2B_REDO_LABELS
]))

print("\n" + "=" * 76)
print("S2 REDO — BASELINE REPRODUCTION")
print("=" * 76)
print(
    "Previously reported baseline Macro-F1:",
    f"{S2B_REDO_STAGE2_BASELINE_METRICS['macro_f1']:.6f}",
)
print(
    "Reconstructed baseline Macro-F1:",
    f"{S2B_REDO_CALIBRATION_BASELINE_F1:.6f}",
)
print(
    "Reconstructed prediction count:",
    len(S2B_REDO_CALIBRATION_BASELINE_PREDICTIONS),
)

# ============================================================
# 6. Choose a span cap for each label using its saved baseline threshold
# ============================================================
S2B_REDO_CAP_SEARCH_RESULTS = {}
S2B_REDO_CALIBRATED_CAPS = {}

for label_id, label in enumerate(S2B_REDO_LABELS):
    gold = S2B_REDO_CALIBRATION_GOLD[label]
    old_threshold = S2B_REDO_CALIBRATION_BASELINE_THRESHOLDS[label]

    best_key = None
    best_cap = None
    best_result = None

    for cap in S2B_REDO_CAP_CANDIDATES[label]:
        predictions = s2b_redo_decode_one_label(
            probabilities=S2B_REDO_STAGE2_BASELINE_PROBABILITIES,
            label=label,
            label_id=label_id,
            cap=cap,
            threshold=old_threshold,
        )

        result = s2b_redo_calibration_metrics(predictions, gold)

        # Ties prefer higher precision, fewer predictions, then a shorter cap.
        rank_key = (
            result["f1"],
            result["precision"],
            -result["predicted"],
            -cap,
        )

        if best_key is None or rank_key > best_key:
            best_key = rank_key
            best_cap = cap
            best_result = result

    S2B_REDO_CALIBRATED_CAPS[label] = int(best_cap)

    S2B_REDO_CAP_SEARCH_RESULTS[label] = {
        "selected_cap": int(best_cap),
        "threshold_used": float(old_threshold),
        **best_result,
    }

print("\n" + "=" * 76)
print("S2 REDO — SELECTED SPAN CAPS")
print("=" * 76)

for label in S2B_REDO_LABELS:
    result = S2B_REDO_CAP_SEARCH_RESULTS[label]
    print(
        f"{label:12s} cap={result['selected_cap']:4d} "
        f"F1={result['f1']:.3f} "
        f"P={result['precision']:.3f} "
        f"R={result['recall']:.3f}"
    )

# ============================================================
# 7. Retune thresholds using the selected cap for each label
# ============================================================
S2B_REDO_CALIBRATED_THRESHOLDS = {}
S2B_REDO_THRESHOLD_SEARCH_RESULTS = {}
S2B_REDO_CALIBRATED_PREDICTIONS = set()

for label_id, label in enumerate(S2B_REDO_LABELS):
    gold = S2B_REDO_CALIBRATION_GOLD[label]
    selected_cap = S2B_REDO_CALIBRATED_CAPS[label]

    best_key = None
    best_threshold = None
    best_predictions = set()
    best_result = None

    for threshold in S2B_REDO_THRESHOLD_GRID:
        predictions = s2b_redo_decode_one_label(
            probabilities=S2B_REDO_STAGE2_BASELINE_PROBABILITIES,
            label=label,
            label_id=label_id,
            cap=selected_cap,
            threshold=float(threshold),
        )

        result = s2b_redo_calibration_metrics(predictions, gold)

        rank_key = (
            result["f1"],
            result["precision"],
            -result["predicted"],
            float(threshold),
        )

        if best_key is None or rank_key > best_key:
            best_key = rank_key
            best_threshold = float(threshold)
            best_predictions = predictions
            best_result = result

    S2B_REDO_CALIBRATED_THRESHOLDS[label] = best_threshold
    S2B_REDO_THRESHOLD_SEARCH_RESULTS[label] = {
        "threshold": best_threshold,
        "span_cap": selected_cap,
        **best_result,
    }
    S2B_REDO_CALIBRATED_PREDICTIONS.update(best_predictions)

S2B_REDO_CALIBRATED_PER_LABEL = {
    label: S2B_REDO_THRESHOLD_SEARCH_RESULTS[label]
    for label in S2B_REDO_LABELS
}

S2B_REDO_CALIBRATED_MACRO_F1 = float(np.mean([
    S2B_REDO_CALIBRATED_PER_LABEL[label]["f1"]
    for label in S2B_REDO_LABELS
]))

# ============================================================
# 8. Keep the original settings if calibration is worse
# ============================================================
S2B_REDO_CALIBRATION_IMPROVED = (
    S2B_REDO_CALIBRATED_MACRO_F1
    > S2B_REDO_CALIBRATION_BASELINE_F1 + 1e-8
)

if S2B_REDO_CALIBRATION_IMPROVED:
    S2B_REDO_SELECTED_THRESHOLDS = dict(
        S2B_REDO_CALIBRATED_THRESHOLDS
    )
    S2B_REDO_SELECTED_CAPS = dict(
        S2B_REDO_CALIBRATED_CAPS
    )
    S2B_REDO_SELECTED_PREDICTIONS = set(
        S2B_REDO_CALIBRATED_PREDICTIONS
    )
    S2B_REDO_SELECTED_MACRO_F1 = (
        S2B_REDO_CALIBRATED_MACRO_F1
    )
else:
    S2B_REDO_SELECTED_THRESHOLDS = dict(
        S2B_REDO_CALIBRATION_BASELINE_THRESHOLDS
    )
    S2B_REDO_SELECTED_CAPS = dict(
        S2B_REDO_CALIBRATION_BASELINE_CAPS
    )
    S2B_REDO_SELECTED_PREDICTIONS = set(
        S2B_REDO_CALIBRATION_BASELINE_PREDICTIONS
    )
    S2B_REDO_SELECTED_MACRO_F1 = (
        S2B_REDO_CALIBRATION_BASELINE_F1
    )

# ============================================================
# 9. Save calibrated decoder configuration and per-label metrics
# ============================================================
S2B_REDO_DECODER_CALIBRATION = {
    "experiment": "S2 Architecture B + LoRA",
    "baseline_macro_f1": S2B_REDO_CALIBRATION_BASELINE_F1,
    "calibrated_candidate_macro_f1": S2B_REDO_CALIBRATED_MACRO_F1,
    "calibration_improved": S2B_REDO_CALIBRATION_IMPROVED,
    "selected_macro_f1": S2B_REDO_SELECTED_MACRO_F1,
    "selected_thresholds": S2B_REDO_SELECTED_THRESHOLDS,
    "selected_span_caps": S2B_REDO_SELECTED_CAPS,
    "candidate_span_caps": S2B_REDO_CAP_CANDIDATES,
    "cap_search_results": S2B_REDO_CAP_SEARCH_RESULTS,
    "threshold_search_results": S2B_REDO_THRESHOLD_SEARCH_RESULTS,
    "fold3_gold_used": False,
}

S2B_REDO_DECODER_CALIBRATION_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_decoder_calibration.json"
)

with S2B_REDO_DECODER_CALIBRATION_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_DECODER_CALIBRATION,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("\n" + "=" * 76)
print("S2 REDO — CALIBRATION RESULTS")
print("=" * 76)
print(
    "Baseline Macro-F1:",
    f"{S2B_REDO_CALIBRATION_BASELINE_F1:.6f}",
)
print(
    "Calibrated candidate Macro-F1:",
    f"{S2B_REDO_CALIBRATED_MACRO_F1:.6f}",
)
print(
    "Selected Macro-F1:",
    f"{S2B_REDO_SELECTED_MACRO_F1:.6f}",
)
print("Calibration improved validation:", S2B_REDO_CALIBRATION_IMPROVED)
print("Selected predicted spans:", len(S2B_REDO_SELECTED_PREDICTIONS))

print("\nSelected per-label results:")
print(
    f"{'Label':12s} {'P':>7s} {'R':>7s} {'F1':>7s} "
    f"{'TP':>6s} {'FP':>6s} {'FN':>6s} "
    f"{'Thr':>7s} {'Cap':>6s}"
)

for label in S2B_REDO_LABELS:
    result = (
        S2B_REDO_THRESHOLD_SEARCH_RESULTS[label]
        if S2B_REDO_CALIBRATION_IMPROVED
        else S2B_REDO_CALIBRATION_BASELINE_PER_LABEL[label]
    )

    threshold = S2B_REDO_SELECTED_THRESHOLDS[label]
    cap = S2B_REDO_SELECTED_CAPS[label]

    print(
        f"{label:12s} "
        f"{result['precision']:7.3f} "
        f"{result['recall']:7.3f} "
        f"{result['f1']:7.3f} "
        f"{result['tp']:6d} "
        f"{result['fp']:6d} "
        f"{result['fn']:6d} "
        f"{threshold:7.3f} "
        f"{cap:6d}"
    )

print("\nCalibration saved:", S2B_REDO_DECODER_CALIBRATION_PATH)
print("Model weights modified: NO")
print("Fold 3 gold labels used: NO")
print("Cell 9 completed.")

S2 REDO — SPAN-CAP CALIBRATION CANDIDATES
APP         : [37, 44, 51, 512]
RESP        : [37, 55, 92, 512]
A.COUNSEL   : [27, 29, 32, 512]
R.COUNSEL   : [23, 27, 32, 512]
JUDGE       : [28, 32, 36, 512]
WIT         : [25, 26, 28, 512]
AUTH        : [37, 52, 98, 512]
COURT       : [36, 43, 65, 512]
STAT        : [48, 88, 256, 512]
PREC        : [110, 136, 215, 512]
DATE        : [24, 28, 31, 512]
CASENO      : [47, 59, 101, 512]

S2 REDO — BASELINE REPRODUCTION
Previously reported baseline Macro-F1: 0.177442
Reconstructed baseline Macro-F1: 0.177442
Reconstructed prediction count: 2596

S2 REDO — SELECTED SPAN CAPS
APP          cap=  37 F1=0.015 P=0.014 R=0.016
RESP         cap=  37 F1=0.012 P=0.007 R=0.045
A.COUNSEL    cap=  27 F1=0.067 P=0.250 R=0.038
R.COUNSEL    cap=  23 F1=0.048 P=0.028 R=0.190
JUDGE        cap=  28 F1=0.132 P=0.098 R=0.203
WIT          cap=  25 F1=0.058 P=0.073 R=0.048
AUTH         cap=  37 F1=0.053 P=0.071 R=0.042
COURT        cap=  36 F1=0.778 P=0.845 R=0.721
STA

In [10]:
# S2 Redo — Cell 10: Expand Architecture B LoRA targets, verify unchanged
# initial predictions, and train the expanded-LoRA variant on train data only.

import json
import math
import time
import numpy as np
import torch
import torch.nn.functional as F

from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from pathlib import Path

# ============================================================
# 1. Verify sequential prerequisites
# ============================================================
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_DEVICE",
    "S2B_REDO_TRAIN_LOADER",
    "S2B_REDO_VAL_LOADER",
    "S2B_REDO_TRAINABLE_PARAMETERS",
    "S2B_REDO_BEST_STATE",
    "S2B_REDO_SELECTED_MACRO_F1",
    "S2B_REDO_SELECTED_THRESHOLDS",
    "S2B_REDO_SELECTED_CAPS",
    "S2B_REDO_SPAN_CAPS",
    "S2B_REDO_LABELS",
    "S2B_REDO_NUM_LABELS",
    "S2B_REDO_OUTPUT_DIR",
    "s2b_redo_boundary_loss",
    "s2b_redo_predict_document_probabilities",
    "s2b_redo_tune_validation_thresholds",
    "s2b_redo_export_trainable_state",
    "s2b_redo_restore_trainable_state",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cells 1–9 in sequence. Missing: "
        f"{missing}"
    )

if globals().get("S2B_REDO_EXPANDED_STAGE_COMPLETED", False):
    raise RuntimeError(
        "The expanded-LoRA experiment has already completed in this session."
    )

S2B_REDO_EXPANDED_BASELINE_F1 = float(
    S2B_REDO_SELECTED_MACRO_F1
)
S2B_REDO_EXPANDED_BASELINE_THRESHOLDS = dict(
    S2B_REDO_SELECTED_THRESHOLDS
)
S2B_REDO_EXPANDED_BASELINE_CAPS = dict(
    S2B_REDO_SELECTED_CAPS
)

# ============================================================
# 2. Restore the original selected S2 state
# ============================================================
s2b_redo_restore_trainable_state(S2B_REDO_BEST_STATE)
S2B_REDO_MODEL.eval()

# Snapshot one fixed validation batch before inserting the new adapters.
S2B_REDO_EXPANSION_SMOKE_BATCH = next(
    iter(S2B_REDO_VAL_LOADER)
)

def s2b_redo_forward_fixed_batch():
    """Return start/end logits for the same fixed validation batch."""
    batch = S2B_REDO_EXPANSION_SMOKE_BATCH

    inputs = {
        key: batch[key].to(
            S2B_REDO_DEVICE,
            non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
        )
        for key in [
            "input_ids",
            "attention_mask",
            "char_ids",
            "char_to_token",
        ]
    }

    with torch.no_grad():
        with torch.autocast(
            device_type=S2B_REDO_DEVICE.type,
            dtype=torch.float16,
            enabled=(S2B_REDO_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S2B_REDO_MODEL(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                char_ids=inputs["char_ids"],
                char_to_token=inputs["char_to_token"],
            )

    return (
        start_logits.float().cpu(),
        end_logits.float().cpu(),
    )

S2B_REDO_PRE_EXPANSION_START, S2B_REDO_PRE_EXPANSION_END = (
    s2b_redo_forward_fixed_batch()
)

# ============================================================
# 3. Define a LoRA-wrapped linear layer
# ============================================================
class S2BRedoLoRALinear(nn.Module):
    """
    Base linear layer plus a low-rank update:

        y = x W^T + b + (alpha / rank) * x A^T B^T

    The original base weight and bias objects are preserved.
    B is initialized to zero, so the initial update is exactly zero.
    """

    def __init__(self, original_linear, rank=8, alpha=16.0):
        super().__init__()

        if not isinstance(original_linear, nn.Linear):
            raise TypeError(
                "LoRA wrapping requires an nn.Linear base module."
            )

        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank

        # Preserve the original pretrained parameter objects.
        self.weight = original_linear.weight
        self.bias = original_linear.bias

        factory_kwargs = {
            "device": original_linear.weight.device,
            "dtype": original_linear.weight.dtype,
        }

        self.lora_A = nn.Parameter(
            torch.empty(
                self.rank,
                self.in_features,
                **factory_kwargs,
            )
        )
        self.lora_B = nn.Parameter(
            torch.zeros(
                self.out_features,
                self.rank,
                **factory_kwargs,
            )
        )

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_output = F.linear(x, self.weight, self.bias)

        low_rank_output = F.linear(
            F.linear(x, self.lora_A),
            self.lora_B,
        )

        return base_output + self.scaling * low_rank_output


# ============================================================
# 4. Add LoRA to attention output and feed-forward projections
# ============================================================
S2B_REDO_EXPANDED_LORA_RANK = 8
S2B_REDO_EXPANDED_LORA_ALPHA = 16.0

S2B_REDO_EXPANDED_TARGET_NAMES = []

for layer_index, layer in enumerate(S2B_REDO_MODEL.encoder_layers):
    targets = [
        ("self_attn", "out_proj"),
        ("ffn", "fc1"),
        ("ffn", "fc2"),
    ]

    for parent_name, module_name in targets:
        parent_module = getattr(layer, parent_name)
        original_linear = getattr(parent_module, module_name)

        if isinstance(original_linear, S2BRedoLoRALinear):
            raise RuntimeError(
                f"LoRA was already installed on "
                f"{layer_index}.{parent_name}.{module_name}."
            )

        wrapped_linear = S2BRedoLoRALinear(
            original_linear,
            rank=S2B_REDO_EXPANDED_LORA_RANK,
            alpha=S2B_REDO_EXPANDED_LORA_ALPHA,
        )

        setattr(parent_module, module_name, wrapped_linear)

        S2B_REDO_EXPANDED_TARGET_NAMES.append(
            f"encoder_layers.{layer_index}.{parent_name}.{module_name}"
        )

# Freeze all base parameters; then enable only LoRA matrices and NER head.
for parameter in S2B_REDO_MODEL.parameters():
    parameter.requires_grad = False

S2B_REDO_EXPANDED_LORA_PARAMS = []
S2B_REDO_EXPANDED_HEAD_PARAMS = []

for name, parameter in S2B_REDO_MODEL.named_parameters():
    is_lora = (
        ".lora_A" in name
        or ".lora_B" in name
    )
    is_head = name.startswith((
        "char_embedding.",
        "char_conv.",
        "start_head.",
        "end_head.",
    ))

    if is_lora:
        parameter.requires_grad = True
        S2B_REDO_EXPANDED_LORA_PARAMS.append(parameter)
    elif is_head:
        parameter.requires_grad = True
        S2B_REDO_EXPANDED_HEAD_PARAMS.append(parameter)

if not S2B_REDO_EXPANDED_LORA_PARAMS:
    raise RuntimeError("No expanded LoRA parameters were created.")

if not S2B_REDO_EXPANDED_HEAD_PARAMS:
    raise RuntimeError("NER head parameters are missing.")

# ============================================================
# 5. Verify that zero-initialized adapters preserve predictions
# ============================================================
S2B_REDO_MODEL.eval()

S2B_REDO_POST_EXPANSION_START, S2B_REDO_POST_EXPANSION_END = (
    s2b_redo_forward_fixed_batch()
)

S2B_REDO_EXPANSION_START_DIFF = float(
    (
        S2B_REDO_PRE_EXPANSION_START
        - S2B_REDO_POST_EXPANSION_START
    ).abs().max().item()
)

S2B_REDO_EXPANSION_END_DIFF = float(
    (
        S2B_REDO_PRE_EXPANSION_END
        - S2B_REDO_POST_EXPANSION_END
    ).abs().max().item()
)

if max(
    S2B_REDO_EXPANSION_START_DIFF,
    S2B_REDO_EXPANSION_END_DIFF,
) > 1e-5:
    raise RuntimeError(
        "Adding zero-initialized LoRA adapters changed the model output. "
        f"Start diff={S2B_REDO_EXPANSION_START_DIFF}, "
        f"end diff={S2B_REDO_EXPANSION_END_DIFF}"
    )

print("=" * 76)
print("S2 REDO — EXPANDED LoRA MODEL")
print("=" * 76)
print("Expanded targets added:", len(S2B_REDO_EXPANDED_TARGET_NAMES))
print("Targets per layer: attention output, FFN fc1, FFN fc2")
print("New adapter rank:", S2B_REDO_EXPANDED_LORA_RANK)
print("New adapter alpha:", S2B_REDO_EXPANDED_LORA_ALPHA)
print("Trainable LoRA parameters:", f"{sum(p.numel() for p in S2B_REDO_EXPANDED_LORA_PARAMS):,}")
print("Trainable NER-head parameters:", f"{sum(p.numel() for p in S2B_REDO_EXPANDED_HEAD_PARAMS):,}")
print("Maximum initial start-logit difference:", S2B_REDO_EXPANSION_START_DIFF)
print("Maximum initial end-logit difference:", S2B_REDO_EXPANSION_END_DIFF)
print("Initial predictions preserved: YES")

# ============================================================
# 6. Keep the calibrated caps as the scoring protocol
# ============================================================
S2B_REDO_PREVIOUS_SPAN_CAPS = dict(S2B_REDO_SPAN_CAPS)
S2B_REDO_SPAN_CAPS = dict(S2B_REDO_EXPANDED_BASELINE_CAPS)

# Re-evaluate the baseline with exactly the selected cap configuration.
S2B_REDO_EXPANDED_BASELINE_PROBABILITIES, (
    S2B_REDO_EXPANDED_BASELINE_UNCOVERED
) = s2b_redo_predict_document_probabilities()

S2B_REDO_EXPANDED_BASELINE_METRICS, (
    S2B_REDO_EXPANDED_BASELINE_PREDICTIONS
) = s2b_redo_tune_validation_thresholds(
    S2B_REDO_EXPANDED_BASELINE_PROBABILITIES
)

S2B_REDO_EXPANDED_BASELINE_F1 = float(
    S2B_REDO_EXPANDED_BASELINE_METRICS["macro_f1"]
)

# Capture the complete expanded-model trainable state, including the new
# zero-effect adapters. If training fails to improve the baseline, this
# state can be restored without losing any original S2 weights.
S2B_REDO_EXPANDED_BEST_STATE = s2b_redo_export_trainable_state()
S2B_REDO_EXPANDED_BEST_F1 = S2B_REDO_EXPANDED_BASELINE_F1
S2B_REDO_EXPANDED_BEST_EPOCH = 0
S2B_REDO_EXPANDED_BEST_THRESHOLDS = dict(
    S2B_REDO_EXPANDED_BASELINE_METRICS["thresholds"]
)
S2B_REDO_EXPANDED_BEST_METRICS = (
    S2B_REDO_EXPANDED_BASELINE_METRICS
)

print("\nCalibrated baseline under the expanded architecture:")
print(
    "Baseline validation Macro-F1:",
    f"{S2B_REDO_EXPANDED_BASELINE_F1:.6f}",
)
print(
    "Baseline predictions:",
    S2B_REDO_EXPANDED_BASELINE_METRICS["predicted_count"],
)
print(
    "Selected thresholds remain validation-only: YES",
)

# ============================================================
# 7. Configure a new conservative optimizer and scheduler
# ============================================================
S2B_REDO_EXPANDED_MAX_EPOCHS = 3
S2B_REDO_EXPANDED_LORA_LR = 2.0e-5
S2B_REDO_EXPANDED_HEAD_LR = 5.0e-5
S2B_REDO_EXPANDED_WEIGHT_DECAY = 0.01

S2B_REDO_EXPANDED_TOTAL_STEPS = (
    S2B_REDO_EXPANDED_MAX_EPOCHS
    * len(S2B_REDO_TRAIN_LOADER)
)

S2B_REDO_EXPANDED_WARMUP_STEPS = max(
    1,
    int(0.05 * S2B_REDO_EXPANDED_TOTAL_STEPS),
)

S2B_REDO_EXPANDED_OPTIMIZER = AdamW(
    [
        {
            "params": S2B_REDO_EXPANDED_LORA_PARAMS,
            "lr": S2B_REDO_EXPANDED_LORA_LR,
            "name": "expanded_lora",
        },
        {
            "params": S2B_REDO_EXPANDED_HEAD_PARAMS,
            "lr": S2B_REDO_EXPANDED_HEAD_LR,
            "name": "ner_head",
        },
    ],
    weight_decay=S2B_REDO_EXPANDED_WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

def s2b_redo_expanded_lr_multiplier(step_index):
    if step_index < S2B_REDO_EXPANDED_WARMUP_STEPS:
        return max(
            0.1,
            (step_index + 1) / S2B_REDO_EXPANDED_WARMUP_STEPS,
        )

    decay_steps = max(
        1,
        S2B_REDO_EXPANDED_TOTAL_STEPS
        - S2B_REDO_EXPANDED_WARMUP_STEPS,
    )

    progress = (
        step_index - S2B_REDO_EXPANDED_WARMUP_STEPS
    ) / decay_steps
    progress = min(1.0, max(0.0, progress))

    return 0.5 * (1.0 + math.cos(math.pi * progress))

S2B_REDO_EXPANDED_SCHEDULER = LambdaLR(
    S2B_REDO_EXPANDED_OPTIMIZER,
    lr_lambda=s2b_redo_expanded_lr_multiplier,
)

S2B_REDO_EXPANDED_SCALER = torch.amp.GradScaler(
    "cuda",
    init_scale=4096.0,
    growth_interval=1000,
    enabled=(S2B_REDO_DEVICE.type == "cuda"),
)

S2B_REDO_EXPANDED_SKIPPED_TOTAL = 0
S2B_REDO_EXPANDED_SUCCESSFUL_TOTAL = 0
S2B_REDO_EXPANDED_EPOCHS_WITHOUT_IMPROVEMENT = 0
S2B_REDO_EXPANDED_PATIENCE = 2
S2B_REDO_EXPANDED_HISTORY = []

S2B_REDO_EXPANDED_CHECKPOINT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_LoRA_expanded_best.pt"
)
S2B_REDO_EXPANDED_HISTORY_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_LoRA_expanded_history.json"
)

S2B_REDO_EXPANDED_TRAINABLE_PARAMS = [
    p for p in S2B_REDO_MODEL.parameters()
    if p.requires_grad
]

# ============================================================
# 8. Train with finite-gradient checks and cumulative skip counts
# ============================================================
for epoch in range(1, S2B_REDO_EXPANDED_MAX_EPOCHS + 1):
    epoch_start = time.time()
    S2B_REDO_MODEL.train()

    running_loss = 0.0
    finite_loss_batches = 0
    successful_this_epoch = 0
    skipped_this_epoch = 0

    print("\n" + "=" * 76)
    print(
        f"S2 REDO — EXPANDED LoRA EPOCH "
        f"{epoch}/{S2B_REDO_EXPANDED_MAX_EPOCHS}"
    )
    print("=" * 76)

    for batch_index, batch in enumerate(
        S2B_REDO_TRAIN_LOADER,
        start=1,
    ):
        device_batch = {
            key: batch[key].to(
                S2B_REDO_DEVICE,
                non_blocking=(S2B_REDO_DEVICE.type == "cuda"),
            )
            for key in [
                "input_ids",
                "attention_mask",
                "char_ids",
                "char_to_token",
                "start_targets",
                "end_targets",
                "loss_mask",
            ]
        }

        S2B_REDO_EXPANDED_OPTIMIZER.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=S2B_REDO_DEVICE.type,
            dtype=torch.float16,
            enabled=(S2B_REDO_DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S2B_REDO_MODEL(
                input_ids=device_batch["input_ids"],
                attention_mask=device_batch["attention_mask"],
                char_ids=device_batch["char_ids"],
                char_to_token=device_batch["char_to_token"],
            )

        loss = s2b_redo_boundary_loss(
            start_logits=start_logits,
            end_logits=end_logits,
            start_targets=device_batch["start_targets"],
            end_targets=device_batch["end_targets"],
            loss_mask=device_batch["loss_mask"],
        )

        if not torch.isfinite(loss):
            skipped_this_epoch += 1
            S2B_REDO_EXPANDED_SKIPPED_TOTAL += 1
            S2B_REDO_EXPANDED_OPTIMIZER.zero_grad(set_to_none=True)

            if skipped_this_epoch <= 5:
                print(
                    f"Skipped non-finite loss at batch {batch_index}."
                )
            continue

        running_loss += float(loss.detach().float().item())
        finite_loss_batches += 1

        S2B_REDO_EXPANDED_SCALER.scale(loss).backward()
        S2B_REDO_EXPANDED_SCALER.unscale_(
            S2B_REDO_EXPANDED_OPTIMIZER
        )

        gradients_finite = all(
            p.grad is None
            or torch.isfinite(p.grad).all().item()
            for p in S2B_REDO_EXPANDED_TRAINABLE_PARAMS
        )

        if not gradients_finite:
            if S2B_REDO_EXPANDED_SCALER.is_enabled():
                S2B_REDO_EXPANDED_SCALER.step(
                    S2B_REDO_EXPANDED_OPTIMIZER
                )
                S2B_REDO_EXPANDED_SCALER.update()

            S2B_REDO_EXPANDED_OPTIMIZER.zero_grad(set_to_none=True)
            skipped_this_epoch += 1
            S2B_REDO_EXPANDED_SKIPPED_TOTAL += 1
            continue

        norm_squared = torch.zeros(
            (),
            dtype=torch.float64,
            device=S2B_REDO_DEVICE,
        )

        for p in S2B_REDO_EXPANDED_TRAINABLE_PARAMS:
            if p.grad is not None:
                norm_squared += (
                    p.grad.detach().to(torch.float64).square().sum()
                )

        total_norm = torch.sqrt(norm_squared)

        if not torch.isfinite(total_norm):
            S2B_REDO_EXPANDED_OPTIMIZER.zero_grad(set_to_none=True)
            skipped_this_epoch += 1
            S2B_REDO_EXPANDED_SKIPPED_TOTAL += 1

            if S2B_REDO_EXPANDED_SCALER.is_enabled():
                S2B_REDO_EXPANDED_SCALER.update(
                    new_scale=max(
                        S2B_REDO_EXPANDED_SCALER.get_scale() / 2.0,
                        1.0,
                    )
                )
            continue

        clip_factor = min(
            1.0,
            1.0 / (float(total_norm.item()) + 1e-12),
        )

        if clip_factor < 1.0:
            for p in S2B_REDO_EXPANDED_TRAINABLE_PARAMS:
                if p.grad is not None:
                    p.grad.mul_(clip_factor)

        S2B_REDO_EXPANDED_SCALER.step(
            S2B_REDO_EXPANDED_OPTIMIZER
        )
        S2B_REDO_EXPANDED_SCALER.update()
        S2B_REDO_EXPANDED_SCHEDULER.step()

        successful_this_epoch += 1
        S2B_REDO_EXPANDED_SUCCESSFUL_TOTAL += 1

        if (
            batch_index % 100 == 0
            or batch_index == len(S2B_REDO_TRAIN_LOADER)
        ):
            average_loss = running_loss / max(1, finite_loss_batches)
            current_lrs = [
                group["lr"]
                for group in S2B_REDO_EXPANDED_OPTIMIZER.param_groups
            ]

            print(
                f"Epoch {epoch} | batch {batch_index}/"
                f"{len(S2B_REDO_TRAIN_LOADER)} | "
                f"loss={average_loss:.5f} | "
                f"LoRA LR={current_lrs[0]:.2e} | "
                f"head LR={current_lrs[1]:.2e} | "
                f"skipped={S2B_REDO_EXPANDED_SKIPPED_TOTAL}"
            )

    if successful_this_epoch == 0:
        raise RuntimeError(
            f"No successful optimizer updates in expanded-LoRA epoch {epoch}."
        )

    train_loss = running_loss / max(1, finite_loss_batches)

    # Evaluate and retune thresholds on validation documents only.
    S2B_REDO_EXPANDED_VAL_PROBABILITIES, (
        S2B_REDO_EXPANDED_UNCOVERED
    ) = s2b_redo_predict_document_probabilities()

    S2B_REDO_EXPANDED_METRICS, (
        S2B_REDO_EXPANDED_PREDICTIONS
    ) = s2b_redo_tune_validation_thresholds(
        S2B_REDO_EXPANDED_VAL_PROBABILITIES
    )

    score = float(S2B_REDO_EXPANDED_METRICS["macro_f1"])
    epoch_seconds = time.time() - epoch_start

    epoch_record = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_macro_f1": score,
        "validation_predictions": int(
            S2B_REDO_EXPANDED_METRICS["predicted_count"]
        ),
        "validation_true_positives": int(
            S2B_REDO_EXPANDED_METRICS["exact_true_positives"]
        ),
        "successful_updates_this_epoch": int(successful_this_epoch),
        "skipped_updates_this_epoch": int(skipped_this_epoch),
        "skipped_updates": int(S2B_REDO_EXPANDED_SKIPPED_TOTAL),
        "epoch_seconds": float(epoch_seconds),
        "per_label": S2B_REDO_EXPANDED_METRICS["per_label"],
        "thresholds": S2B_REDO_EXPANDED_METRICS["thresholds"],
        "span_caps": dict(S2B_REDO_SPAN_CAPS),
    }

    S2B_REDO_EXPANDED_HISTORY.append(epoch_record)

    print("\nValidation summary:")
    print("  Training loss:", f"{train_loss:.6f}")
    print("  Strict Span Macro-F1:", f"{score:.6f}")
    print(
        "  Predicted spans:",
        S2B_REDO_EXPANDED_METRICS["predicted_count"],
    )
    print(
        "  Exact true positives:",
        S2B_REDO_EXPANDED_METRICS["exact_true_positives"],
    )
    print("  Successful updates:", successful_this_epoch)
    print("  Skipped updates this epoch:", skipped_this_epoch)
    print("  Cumulative skipped updates:", S2B_REDO_EXPANDED_SKIPPED_TOTAL)
    print("  Epoch seconds:", f"{epoch_seconds:.1f}")

    print("\nPer-label validation F1:")
    for label in S2B_REDO_LABELS:
        m = S2B_REDO_EXPANDED_METRICS["per_label"][label]
        print(
            f"  {label:12s} F1={m['f1']:.3f} "
            f"P={m['precision']:.3f} R={m['recall']:.3f} "
            f"TP={m['tp']} FP={m['fp']} FN={m['fn']} "
            f"Thr={S2B_REDO_EXPANDED_METRICS['thresholds'][label]:.3f}"
        )

    if score > S2B_REDO_EXPANDED_BEST_F1 + 1e-6:
        S2B_REDO_EXPANDED_BEST_F1 = score
        S2B_REDO_EXPANDED_BEST_EPOCH = epoch
        S2B_REDO_EXPANDED_BEST_STATE = (
            s2b_redo_export_trainable_state()
        )
        S2B_REDO_EXPANDED_BEST_THRESHOLDS = dict(
            S2B_REDO_EXPANDED_METRICS["thresholds"]
        )
        S2B_REDO_EXPANDED_BEST_METRICS = (
            S2B_REDO_EXPANDED_METRICS
        )
        S2B_REDO_EXPANDED_EPOCHS_WITHOUT_IMPROVEMENT = 0

        torch.save(
            {
                "experiment": "S2 Architecture B expanded LoRA",
                "best_epoch": epoch,
                "best_validation_macro_f1": score,
                "trainable_state": S2B_REDO_EXPANDED_BEST_STATE,
                "thresholds": S2B_REDO_EXPANDED_BEST_THRESHOLDS,
                "span_caps": dict(S2B_REDO_SPAN_CAPS),
                "labels": S2B_REDO_LABELS,
                "lora_rank": S2B_REDO_EXPANDED_LORA_RANK,
                "lora_alpha": S2B_REDO_EXPANDED_LORA_ALPHA,
                "targets": S2B_REDO_EXPANDED_TARGET_NAMES,
                "fold3_gold_used": False,
            },
            S2B_REDO_EXPANDED_CHECKPOINT_PATH,
        )

        print("\nExpanded-LoRA best checkpoint updated.")

    else:
        S2B_REDO_EXPANDED_EPOCHS_WITHOUT_IMPROVEMENT += 1
        print(
            "No validation improvement. Consecutive non-improving epochs:",
            S2B_REDO_EXPANDED_EPOCHS_WITHOUT_IMPROVEMENT,
        )

    with S2B_REDO_EXPANDED_HISTORY_PATH.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            S2B_REDO_EXPANDED_HISTORY,
            f,
            indent=2,
            ensure_ascii=False,
        )

    if (
        S2B_REDO_EXPANDED_EPOCHS_WITHOUT_IMPROVEMENT
        >= S2B_REDO_EXPANDED_PATIENCE
    ):
        print("\nEarly stopping due to lack of validation improvement.")
        break

# ============================================================
# 9. Restore the best expanded model only if it improves the calibrated baseline
# ============================================================
S2B_REDO_EXPANDED_IMPROVED = (
    S2B_REDO_EXPANDED_BEST_F1
    > S2B_REDO_EXPANDED_BASELINE_F1 + 1e-6
)

if S2B_REDO_EXPANDED_IMPROVED:
    s2b_redo_restore_trainable_state(
        S2B_REDO_EXPANDED_BEST_STATE
    )

    S2B_REDO_SELECTED_THRESHOLDS = dict(
        S2B_REDO_EXPANDED_BEST_THRESHOLDS
    )
    S2B_REDO_SELECTED_CAPS = dict(S2B_REDO_SPAN_CAPS)
    S2B_REDO_SELECTED_MACRO_F1 = (
        S2B_REDO_EXPANDED_BEST_F1
    )
    S2B_REDO_SELECTED_STAGE = "expanded_lora"

    S2B_REDO_BEST_STATE = S2B_REDO_EXPANDED_BEST_STATE
    S2B_REDO_BEST_VALIDATION_F1 = (
        S2B_REDO_EXPANDED_BEST_F1
    )
    S2B_REDO_BEST_THRESHOLDS = dict(
        S2B_REDO_EXPANDED_BEST_THRESHOLDS
    )
    S2B_REDO_BEST_CHECKPOINT_PATH = (
        S2B_REDO_EXPANDED_CHECKPOINT_PATH
    )

else:
    # Restore the exact initial state for this expanded model. The added
    # adapters have zero B matrices, so output is unchanged from the
    # earlier calibrated baseline.
    s2b_redo_restore_trainable_state(
        S2B_REDO_EXPANDED_BASELINE_STATE
    )

    S2B_REDO_SELECTED_THRESHOLDS = dict(
        S2B_REDO_EXPANDED_BASELINE_THRESHOLDS
    )
    S2B_REDO_SELECTED_CAPS = dict(
        S2B_REDO_EXPANDED_BASELINE_CAPS
    )
    S2B_REDO_SELECTED_MACRO_F1 = (
        S2B_REDO_EXPANDED_BASELINE_F1
    )
    S2B_REDO_SELECTED_STAGE = "original_lora_calibrated"

    S2B_REDO_BEST_STATE = S2B_REDO_EXPANDED_BASELINE_STATE
    S2B_REDO_BEST_VALIDATION_F1 = (
        S2B_REDO_EXPANDED_BASELINE_F1
    )
    S2B_REDO_BEST_THRESHOLDS = dict(
        S2B_REDO_EXPANDED_BASELINE_THRESHOLDS
    )

S2B_REDO_MODEL.eval()

# Save one final selected trainable checkpoint under a distinct filename.
S2B_REDO_FINAL_SELECTED_CHECKPOINT = (
    S2B_REDO_OUTPUT_DIR / "S2B_LoRA_final_selected.pt"
)

torch.save(
    {
        "experiment": "S2 Architecture B + LoRA",
        "selected_stage": S2B_REDO_SELECTED_STAGE,
        "validation_macro_f1": float(S2B_REDO_SELECTED_MACRO_F1),
        "trainable_state": S2B_REDO_BEST_STATE,
        "thresholds": S2B_REDO_SELECTED_THRESHOLDS,
        "span_caps": S2B_REDO_SELECTED_CAPS,
        "labels": S2B_REDO_LABELS,
        "fold3_gold_used": False,
    },
    S2B_REDO_FINAL_SELECTED_CHECKPOINT,
)

S2B_REDO_EXPANDED_SUMMARY = {
    "baseline_validation_macro_f1": float(
        S2B_REDO_EXPANDED_BASELINE_F1
    ),
    "best_expanded_validation_macro_f1": float(
        S2B_REDO_EXPANDED_BEST_F1
    ),
    "expanded_lora_improved": bool(
        S2B_REDO_EXPANDED_IMPROVED
    ),
    "selected_stage": S2B_REDO_SELECTED_STAGE,
    "selected_validation_macro_f1": float(
        S2B_REDO_SELECTED_MACRO_F1
    ),
    "best_epoch": S2B_REDO_EXPANDED_BEST_EPOCH,
    "successful_updates": int(
        S2B_REDO_EXPANDED_SUCCESSFUL_TOTAL
    ),
    "skipped_updates": int(
        S2B_REDO_EXPANDED_SKIPPED_TOTAL
    ),
    "checkpoint": str(S2B_REDO_FINAL_SELECTED_CHECKPOINT),
    "history": str(S2B_REDO_EXPANDED_HISTORY_PATH),
    "fold3_gold_used": False,
}

S2B_REDO_EXPANDED_SUMMARY_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_expanded_lora_summary.json"
)

with S2B_REDO_EXPANDED_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_EXPANDED_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False,
    )

S2B_REDO_EXPANDED_STAGE_COMPLETED = True

print("\n" + "=" * 76)
print("S2 REDO — EXPANDED LoRA EXPERIMENT COMPLETE")
print("=" * 76)
print(
    "Calibrated baseline Macro-F1:",
    f"{S2B_REDO_EXPANDED_BASELINE_F1:.6f}",
)
print(
    "Best expanded-LoRA Macro-F1:",
    f"{S2B_REDO_EXPANDED_BEST_F1:.6f}",
)
print("Expanded-LoRA improved baseline:", S2B_REDO_EXPANDED_IMPROVED)
print("Selected stage:", S2B_REDO_SELECTED_STAGE)
print(
    "Selected validation Macro-F1:",
    f"{S2B_REDO_SELECTED_MACRO_F1:.6f}",
)
print("Successful optimizer updates:", S2B_REDO_EXPANDED_SUCCESSFUL_TOTAL)
print("Cumulative skipped updates:", S2B_REDO_EXPANDED_SKIPPED_TOTAL)
print("Final selected checkpoint:", S2B_REDO_FINAL_SELECTED_CHECKPOINT)
print("Training history:", S2B_REDO_EXPANDED_HISTORY_PATH)
print("Architecture C/S3 weights used: NO")
print("Fold 3 gold labels used: NO")
print("\nCell 10 completed.")

S2 REDO — EXPANDED LoRA MODEL
Expanded targets added: 18
Targets per layer: attention output, FFN fc1, FFN fc2
New adapter rank: 8
New adapter alpha: 16.0
Trainable LoRA parameters: 589,824
Trainable NER-head parameters: 315,640
Maximum initial start-logit difference: 0.0
Maximum initial end-logit difference: 0.0
Initial predictions preserved: YES

Calibrated baseline under the expanded architecture:
Baseline validation Macro-F1: 0.191952
Baseline predictions: 2658
Selected thresholds remain validation-only: YES

S2 REDO — EXPANDED LoRA EPOCH 1/3
Epoch 1 | batch 100/498 | loss=0.00603 | LoRA LR=2.00e-05 | head LR=5.00e-05 | skipped=0
Epoch 1 | batch 200/498 | loss=0.00605 | LoRA LR=1.96e-05 | head LR=4.90e-05 | skipped=0
Epoch 1 | batch 300/498 | loss=0.00584 | LoRA LR=1.88e-05 | head LR=4.69e-05 | skipped=0
Epoch 1 | batch 400/498 | loss=0.00567 | LoRA LR=1.75e-05 | head LR=4.38e-05 | skipped=0
Epoch 1 | batch 498/498 | loss=0.00570 | LoRA LR=1.59e-05 | head LR=3.98e-05 | skipped=0

V

In [11]:
# S2 Redo — Cell 11: Refine per-label thresholds with a larger candidate pool.
# This cell uses validation labels only and does not update model weights.

import json
import numpy as np
from pathlib import Path

# ============================================================
# 1. Verify sequential prerequisites
# ============================================================
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_DEVICE",
    "S2B_REDO_VAL_TEXTS",
    "S2B_REDO_VAL_LOADER",
    "S2B_REDO_DOC_SPANS",
    "S2B_REDO_LABELS",
    "S2B_REDO_LABEL_TO_ID",
    "S2B_REDO_NUM_LABELS",
    "S2B_REDO_SELECTED_THRESHOLDS",
    "S2B_REDO_SELECTED_CAPS",
    "S2B_REDO_SELECTED_MACRO_F1",
    "S2B_REDO_OUTPUT_DIR",
    "s2b_redo_predict_document_probabilities",
    "s2b_redo_build_candidates",
    "s2b_redo_decode_candidates",
    "s2b_redo_tune_validation_thresholds",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cells 1–10 sequentially first. Missing variables: "
        f"{missing}"
    )

if globals().get("S2B_REDO_FINAL_CALIBRATION_COMPLETED", False):
    raise RuntimeError(
        "Cell 11 already completed in this session. "
        "Inspect its saved configuration rather than rerunning it."
    )

# Preserve the existing selected decoder settings.
S2B_REDO_CELL11_OLD_CAPS = dict(S2B_REDO_SELECTED_CAPS)
S2B_REDO_CELL11_OLD_THRESHOLDS = dict(S2B_REDO_SELECTED_THRESHOLDS)
S2B_REDO_CELL11_OLD_MAX_CANDIDATES = int(
    S2B_REDO_MAX_BOUNDARY_CANDIDATES
)
S2B_REDO_CELL11_OLD_MAX_END_OPTIONS = int(
    S2B_REDO_MAX_END_OPTIONS
)
S2B_REDO_CELL11_OLD_THRESHOLD_GRID = list(
    S2B_REDO_THRESHOLD_GRID
)

# Use the selected expanded-LoRA model's validation probabilities, not
# probabilities left over from a non-best epoch.
S2B_REDO_SPAN_CAPS = dict(S2B_REDO_CELL11_OLD_CAPS)

S2B_REDO_VAL_PROBABILITIES_FINAL, (
    S2B_REDO_VAL_UNCOVERED_CHARS_FINAL
) = s2b_redo_predict_document_probabilities()

S2B_REDO_CELL11_GOLD_BY_LABEL = {
    label: set()
    for label in S2B_REDO_LABELS
}

for doc_id in S2B_REDO_VAL_TEXTS:
    for start, end, label, _ann_idx in S2B_REDO_DOC_SPANS[doc_id]:
        S2B_REDO_CELL11_GOLD_BY_LABEL[label].add(
            (doc_id, start, end, label)
        )

if any(
    not S2B_REDO_CELL11_GOLD_BY_LABEL[label]
    for label in S2B_REDO_LABELS
):
    raise RuntimeError(
        "At least one entity label has no validation gold spans."
    )

# ============================================================
# 2. Define a fixed-decoder evaluator
# ============================================================
# S2 Redo — Cell 11 fix: Correct Python global-variable scope in the evaluator.

def s2b_redo_cell11_evaluate_fixed_decoder(
    probabilities,
    thresholds,
    span_caps,
    max_candidates,
    max_end_options,
):
    """Evaluate fixed decoder settings without retuning thresholds."""

    # These variables are deliberately changed temporarily and then restored.
    global S2B_REDO_MAX_BOUNDARY_CANDIDATES
    global S2B_REDO_MAX_END_OPTIONS

    old_candidates = S2B_REDO_MAX_BOUNDARY_CANDIDATES
    old_end_options = S2B_REDO_MAX_END_OPTIONS

    S2B_REDO_MAX_BOUNDARY_CANDIDATES = int(max_candidates)
    S2B_REDO_MAX_END_OPTIONS = int(max_end_options)

    try:
        all_predictions = set()
        per_label = {}

        for label_id, label in enumerate(S2B_REDO_LABELS):
            label_predictions = set()

            for doc_id in S2B_REDO_VAL_TEXTS:
                scores = probabilities[doc_id]

                candidates = s2b_redo_build_candidates(
                    scores["start"],
                    scores["end"],
                    label_id,
                    int(span_caps[label]),
                )

                decoded = s2b_redo_decode_candidates(
                    candidates,
                    float(thresholds[label]),
                )

                for start, end in decoded:
                    if not (
                        0 <= start < end
                        <= len(S2B_REDO_VAL_TEXTS[doc_id])
                    ):
                        raise ValueError(
                            f"Invalid validation span for {doc_id}: "
                            f"{(start, end)}"
                        )

                    if end - start > int(span_caps[label]):
                        raise ValueError(
                            f"Span cap violated for label {label}."
                        )

                    label_predictions.add(
                        (doc_id, start, end, label)
                    )

            gold = S2B_REDO_CELL11_GOLD_BY_LABEL[label]

            tp = len(label_predictions & gold)
            fp = len(label_predictions - gold)
            fn = len(gold - label_predictions)

            precision = (
                tp / (tp + fp) if tp + fp else 0.0
            )
            recall = (
                tp / (tp + fn) if tp + fn else 0.0
            )
            f1 = (
                2.0 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )

            per_label[label] = {
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "predicted": len(label_predictions),
                "gold": len(gold),
            }

            all_predictions.update(label_predictions)

        macro_f1 = float(np.mean([
            per_label[label]["f1"]
            for label in S2B_REDO_LABELS
        ]))

        all_gold = set().union(
            *S2B_REDO_CELL11_GOLD_BY_LABEL.values()
        )

        return {
            "macro_f1": macro_f1,
            "predicted_count": len(all_predictions),
            "gold_count": len(all_gold),
            "exact_true_positives": len(all_predictions & all_gold),
            "per_label": per_label,
        }

    finally:
        # Always restore previous settings, even if evaluation raises an error.
        S2B_REDO_MAX_BOUNDARY_CANDIDATES = old_candidates
        S2B_REDO_MAX_END_OPTIONS = old_end_options







# ============================================================
# 3. Re-evaluate the existing selected decoder
# ============================================================
S2B_REDO_CELL11_BASELINE_METRICS = (
    s2b_redo_cell11_evaluate_fixed_decoder(
        probabilities=S2B_REDO_VAL_PROBABILITIES_FINAL,
        thresholds=S2B_REDO_CELL11_OLD_THRESHOLDS,
        span_caps=S2B_REDO_CELL11_OLD_CAPS,
        max_candidates=S2B_REDO_CELL11_OLD_MAX_CANDIDATES,
        max_end_options=S2B_REDO_CELL11_OLD_MAX_END_OPTIONS,
    )
)

print("=" * 76)
print("S2 REDO — CURRENT DECODER RE-EVALUATED")
print("=" * 76)
print(
    "Previously selected validation Macro-F1:",
    f"{S2B_REDO_SELECTED_MACRO_F1:.6f}",
)
print(
    "Current fixed-decoder Macro-F1:",
    f"{S2B_REDO_CELL11_BASELINE_METRICS['macro_f1']:.6f}",
)
print(
    "Current predicted spans:",
    S2B_REDO_CELL11_BASELINE_METRICS["predicted_count"],
)

# ============================================================
# 4. Expand candidate capacity and refine threshold candidates
# ============================================================
S2B_REDO_FINE_THRESHOLD_GRID = [
    0.001, 0.002, 0.003, 0.005, 0.008,
    0.010, 0.015, 0.020, 0.025, 0.030,
    0.040, 0.050, 0.060, 0.080, 0.100,
    0.120, 0.140, 0.160, 0.180, 0.200,
    0.225, 0.250, 0.275, 0.300, 0.325,
    0.350, 0.375, 0.400, 0.450, 0.500,
    0.550, 0.600, 0.650, 0.700, 0.750,
    0.800, 0.850, 0.900,
]

S2B_REDO_FINE_MAX_CANDIDATES = 512
S2B_REDO_FINE_MAX_END_OPTIONS = 20

S2B_REDO_MAX_BOUNDARY_CANDIDATES = (
    S2B_REDO_FINE_MAX_CANDIDATES
)
S2B_REDO_MAX_END_OPTIONS = S2B_REDO_FINE_MAX_END_OPTIONS
S2B_REDO_THRESHOLD_GRID = list(S2B_REDO_FINE_THRESHOLD_GRID)

print("\nFine calibration configuration:")
print("Threshold candidates:", len(S2B_REDO_THRESHOLD_GRID))
print(
    "Maximum boundary candidates:",
    S2B_REDO_MAX_BOUNDARY_CANDIDATES,
)
print(
    "Maximum end options per start:",
    S2B_REDO_MAX_END_OPTIONS,
)

# ============================================================
# 5. Retune thresholds on validation, using the selected span caps
# ============================================================
S2B_REDO_FINE_METRICS, S2B_REDO_FINE_PREDICTIONS = (
    s2b_redo_tune_validation_thresholds(
        S2B_REDO_VAL_PROBABILITIES_FINAL
    )
)

S2B_REDO_FINE_MACRO_F1 = float(
    S2B_REDO_FINE_METRICS["macro_f1"]
)

S2B_REDO_FINE_IMPROVED = (
    S2B_REDO_FINE_MACRO_F1
    > S2B_REDO_CELL11_BASELINE_METRICS["macro_f1"] + 1e-8
)

# ============================================================
# 6. Select the better validated decoder
# ============================================================
if S2B_REDO_FINE_IMPROVED:
    S2B_REDO_FINAL_THRESHOLDS = dict(
        S2B_REDO_FINE_METRICS["thresholds"]
    )
    S2B_REDO_FINAL_CAPS = dict(S2B_REDO_CELL11_OLD_CAPS)
    S2B_REDO_FINAL_MACRO_F1 = S2B_REDO_FINE_MACRO_F1
    S2B_REDO_FINAL_VALIDATION_METRICS = S2B_REDO_FINE_METRICS
    S2B_REDO_FINAL_VALIDATION_PREDICTIONS = set(
        S2B_REDO_FINE_PREDICTIONS
    )
    S2B_REDO_FINAL_MAX_CANDIDATES = (
        S2B_REDO_FINE_MAX_CANDIDATES
    )
    S2B_REDO_FINAL_MAX_END_OPTIONS = (
        S2B_REDO_FINE_MAX_END_OPTIONS
    )
    S2B_REDO_FINAL_THRESHOLD_GRID = list(
        S2B_REDO_FINE_THRESHOLD_GRID
    )
else:
    S2B_REDO_FINAL_THRESHOLDS = dict(
        S2B_REDO_CELL11_OLD_THRESHOLDS
    )
    S2B_REDO_FINAL_CAPS = dict(S2B_REDO_CELL11_OLD_CAPS)
    S2B_REDO_FINAL_MACRO_F1 = float(
        S2B_REDO_CELL11_BASELINE_METRICS["macro_f1"]
    )
    S2B_REDO_FINAL_VALIDATION_METRICS = (
        S2B_REDO_CELL11_BASELINE_METRICS
    )
    S2B_REDO_FINAL_VALIDATION_PREDICTIONS = set()

    S2B_REDO_FINAL_MAX_CANDIDATES = (
        S2B_REDO_CELL11_OLD_MAX_CANDIDATES
    )
    S2B_REDO_FINAL_MAX_END_OPTIONS = (
        S2B_REDO_CELL11_OLD_MAX_END_OPTIONS
    )
    S2B_REDO_FINAL_THRESHOLD_GRID = list(
        S2B_REDO_CELL11_OLD_THRESHOLD_GRID
    )

# Update downstream selected settings.
S2B_REDO_SELECTED_THRESHOLDS = dict(
    S2B_REDO_FINAL_THRESHOLDS
)
S2B_REDO_SELECTED_CAPS = dict(
    S2B_REDO_FINAL_CAPS
)
S2B_REDO_SELECTED_MACRO_F1 = float(
    S2B_REDO_FINAL_MACRO_F1
)
S2B_REDO_SPAN_CAPS = dict(
    S2B_REDO_FINAL_CAPS
)
S2B_REDO_MAX_BOUNDARY_CANDIDATES = int(
    S2B_REDO_FINAL_MAX_CANDIDATES
)
S2B_REDO_MAX_END_OPTIONS = int(
    S2B_REDO_FINAL_MAX_END_OPTIONS
)
S2B_REDO_THRESHOLD_GRID = list(
    S2B_REDO_FINAL_THRESHOLD_GRID
)

# ============================================================
# 7. Save decoder configuration
# ============================================================
S2B_REDO_FINAL_DECODER_CONFIG = {
    "experiment": "S2 Architecture B + LoRA",
    "selected_stage": S2B_REDO_SELECTED_STAGE,
    "baseline_macro_f1": float(
        S2B_REDO_CELL11_BASELINE_METRICS["macro_f1"]
    ),
    "fine_grid_candidate_macro_f1": S2B_REDO_FINE_MACRO_F1,
    "fine_grid_improved": bool(S2B_REDO_FINE_IMPROVED),
    "selected_validation_macro_f1": S2B_REDO_SELECTED_MACRO_F1,
    "selected_thresholds": S2B_REDO_SELECTED_THRESHOLDS,
    "selected_span_caps": S2B_REDO_SELECTED_CAPS,
    "max_boundary_candidates": S2B_REDO_MAX_BOUNDARY_CANDIDATES,
    "max_end_options_per_start": S2B_REDO_MAX_END_OPTIONS,
    "threshold_grid": S2B_REDO_THRESHOLD_GRID,
    "validation_predicted_span_count": int(
        S2B_REDO_FINAL_VALIDATION_METRICS["predicted_count"]
    ),
    "validation_exact_true_positives": int(
        S2B_REDO_FINAL_VALIDATION_METRICS["exact_true_positives"]
    ),
    "validation_documents": len(S2B_REDO_VAL_TEXTS),
    "fold3_gold_used": False,
    "model_weights_changed": False,
}

S2B_REDO_FINAL_DECODER_CONFIG_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_final_decoder_config.json"
)

with S2B_REDO_FINAL_DECODER_CONFIG_PATH.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        S2B_REDO_FINAL_DECODER_CONFIG,
        f,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# 8. Report per-label validation results
# ============================================================
print("\n" + "=" * 76)
print("S2 REDO — FINAL DECODER CALIBRATION")
print("=" * 76)
print(
    "Current decoder Macro-F1:",
    f"{S2B_REDO_CELL11_BASELINE_METRICS['macro_f1']:.6f}",
)
print(
    "Fine-grid Macro-F1:",
    f"{S2B_REDO_FINE_MACRO_F1:.6f}",
)
print("Fine-grid calibration improved:", S2B_REDO_FINE_IMPROVED)
print(
    "Selected validation Macro-F1:",
    f"{S2B_REDO_SELECTED_MACRO_F1:.6f}",
)
print(
    "Selected validation predictions:",
    S2B_REDO_FINAL_VALIDATION_METRICS["predicted_count"],
)

print("\nSelected label results:")
print(
    f"{'Label':12s} {'P':>7s} {'R':>7s} {'F1':>7s} "
    f"{'TP':>6s} {'FP':>6s} {'FN':>6s} "
    f"{'Thr':>7s} {'Cap':>6s}"
)

if S2B_REDO_FINE_IMPROVED:
    selected_per_label = S2B_REDO_FINE_METRICS["per_label"]
else:
    selected_per_label = (
        S2B_REDO_CELL11_BASELINE_METRICS["per_label"]
    )

for label in S2B_REDO_LABELS:
    m = selected_per_label[label]
    print(
        f"{label:12s} "
        f"{m['precision']:7.3f} "
        f"{m['recall']:7.3f} "
        f"{m['f1']:7.3f} "
        f"{m['tp']:6d} "
        f"{m['fp']:6d} "
        f"{m['fn']:6d} "
        f"{S2B_REDO_SELECTED_THRESHOLDS[label]:7.3f} "
        f"{S2B_REDO_SELECTED_CAPS[label]:6d}"
    )

print("\nSaved decoder configuration:", S2B_REDO_FINAL_DECODER_CONFIG_PATH)
print("Model weights changed: NO")
print("Fold 3 gold labels used: NO")
print("Cell 11 completed.")

S2 REDO — CURRENT DECODER RE-EVALUATED
Previously selected validation Macro-F1: 0.195613
Current fixed-decoder Macro-F1: 0.195613
Current predicted spans: 2610

Fine calibration configuration:
Threshold candidates: 38
Maximum boundary candidates: 512
Maximum end options per start: 20

S2 REDO — FINAL DECODER CALIBRATION
Current decoder Macro-F1: 0.195613
Fine-grid Macro-F1: 0.197571
Fine-grid calibration improved: True
Selected validation Macro-F1: 0.197571
Selected validation predictions: 2663

Selected label results:
Label              P       R      F1     TP     FP     FN     Thr    Cap
APP            0.017   0.016   0.017      1     58     60   0.200     37
RESP           0.011   0.076   0.020      5    432     61   0.080     37
A.COUNSEL      0.069   0.077   0.073      2     27     24   0.140     27
R.COUNSEL      0.030   0.238   0.053      5    163     16   0.080     23
JUDGE          0.110   0.266   0.155     17    138     47   0.080     28
WIT            0.077   0.048   0.059 

In [12]:
# ============================================================
# S2 REDO — CELL 12
# Prepare Fold 3 test windows; do not use test gold labels.
# ============================================================

import json
from pathlib import Path

import numpy as np

print("=" * 76)
print("S2 REDO — FOLD 3 TEST PREPROCESSING")
print("=" * 76)

# ---------- 1. Check prerequisites ----------
required_names = [
    "S2B_REDO_FOLD3_TEXT",
    "S2B_REDO_LABELS",
    "S2B_REDO_OUTPUT_DIR",
    "S2B_REDO_CHAR_TO_ID",
    "s2b_redo_build_document_windows",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

if not isinstance(S2B_REDO_FOLD3_TEXT, dict):
    raise TypeError(
        "Expected S2B_REDO_FOLD3_TEXT to be a dictionary "
        "mapping document IDs to text."
    )

# ---------- 2. Preserve document IDs and raw text ----------
S2B_REDO_TEST_TEXTS = dict(S2B_REDO_FOLD3_TEXT)
S2B_REDO_TEST_DOC_IDS = list(S2B_REDO_TEST_TEXTS.keys())

if len(S2B_REDO_TEST_DOC_IDS) != 35:
    raise ValueError(
        f"Expected 35 Fold 3 documents, found "
        f"{len(S2B_REDO_TEST_DOC_IDS)}."
    )

if len(set(map(str, S2B_REDO_TEST_DOC_IDS))) != len(S2B_REDO_TEST_DOC_IDS):
    raise ValueError("Document IDs are not unique after string conversion.")

for doc_id, text in S2B_REDO_TEST_TEXTS.items():
    if not isinstance(text, str):
        raise TypeError(
            f"Test document {doc_id!r} has type "
            f"{type(text).__name__}, not str."
        )

# Preserve empty/whitespace-only documents for the final submission.
S2B_REDO_TEST_EMPTY_DOC_IDS = [
    doc_id
    for doc_id, text in S2B_REDO_TEST_TEXTS.items()
    if not text.strip()
]

# ---------- 3. Build windows with no gold spans ----------
# Empty span lists are used only to construct inference inputs.
# No Fold 3 annotations are loaded or consulted.
S2B_REDO_TEST_WINDOWS = []
S2B_REDO_TEST_WINDOWS_BY_DOC = {}

for doc_id, text in S2B_REDO_TEST_TEXTS.items():
    windows = s2b_redo_build_document_windows(
        doc_id,
        text,
        [],
    )

    if windows is None:
        raise RuntimeError(
            f"Window builder returned None for document {doc_id!r}."
        )

    windows = list(windows)
    S2B_REDO_TEST_WINDOWS_BY_DOC[doc_id] = windows
    S2B_REDO_TEST_WINDOWS.extend(windows)

# ---------- 4. Basic integrity checks ----------
window_doc_ids = {
    window.get("doc_id")
    for window in S2B_REDO_TEST_WINDOWS
    if isinstance(window, dict) and "doc_id" in window
}

# The builder must produce windows for every non-empty document.
S2B_REDO_TEST_NONEMPTY_DOC_IDS = [
    doc_id
    for doc_id, text in S2B_REDO_TEST_TEXTS.items()
    if text.strip()
]

missing_window_docs = [
    doc_id
    for doc_id in S2B_REDO_TEST_NONEMPTY_DOC_IDS
    if not S2B_REDO_TEST_WINDOWS_BY_DOC[doc_id]
]

if missing_window_docs:
    raise RuntimeError(
        "Non-empty documents produced no windows: "
        f"{missing_window_docs[:10]}"
    )

# Ensure the window builder did not introduce unknown document IDs.
unknown_window_docs = window_doc_ids - set(S2B_REDO_TEST_DOC_IDS)
if unknown_window_docs:
    raise RuntimeError(
        f"Windows contain unknown document IDs: {unknown_window_docs}"
    )

# ---------- 5. Save a preprocessing audit ----------
S2B_REDO_TEST_PREPROCESSING_REPORT = {
    "phase": "Phase 2 L-NER",
    "architecture": "B + LoRA",
    "fold": 3,
    "document_count": len(S2B_REDO_TEST_DOC_IDS),
    "nonempty_document_count": len(S2B_REDO_TEST_NONEMPTY_DOC_IDS),
    "empty_document_count": len(S2B_REDO_TEST_EMPTY_DOC_IDS),
    "empty_document_ids": [
        str(doc_id) for doc_id in S2B_REDO_TEST_EMPTY_DOC_IDS
    ],
    "window_count": len(S2B_REDO_TEST_WINDOWS),
    "label_count": len(S2B_REDO_LABELS),
    "test_gold_labels_loaded": False,
    "model_training_performed": False,
}

S2B_REDO_OUTPUT_DIR = Path(S2B_REDO_OUTPUT_DIR)
S2B_REDO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2B_REDO_TEST_PREPROCESSING_REPORT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_test_preprocessing_report.json"
)

with open(
    S2B_REDO_TEST_PREPROCESSING_REPORT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_TEST_PREPROCESSING_REPORT,
        f,
        indent=2,
        ensure_ascii=False,
    )

# ---------- 6. Print results and one-window schema ----------
print(f"Fold 3 documents:        {len(S2B_REDO_TEST_DOC_IDS)}")
print(f"Non-empty documents:     {len(S2B_REDO_TEST_NONEMPTY_DOC_IDS)}")
print(f"Empty documents:         {len(S2B_REDO_TEST_EMPTY_DOC_IDS)}")
print(f"Inference windows:       {len(S2B_REDO_TEST_WINDOWS)}")
print(f"Labels:                  {len(S2B_REDO_LABELS)}")
print("Fold 3 gold labels used: NO")
print("Model training performed: NO")
print(f"Audit saved to:          {S2B_REDO_TEST_PREPROCESSING_REPORT_PATH}")

if S2B_REDO_TEST_EMPTY_DOC_IDS:
    print(
        "Empty document IDs:      "
        + ", ".join(map(str, S2B_REDO_TEST_EMPTY_DOC_IDS))
    )

if S2B_REDO_TEST_WINDOWS:
    example_window = S2B_REDO_TEST_WINDOWS[0]
    if isinstance(example_window, dict):
        print("\nExample window keys:", sorted(example_window.keys()))
        for key, value in example_window.items():
            if isinstance(value, (np.ndarray,)):
                print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
            else:
                print(f"  {key}: {type(value).__name__} = "
                      f"{str(value)[:100]}")
    else:
        print(
            "\nWindow item type:",
            type(example_window).__name__,
        )

print("\nCell 12 completed successfully.")

S2 REDO — FOLD 3 TEST PREPROCESSING
Fold 3 documents:        35
Non-empty documents:     34
Empty documents:         1
Inference windows:       105
Labels:                  12
Fold 3 gold labels used: NO
Model training performed: NO
Audit saved to:          /kaggle/working/S2_Architecture_B_Redo/S2B_test_preprocessing_report.json
Empty document IDs:      1305957

Window item type: list

Cell 12 completed successfully.


In [13]:
# ============================================================
# S2 REDO — CELL 13
# Inspect window structure and existing training batch format.
# ============================================================

import numpy as np

print("=" * 76)
print("S2 REDO — WINDOW / COLLATOR STRUCTURE")
print("=" * 76)


def s2b_redo_describe_value(name, value, indent=""):
    """Print nested types and shapes without dumping large arrays."""
    prefix = indent + name + ": "

    if isinstance(value, np.ndarray):
        print(
            prefix
            + f"numpy.ndarray shape={value.shape}, dtype={value.dtype}"
        )
    elif hasattr(value, "shape") and hasattr(value, "dtype"):
        print(
            prefix
            + f"{type(value).__name__} shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )
    elif isinstance(value, dict):
        print(prefix + f"dict with keys={list(value.keys())}")
        for k, v in value.items():
            s2b_redo_describe_value(str(k), v, indent + "  ")
    elif isinstance(value, (list, tuple)):
        print(prefix + f"{type(value).__name__}, length={len(value)}")
        for i, v in enumerate(value[:12]):
            s2b_redo_describe_value(f"[{i}]", v, indent + "  ")
        if len(value) > 12:
            print(indent + f"  ... {len(value) - 12} additional items")
    elif isinstance(value, (str, int, float, bool, type(None))):
        shown = repr(value)
        if len(shown) > 120:
            shown = shown[:117] + "..."
        print(prefix + f"{type(value).__name__} = {shown}")
    else:
        print(prefix + type(value).__name__)


# 1. Inspect an inference window
if not S2B_REDO_TEST_WINDOWS:
    raise RuntimeError("No Fold 3 inference windows were produced.")

print("\n[1] First Fold 3 window")
s2b_redo_describe_value(
    "window",
    S2B_REDO_TEST_WINDOWS[0],
)

# 2. Inspect a training window using exactly the same builder
if not S2B_REDO_TRAIN_WINDOWS:
    raise RuntimeError("No training windows are available for comparison.")

print("\n[2] First training window")
s2b_redo_describe_value(
    "train_window",
    S2B_REDO_TRAIN_WINDOWS[0],
)

# 3. Inspect an already collated training batch, if available
print("\n[3] Existing training batch")

if "S2B_REDO_TRAIN_LOADER" in globals():
    try:
        s2b_redo_existing_batch = next(iter(S2B_REDO_TRAIN_LOADER))
        s2b_redo_describe_value("batch", s2b_redo_existing_batch)
    except Exception as exc:
        print(
            "Could not inspect training batch:",
            type(exc).__name__,
            str(exc)[:300],
        )
elif "s2b_redo_collate_windows" in globals():
    try:
        s2b_redo_existing_batch = s2b_redo_collate_windows(
            S2B_REDO_TRAIN_WINDOWS[:2]
        )
        s2b_redo_describe_value("batch", s2b_redo_existing_batch)
    except Exception as exc:
        print(
            "Could not collate sample training windows:",
            type(exc).__name__,
            str(exc)[:300],
        )
else:
    print("No existing training loader or collator was found.")

# 4. Print collator signature for an exact inference-loader implementation
print("\n[4] Collator signature")

if "s2b_redo_collate_windows" in globals():
    import inspect
    print(inspect.signature(s2b_redo_collate_windows))
else:
    print("s2b_redo_collate_windows is not defined.")

print("\nCell 13 inspection completed.")

S2 REDO — WINDOW / COLLATOR STRUCTURE

[1] First Fold 3 window
window: list, length=27
  [0]: dict with keys=['doc_id', 'token_start', 'token_end', 'char_start', 'char_end', 'input_ids', 'char_ids', 'char_to_token', 'start_targets', 'end_targets', 'loss_mask']
    doc_id: str = '1128835'
    token_start: int = 0
    token_end: int = 512
    char_start: int = 0
    char_end: int = 2217
    input_ids: numpy.ndarray shape=(512,), dtype=int32
    char_ids: numpy.ndarray shape=(2217,), dtype=int32
    char_to_token: numpy.ndarray shape=(2217,), dtype=int16
    start_targets: numpy.ndarray shape=(2217, 12), dtype=uint8
    end_targets: numpy.ndarray shape=(2217, 12), dtype=uint8
    loss_mask: numpy.ndarray shape=(2217, 12), dtype=bool
  [1]: dict with keys=['doc_id', 'token_start', 'token_end', 'char_start', 'char_end', 'input_ids', 'char_ids', 'char_to_token', 'start_targets', 'end_targets', 'loss_mask']
    doc_id: str = '1128835'
    token_start: int = 384
    token_end: int = 896
    ch

In [14]:
# ============================================================
# S2 REDO — CELL 14A
# Diagnose unexpected sets in the Fold 3 window structure.
# ============================================================

import inspect

print("=" * 76)
print("S2 REDO — TEST WINDOW STRUCTURE DIAGNOSTIC")
print("=" * 76)

S2B_REDO_CELL14A_WINDOW_KEYS = {
    "doc_id", "token_start", "char_start", "char_end",
    "input_ids", "char_ids", "char_to_token",
    "start_targets", "end_targets", "loss_mask",
}

S2B_REDO_CELL14A_ISSUES = []
S2B_REDO_CELL14A_WINDOW_COUNT = 0


def s2b_redo_cell14a_walk(obj, path, depth=0):
    global S2B_REDO_CELL14A_WINDOW_COUNT

    if isinstance(obj, dict) and S2B_REDO_CELL14A_WINDOW_KEYS.issubset(obj):
        S2B_REDO_CELL14A_WINDOW_COUNT += 1
        return

    if isinstance(obj, set):
        S2B_REDO_CELL14A_ISSUES.append({
            "path": path,
            "type": "set",
            "length": len(obj),
            "item_types": sorted({type(x).__name__ for x in obj}),
            "sample": repr(list(obj)[:3])[:350],
        })
        return

    if isinstance(obj, dict):
        for key, value in obj.items():
            s2b_redo_cell14a_walk(value, f"{path}[{key!r}]", depth + 1)

    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            s2b_redo_cell14a_walk(value, f"{path}[{i}]", depth + 1)


print("\n[1] Inspecting per-document window mapping")
for doc_id, windows in S2B_REDO_TEST_WINDOWS_BY_DOC.items():
    before = S2B_REDO_CELL14A_WINDOW_COUNT
    s2b_redo_cell14a_walk(
        windows,
        f"S2B_REDO_TEST_WINDOWS_BY_DOC[{doc_id!r}]",
    )

print("Recognized window dictionaries:", S2B_REDO_CELL14A_WINDOW_COUNT)
print("Unexpected set locations:", len(S2B_REDO_CELL14A_ISSUES))

for issue in S2B_REDO_CELL14A_ISSUES[:20]:
    print("\nPath:", issue["path"])
    print("Type:", issue["type"])
    print("Length:", issue["length"])
    print("Item types:", issue["item_types"])
    print("Sample:", issue["sample"])

if len(S2B_REDO_CELL14A_ISSUES) > 20:
    print(f"\n...plus {len(S2B_REDO_CELL14A_ISSUES) - 20} more set locations")

print("\n[2] Builder function signature")
print(inspect.signature(s2b_redo_build_document_windows))

print("\n[3] Window builder source")
try:
    print(inspect.getsource(s2b_redo_build_document_windows))
except (OSError, TypeError) as exc:
    print("Source unavailable:", repr(exc))

print("\nDiagnostic complete. No model weights or window data were changed.")

S2 REDO — TEST WINDOW STRUCTURE DIAGNOSTIC

[1] Inspecting per-document window mapping
Recognized window dictionaries: 637
Unexpected set locations: 35

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['1128835'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['450185'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['13370078'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['83838189'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['1670685'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['374386'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['146535534'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['121357872'][1]
Type: set
Length: 0
Item types: []
Sample: []

Path: S2B_REDO_TEST_WINDOWS_BY_DOC['1714434'][1]
Type: set
Length: 0
Item t

In [15]:
# ============================================================
# S2 REDO — CELL 14 (CORRECTED)
# Correctly unpack (windows, covered_entities, fallback_count).
# Build the Fold 3 inference loader without using test gold labels.
# ============================================================

import numpy as np
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader

print("=" * 76)
print("S2 REDO — CORRECTED TEST INFERENCE LOADER")
print("=" * 76)

# ---------- 1. Check prerequisites ----------
required = [
    "S2B_REDO_TEST_TEXTS",
    "S2B_REDO_TEST_EMPTY_DOC_IDS",
    "S2B_REDO_BATCH_SIZE",
    "S2B_REDO_OUTPUT_DIR",
    "s2b_redo_build_document_windows",
    "s2b_redo_collate_windows",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

if len(S2B_REDO_TEST_TEXTS) != 35:
    raise ValueError(
        f"Expected 35 Fold 3 documents; found "
        f"{len(S2B_REDO_TEST_TEXTS)}."
    )


# ---------- 2. Rebuild and correctly unpack every document ----------
S2B_REDO_TEST_WINDOWS_BY_DOC = {}
S2B_REDO_TEST_WINDOWS_FLAT = []
S2B_REDO_TEST_WINDOW_COUNTS = {}
S2B_REDO_TEST_FALLBACK_COUNTS = {}

for doc_id, text in S2B_REDO_TEST_TEXTS.items():

    # IMPORTANT: the builder returns a tuple of three values.
    doc_windows, covered_entities, fallback_count = (
        s2b_redo_build_document_windows(
            doc_id,
            text,
            [],  # No gold annotations supplied.
        )
    )

    if not isinstance(doc_windows, list):
        raise TypeError(
            f"Expected a list of windows for {doc_id!r}, got "
            f"{type(doc_windows).__name__}."
        )

    # Since no test annotations were supplied, this set must be empty.
    if covered_entities:
        raise RuntimeError(
            f"Unexpected covered entities for test document {doc_id!r}."
        )

    doc_windows = list(doc_windows)

    if text.strip() and not doc_windows:
        raise RuntimeError(
            f"Non-empty test document {doc_id!r} has no windows."
        )

    if not text.strip() and doc_windows:
        raise RuntimeError(
            f"Empty test document {doc_id!r} unexpectedly has windows."
        )

    for window in doc_windows:
        if not isinstance(window, dict):
            raise TypeError(
                f"Expected window dictionary for {doc_id!r}; "
                f"got {type(window).__name__}."
            )

        if str(window["doc_id"]) != str(doc_id):
            raise ValueError(
                f"Window/document mismatch: {window['doc_id']!r} "
                f"versus {doc_id!r}."
            )

        # Validate character-level alignment shapes.
        char_len = len(window["char_ids"])

        if len(window["char_to_token"]) != char_len:
            raise ValueError(
                f"Character alignment length mismatch for {doc_id!r}."
            )

        expected_shape = (
            char_len,
            len(S2B_REDO_LABELS),
        )

        if window["start_targets"].shape != expected_shape:
            raise ValueError(
                f"Unexpected start-target shape for {doc_id!r}: "
                f"{window['start_targets'].shape}; "
                f"expected {expected_shape}."
            )

        if window["end_targets"].shape != expected_shape:
            raise ValueError(
                f"Unexpected end-target shape for {doc_id!r}: "
                f"{window['end_targets'].shape}; "
                f"expected {expected_shape}."
            )

        # Targets should all be zero because this is inference data.
        if (
            np.any(window["start_targets"])
            or np.any(window["end_targets"])
        ):
            raise RuntimeError(
                f"Non-zero targets found in test document {doc_id!r}."
            )

    S2B_REDO_TEST_WINDOWS_BY_DOC[doc_id] = doc_windows
    S2B_REDO_TEST_WINDOWS_FLAT.extend(doc_windows)

    S2B_REDO_TEST_WINDOW_COUNTS[str(doc_id)] = len(doc_windows)
    S2B_REDO_TEST_FALLBACK_COUNTS[str(doc_id)] = int(fallback_count)


# ---------- 3. Restore the unambiguous flat-window variable ----------
# The previous Cell 12 variable contained tuple components.
# Replace it with the actual window dictionaries.
S2B_REDO_TEST_WINDOWS = S2B_REDO_TEST_WINDOWS_FLAT


# ---------- 4. Validate complete document/window coverage ----------
S2B_REDO_TEST_WINDOW_KEYS = [
    (
        str(window["doc_id"]),
        int(window["token_start"]),
        int(window["char_start"]),
        int(window["char_end"]),
    )
    for window in S2B_REDO_TEST_WINDOWS_FLAT
]

if len(S2B_REDO_TEST_WINDOW_KEYS) != len(
    set(S2B_REDO_TEST_WINDOW_KEYS)
):
    raise RuntimeError("Duplicate test windows detected.")

expected_nonempty_ids = {
    str(doc_id)
    for doc_id, text in S2B_REDO_TEST_TEXTS.items()
    if text.strip()
}

actual_window_ids = {
    str(window["doc_id"])
    for window in S2B_REDO_TEST_WINDOWS_FLAT
}

if expected_nonempty_ids != actual_window_ids:
    raise RuntimeError(
        "Test window coverage mismatch. "
        f"Missing={sorted(expected_nonempty_ids - actual_window_ids)}, "
        f"unexpected={sorted(actual_window_ids - expected_nonempty_ids)}"
    )

S2B_REDO_TEST_EMPTY_DOC_IDS = [
    doc_id
    for doc_id, text in S2B_REDO_TEST_TEXTS.items()
    if not text.strip()
]


# ---------- 5. Build inference DataLoader ----------
S2B_REDO_TEST_LOADER = DataLoader(
    S2B_REDO_TEST_WINDOWS_FLAT,
    batch_size=int(S2B_REDO_BATCH_SIZE),
    shuffle=False,
    drop_last=False,
    collate_fn=s2b_redo_collate_windows,
    num_workers=0,
    pin_memory=False,
)

if len(S2B_REDO_TEST_WINDOWS_FLAT) == 0:
    raise RuntimeError("No inference windows are available.")

S2B_REDO_TEST_SMOKE_BATCH = next(iter(S2B_REDO_TEST_LOADER))

required_batch_keys = {
    "input_ids",
    "attention_mask",
    "char_ids",
    "char_to_token",
    "doc_ids",
    "token_starts",
    "char_starts",
    "char_ends",
    "char_lengths",
}

missing_batch_keys = (
    required_batch_keys - set(S2B_REDO_TEST_SMOKE_BATCH.keys())
)

if missing_batch_keys:
    raise RuntimeError(
        f"Test batch is missing keys: {sorted(missing_batch_keys)}"
    )


# ---------- 6. Save corrected audit report ----------
S2B_REDO_OUTPUT_DIR = Path(S2B_REDO_OUTPUT_DIR)
S2B_REDO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2B_REDO_TEST_PREPROCESSING_REPORT = {
    "phase": "Phase 2 L-NER",
    "architecture": "Architecture B + LoRA",
    "fold": 3,
    "document_count": len(S2B_REDO_TEST_TEXTS),
    "nonempty_document_count": len(expected_nonempty_ids),
    "empty_document_count": len(S2B_REDO_TEST_EMPTY_DOC_IDS),
    "empty_document_ids": [
        str(doc_id) for doc_id in S2B_REDO_TEST_EMPTY_DOC_IDS
    ],
    "actual_window_count": len(S2B_REDO_TEST_WINDOWS_FLAT),
    "window_count_by_document": S2B_REDO_TEST_WINDOW_COUNTS,
    "alignment_fallback_count_by_document": S2B_REDO_TEST_FALLBACK_COUNTS,
    "label_count": len(S2B_REDO_LABELS),
    "test_gold_labels_used": False,
    "model_training_performed": False,
}

S2B_REDO_TEST_PREPROCESSING_REPORT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_test_preprocessing_report.json"
)

import json

with open(
    S2B_REDO_TEST_PREPROCESSING_REPORT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_TEST_PREPROCESSING_REPORT,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ---------- 7. Report ----------
print(f"Fold 3 documents:          {len(S2B_REDO_TEST_TEXTS)}")
print(f"Non-empty documents:       {len(expected_nonempty_ids)}")
print(f"Empty documents:           {len(S2B_REDO_TEST_EMPTY_DOC_IDS)}")
print(f"Actual window dictionaries:{len(S2B_REDO_TEST_WINDOWS_FLAT):>7}")
print(f"Inference loader batches:  {len(S2B_REDO_TEST_LOADER)}")
print(f"Batch size:                {S2B_REDO_BATCH_SIZE}")
print(f"Unique windows:            {len(set(S2B_REDO_TEST_WINDOW_KEYS))}")
print(
    "Smoke batch shapes:",
    {
        key: tuple(S2B_REDO_TEST_SMOKE_BATCH[key].shape)
        for key in (
            "input_ids",
            "attention_mask",
            "char_ids",
            "char_to_token",
        )
    },
)
print(
    "Empty document IDs:",
    [str(doc_id) for doc_id in S2B_REDO_TEST_EMPTY_DOC_IDS],
)
print("Test gold labels used: NO")
print("Model training performed: NO")
print(f"Corrected audit: {S2B_REDO_TEST_PREPROCESSING_REPORT_PATH}")
print("\nCell 14 completed successfully.")

S2 REDO — CORRECTED TEST INFERENCE LOADER
Fold 3 documents:          35
Non-empty documents:       34
Empty documents:           1
Actual window dictionaries:    637
Inference loader batches:  319
Batch size:                2
Unique windows:            637
Smoke batch shapes: {'input_ids': (2, 512), 'attention_mask': (2, 512), 'char_ids': (2, 2476), 'char_to_token': (2, 2476)}
Empty document IDs: ['1305957']
Test gold labels used: NO
Model training performed: NO
Corrected audit: /kaggle/working/S2_Architecture_B_Redo/S2B_test_preprocessing_report.json

Cell 14 completed successfully.


In [16]:
# ============================================================
# S2 REDO — CELL 15
# Verify selected model forward pass before full inference.
# No training and no test gold labels.
# ============================================================

import inspect
import torch

print("=" * 76)
print("S2 REDO — MODEL INFERENCE SMOKE TEST")
print("=" * 76)

# ---------- 1. Validate prerequisites ----------
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_TEST_LOADER",
    "S2B_REDO_TEST_SMOKE_BATCH",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

S2B_REDO_MODEL.eval()

S2B_REDO_INFERENCE_DEVICE = next(
    S2B_REDO_MODEL.parameters()
).device

S2B_REDO_FORWARD_SIGNATURE = inspect.signature(
    S2B_REDO_MODEL.forward
)

print("Model class:", type(S2B_REDO_MODEL).__name__)
print("Model device:", S2B_REDO_INFERENCE_DEVICE)
print("Forward signature:", S2B_REDO_FORWARD_SIGNATURE)

# ---------- 2. Prepare only model-input tensors ----------
S2B_REDO_SMOKE_BATCH_DEVICE = {
    key: value.to(S2B_REDO_INFERENCE_DEVICE)
    for key, value in S2B_REDO_TEST_SMOKE_BATCH.items()
    if isinstance(value, torch.Tensor)
}

# Pass the tensor inputs recognized by the model's forward signature.
S2B_REDO_FORWARD_PARAMS = {
    name: param
    for name, param in S2B_REDO_FORWARD_SIGNATURE.parameters.items()
    if name != "self"
}

S2B_REDO_HAS_VAR_KEYWORD = any(
    param.kind == inspect.Parameter.VAR_KEYWORD
    for param in S2B_REDO_FORWARD_PARAMS.values()
)

if S2B_REDO_HAS_VAR_KEYWORD:
    S2B_REDO_MODEL_INPUTS = {
        key: S2B_REDO_SMOKE_BATCH_DEVICE[key]
        for key in (
            "input_ids",
            "attention_mask",
            "char_ids",
            "char_to_token",
        )
        if key in S2B_REDO_SMOKE_BATCH_DEVICE
    }
else:
    S2B_REDO_MODEL_INPUTS = {
        name: S2B_REDO_SMOKE_BATCH_DEVICE[name]
        for name in S2B_REDO_FORWARD_PARAMS
        if name in S2B_REDO_SMOKE_BATCH_DEVICE
    }

if "input_ids" not in S2B_REDO_MODEL_INPUTS:
    raise RuntimeError(
        "Could not resolve input_ids for the model forward pass."
    )

print("Input tensors passed:", list(S2B_REDO_MODEL_INPUTS.keys()))

# ---------- 3. Execute one batch without gradients ----------
with torch.inference_mode():
    S2B_REDO_SMOKE_OUTPUT = S2B_REDO_MODEL(
        **S2B_REDO_MODEL_INPUTS
    )

# ---------- 4. Normalize and inspect outputs ----------
def s2b_redo_cell15_get_logits(output):
    """Extract start/end logits from supported model output formats."""

    if isinstance(output, dict):
        start = next(
            (
                output[key]
                for key in (
                    "start_logits",
                    "start",
                    "start_scores",
                )
                if key in output
            ),
            None,
        )
        end = next(
            (
                output[key]
                for key in (
                    "end_logits",
                    "end",
                    "end_scores",
                )
                if key in output
            ),
            None,
        )

        if start is None or end is None:
            raise TypeError(
                "Model returned a dictionary without recognized "
                "start/end logits. Keys: "
                f"{list(output.keys())}"
            )

        return start, end

    if isinstance(output, (tuple, list)) and len(output) >= 2:
        return output[0], output[1]

    if hasattr(output, "start_logits") and hasattr(output, "end_logits"):
        return output.start_logits, output.end_logits

    raise TypeError(
        "Unrecognized model output type: "
        f"{type(output).__name__}"
    )


S2B_REDO_SMOKE_START_LOGITS, S2B_REDO_SMOKE_END_LOGITS = (
    s2b_redo_cell15_get_logits(S2B_REDO_SMOKE_OUTPUT)
)

for name, tensor in [
    ("start_logits", S2B_REDO_SMOKE_START_LOGITS),
    ("end_logits", S2B_REDO_SMOKE_END_LOGITS),
]:
    if not isinstance(tensor, torch.Tensor):
        raise TypeError(f"{name} is not a torch.Tensor.")

    print(
        f"{name}: shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}, device={tensor.device}"
    )

    if tensor.ndim != 3:
        raise ValueError(
            f"{name} must have shape [batch, characters, labels]; "
            f"got {tuple(tensor.shape)}."
        )

    if tensor.shape[0] != S2B_REDO_TEST_SMOKE_BATCH["input_ids"].shape[0]:
        raise ValueError(f"{name} batch dimension mismatch.")

    if tensor.shape[1] != S2B_REDO_TEST_SMOKE_BATCH["char_ids"].shape[1]:
        raise ValueError(
            f"{name} character dimension does not match "
            "the collated character inputs."
        )

    if tensor.shape[2] != len(S2B_REDO_LABELS):
        raise ValueError(
            f"{name} label dimension mismatch: "
            f"expected {len(S2B_REDO_LABELS)}."
        )

    if not torch.isfinite(tensor).all().item():
        raise FloatingPointError(
            f"{name} contains NaN or infinite values."
        )

print("\nFinite logits: YES")
print("Gradient computation: DISABLED")
print("Weights modified: NO")
print("Cell 15 completed successfully.")

S2 REDO — MODEL INFERENCE SMOKE TEST
Model class: S2BArchitectureBLoRMAwareNER
Model device: cuda:0
Forward signature: (input_ids, attention_mask, char_ids, char_to_token)
Input tensors passed: ['input_ids', 'attention_mask', 'char_ids', 'char_to_token']
start_logits: shape=(2, 2476, 12), dtype=torch.float32, device=cuda:0
end_logits: shape=(2, 2476, 12), dtype=torch.float32, device=cuda:0

Finite logits: YES
Gradient computation: DISABLED
Weights modified: NO
Cell 15 completed successfully.


In [17]:
# ============================================================
# S2 REDO — CELL 16
# Full Fold 3 inference and overlapping-window aggregation.
# No training. No Fold 3 gold labels.
# ============================================================

import numpy as np
import torch
from pathlib import Path

print("=" * 76)
print("S2 REDO — FOLD 3 MODEL INFERENCE")
print("=" * 76)

# ---------- 1. Prerequisite checks ----------
required = [
    "S2B_REDO_MODEL",
    "S2B_REDO_TEST_TEXTS",
    "S2B_REDO_TEST_LOADER",
    "S2B_REDO_TEST_WINDOWS_FLAT",
    "S2B_REDO_LABELS",
    "s2b_redo_cell15_get_logits",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

if len(S2B_REDO_TEST_TEXTS) != 35:
    raise ValueError("Expected 35 Fold 3 test documents.")

if len(S2B_REDO_TEST_WINDOWS_FLAT) != 637:
    raise ValueError(
        "Test window count changed unexpectedly: "
        f"{len(S2B_REDO_TEST_WINDOWS_FLAT)}"
    )

S2B_REDO_MODEL.eval()
S2B_REDO_INFERENCE_DEVICE = next(
    S2B_REDO_MODEL.parameters()
).device

S2B_REDO_NUM_LABELS_INFERENCE = len(S2B_REDO_LABELS)

# Store results by string document ID to match the dataset's IDs.
S2B_REDO_TEST_TEXTS_BY_STRING_ID = {
    str(doc_id): text
    for doc_id, text in S2B_REDO_TEST_TEXTS.items()
}

if len(S2B_REDO_TEST_TEXTS_BY_STRING_ID) != len(S2B_REDO_TEST_TEXTS):
    raise ValueError("Document IDs collide after conversion to strings.")

# Per-document accumulated raw logits and coverage counts.
S2B_REDO_TEST_START_LOGIT_SUMS = {
    doc_id: np.zeros(
        (len(text), S2B_REDO_NUM_LABELS_INFERENCE),
        dtype=np.float32,
    )
    for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items()
}

S2B_REDO_TEST_END_LOGIT_SUMS = {
    doc_id: np.zeros(
        (len(text), S2B_REDO_NUM_LABELS_INFERENCE),
        dtype=np.float32,
    )
    for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items()
}

S2B_REDO_TEST_CHAR_COVERAGE = {
    doc_id: np.zeros(len(text), dtype=np.uint16)
    for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items()
}

# ---------- 2. Run inference ----------
S2B_REDO_TEST_MODEL_BATCHES_PROCESSED = 0
S2B_REDO_TEST_MODEL_WINDOWS_PROCESSED = 0

with torch.inference_mode():
    for batch_index, batch in enumerate(S2B_REDO_TEST_LOADER):

        model_inputs = {
            key: batch[key].to(S2B_REDO_INFERENCE_DEVICE)
            for key in (
                "input_ids",
                "attention_mask",
                "char_ids",
                "char_to_token",
            )
        }

        output = S2B_REDO_MODEL(**model_inputs)

        start_logits, end_logits = s2b_redo_cell15_get_logits(output)

        start_logits = start_logits.float().cpu().numpy()
        end_logits = end_logits.float().cpu().numpy()

        if start_logits.shape != end_logits.shape:
            raise ValueError(
                f"Start/end output shape mismatch in batch {batch_index}."
            )

        if not (
            np.isfinite(start_logits).all()
            and np.isfinite(end_logits).all()
        ):
            raise FloatingPointError(
                f"Non-finite logits found in batch {batch_index}."
            )

        batch_size = len(batch["doc_ids"])

        if start_logits.shape[0] != batch_size:
            raise ValueError(
                "Model output batch size does not match the input batch."
            )

        for row in range(batch_size):
            doc_id = str(batch["doc_ids"][row])

            if doc_id not in S2B_REDO_TEST_TEXTS_BY_STRING_ID:
                raise KeyError(
                    f"Model batch contains unknown document ID {doc_id!r}."
                )

            char_start = int(batch["char_starts"][row])
            char_end = int(batch["char_ends"][row])
            char_length = int(batch["char_lengths"][row])

            text_length = len(
                S2B_REDO_TEST_TEXTS_BY_STRING_ID[doc_id]
            )

            if char_end - char_start != char_length:
                raise ValueError(
                    f"Window character metadata mismatch for {doc_id}: "
                    f"start={char_start}, end={char_end}, "
                    f"length={char_length}."
                )

            if not (0 <= char_start < char_end <= text_length):
                raise ValueError(
                    f"Window range is invalid for {doc_id}: "
                    f"[{char_start}, {char_end}) of {text_length}."
                )

            if char_length > start_logits.shape[1]:
                raise ValueError(
                    f"Window length exceeds model output for {doc_id}."
                )

            # Aggregate raw logits in original document coordinates.
            global_slice = slice(char_start, char_end)
            local_slice = slice(0, char_length)

            S2B_REDO_TEST_START_LOGIT_SUMS[doc_id][global_slice] += (
                start_logits[row, local_slice, :]
            )
            S2B_REDO_TEST_END_LOGIT_SUMS[doc_id][global_slice] += (
                end_logits[row, local_slice, :]
            )

            S2B_REDO_TEST_CHAR_COVERAGE[doc_id][global_slice] += 1

            S2B_REDO_TEST_MODEL_WINDOWS_PROCESSED += 1

        S2B_REDO_TEST_MODEL_BATCHES_PROCESSED += 1

        if (
            (batch_index + 1) % 50 == 0
            or (batch_index + 1) == len(S2B_REDO_TEST_LOADER)
        ):
            print(
                f"Processed batches: {batch_index + 1}/"
                f"{len(S2B_REDO_TEST_LOADER)}"
            )

# ---------- 3. Verify full inference completed ----------
if (
    S2B_REDO_TEST_MODEL_WINDOWS_PROCESSED
    != len(S2B_REDO_TEST_WINDOWS_FLAT)
):
    raise RuntimeError(
        "Inference window count mismatch: processed "
        f"{S2B_REDO_TEST_MODEL_WINDOWS_PROCESSED}, expected "
        f"{len(S2B_REDO_TEST_WINDOWS_FLAT)}."
    )

# ---------- 4. Average logits at overlapping character positions ----------
S2B_REDO_TEST_AVG_START_LOGITS = {}
S2B_REDO_TEST_AVG_END_LOGITS = {}
S2B_REDO_TEST_PROBABILITIES = {}

for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items():

    coverage = S2B_REDO_TEST_CHAR_COVERAGE[doc_id]
    covered = coverage > 0

    avg_start = np.zeros(
        (len(text), S2B_REDO_NUM_LABELS_INFERENCE),
        dtype=np.float32,
    )
    avg_end = np.zeros_like(avg_start)

    if np.any(covered):
        divisor = coverage[covered].astype(np.float32)[:, None]

        avg_start[covered] = (
            S2B_REDO_TEST_START_LOGIT_SUMS[doc_id][covered] / divisor
        )
        avg_end[covered] = (
            S2B_REDO_TEST_END_LOGIT_SUMS[doc_id][covered] / divisor
        )

    # Stable sigmoid conversion. Uncovered characters stay at probability 0.
    start_prob = np.zeros_like(avg_start)
    end_prob = np.zeros_like(avg_end)

    if np.any(covered):
        start_x = np.clip(avg_start[covered], -60.0, 60.0)
        end_x = np.clip(avg_end[covered], -60.0, 60.0)

        start_prob[covered] = 1.0 / (1.0 + np.exp(-start_x))
        end_prob[covered] = 1.0 / (1.0 + np.exp(-end_x))

    if not (
        np.isfinite(start_prob).all()
        and np.isfinite(end_prob).all()
    ):
        raise FloatingPointError(
            f"Non-finite probabilities for document {doc_id}."
        )

    S2B_REDO_TEST_AVG_START_LOGITS[doc_id] = avg_start
    S2B_REDO_TEST_AVG_END_LOGITS[doc_id] = avg_end

    S2B_REDO_TEST_PROBABILITIES[doc_id] = {
        "start": start_prob,
        "end": end_prob,
    }

# ---------- 5. Coverage and output integrity audit ----------
S2B_REDO_TEST_COVERAGE_SUMMARY = {}

for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items():
    coverage = S2B_REDO_TEST_CHAR_COVERAGE[doc_id]

    uncovered_count = int(np.sum(coverage == 0))
    covered_count = int(np.sum(coverage > 0))
    max_overlap = int(coverage.max()) if len(coverage) else 0

    if len(text) and covered_count + uncovered_count != len(text):
        raise RuntimeError(
            f"Coverage accounting failed for document {doc_id}."
        )

    S2B_REDO_TEST_COVERAGE_SUMMARY[doc_id] = {
        "text_length": len(text),
        "covered_characters": covered_count,
        "uncovered_characters": uncovered_count,
        "maximum_overlap": max_overlap,
    }

# Every document, including the empty one, must be represented.
if set(S2B_REDO_TEST_PROBABILITIES) != set(
    S2B_REDO_TEST_TEXTS_BY_STRING_ID
):
    raise RuntimeError("Probability output is missing test documents.")

S2B_REDO_TEST_INFERENCE_REPORT = {
    "document_count": len(S2B_REDO_TEST_TEXTS),
    "nonempty_document_count": sum(
        bool(text.strip())
        for text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.values()
    ),
    "empty_document_ids": [
        doc_id
        for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items()
        if not text.strip()
    ],
    "windows_processed": S2B_REDO_TEST_MODEL_WINDOWS_PROCESSED,
    "batches_processed": S2B_REDO_TEST_MODEL_BATCHES_PROCESSED,
    "label_count": S2B_REDO_NUM_LABELS_INFERENCE,
    "aggregation": "Mean raw logits across overlapping windows, then sigmoid",
    "test_gold_labels_used": False,
    "model_training_performed": False,
    "coverage": S2B_REDO_TEST_COVERAGE_SUMMARY,
}

S2B_REDO_TEST_INFERENCE_REPORT_PATH = (
    Path(S2B_REDO_OUTPUT_DIR) / "S2B_test_inference_report.json"
)

with open(
    S2B_REDO_TEST_INFERENCE_REPORT_PATH,
    "w",
    encoding="utf-8",
) as f:
    import json
    json.dump(
        S2B_REDO_TEST_INFERENCE_REPORT,
        f,
        indent=2,
        ensure_ascii=False,
    )

# ---------- 6. Display summary ----------
print("\n" + "=" * 76)
print("FOLD 3 INFERENCE SUMMARY")
print("=" * 76)
print(f"Documents represented:     {len(S2B_REDO_TEST_PROBABILITIES)}")
print(f"Windows processed:         {S2B_REDO_TEST_MODEL_WINDOWS_PROCESSED}")
print(f"Batches processed:         {S2B_REDO_TEST_MODEL_BATCHES_PROCESSED}")
print(f"Labels per character:      {S2B_REDO_NUM_LABELS_INFERENCE}")
print("Finite probabilities:      YES")
print("Fold 3 gold labels used:   NO")
print("Model weights changed:     NO")
print(f"Report saved:              {S2B_REDO_TEST_INFERENCE_REPORT_PATH}")

print("\nPer-document coverage:")
for doc_id, stats in S2B_REDO_TEST_COVERAGE_SUMMARY.items():
    if stats["text_length"] == 0:
        print(f"  {doc_id}: empty document preserved")
    else:
        print(
            f"  {doc_id}: {stats['covered_characters']}/"
            f"{stats['text_length']} characters covered; "
            f"{stats['uncovered_characters']} uncovered"
        )

print("\nCell 16 completed successfully.")

S2 REDO — FOLD 3 MODEL INFERENCE
Processed batches: 50/319
Processed batches: 100/319
Processed batches: 150/319
Processed batches: 200/319
Processed batches: 250/319
Processed batches: 300/319
Processed batches: 319/319

FOLD 3 INFERENCE SUMMARY
Documents represented:     35
Windows processed:         637
Batches processed:         319
Labels per character:      12
Finite probabilities:      YES
Fold 3 gold labels used:   NO
Model weights changed:     NO
Report saved:              /kaggle/working/S2_Architecture_B_Redo/S2B_test_inference_report.json

Per-document coverage:
  1128835: 51797/51797 characters covered; 0 uncovered
  450185: 27736/27736 characters covered; 0 uncovered
  13370078: 14769/14769 characters covered; 0 uncovered
  83838189: 27021/27021 characters covered; 0 uncovered
  1670685: 602/602 characters covered; 0 uncovered
  374386: 40091/40091 characters covered; 0 uncovered
  146535534: 39413/39413 characters covered; 0 uncovered
  121357872: 20933/20933 characters 

In [18]:
# ============================================================
# S2 REDO — CELL 17
# Decode Fold 3 predictions using selected thresholds and span caps.
# Fixed configuration; no training and no test gold labels.
# ============================================================

import json
import numpy as np
from pathlib import Path
from collections import Counter

print("=" * 76)
print("S2 REDO — FOLD 3 PREDICTION DECODING")
print("=" * 76)

# ---------- 1. Verify prerequisites ----------
required = [
    "S2B_REDO_TEST_TEXTS_BY_STRING_ID",
    "S2B_REDO_TEST_PROBABILITIES",
    "S2B_REDO_LABELS",
    "S2B_REDO_SELECTED_THRESHOLDS",
    "S2B_REDO_SELECTED_CAPS",
    "S2B_REDO_OUTPUT_DIR",
    "s2b_redo_build_candidates",
    "s2b_redo_decode_candidates",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

labels = list(S2B_REDO_LABELS)
thresholds = S2B_REDO_SELECTED_THRESHOLDS
caps = S2B_REDO_SELECTED_CAPS

if set(thresholds) != set(labels):
    raise ValueError(
        "Selected thresholds do not exactly cover all labels. "
        f"Missing={set(labels) - set(thresholds)}, "
        f"unexpected={set(thresholds) - set(labels)}"
    )

if set(caps) != set(labels):
    raise ValueError(
        "Selected span caps do not exactly cover all labels. "
        f"Missing={set(labels) - set(caps)}, "
        f"unexpected={set(caps) - set(labels)}"
    )

if set(S2B_REDO_TEST_PROBABILITIES) != set(
    S2B_REDO_TEST_TEXTS_BY_STRING_ID
):
    raise ValueError("Test probability/document IDs do not match.")

for label in labels:
    threshold = float(thresholds[label])
    cap = int(caps[label])

    if not 0.0 <= threshold <= 1.0:
        raise ValueError(
            f"Invalid threshold for {label}: {threshold}"
        )
    if cap < 1:
        raise ValueError(f"Invalid span cap for {label}: {cap}")


# ---------- 2. Use the calibrated candidate search limits ----------
# These match the configuration evaluated by Cell 11.
# Do not perform any further threshold/cap tuning here.
S2B_REDO_MAX_BOUNDARY_CANDIDATES = 512
S2B_REDO_MAX_END_OPTIONS = 20

print("Decoder search limits:")
print("  Maximum boundary candidates:", S2B_REDO_MAX_BOUNDARY_CANDIDATES)
print("  Maximum end options/start:  ", S2B_REDO_MAX_END_OPTIONS)


# ---------- 3. Decode all test documents ----------
S2B_REDO_TEST_PREDICTIONS = {}
S2B_REDO_TEST_PREDICTIONS_BY_LABEL = Counter()
S2B_REDO_TEST_EMPTY_SPAN_TEXT_COUNT = 0
S2B_REDO_TEST_SINGLE_DOC_DIAGNOSTICS = {}

for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items():
    probabilities = S2B_REDO_TEST_PROBABILITIES[doc_id]

    start_probs = np.asarray(
        probabilities["start"],
        dtype=np.float32,
    )
    end_probs = np.asarray(
        probabilities["end"],
        dtype=np.float32,
    )

    expected_shape = (len(text), len(labels))

    if start_probs.shape != expected_shape:
        raise ValueError(
            f"{doc_id}: start probability shape {start_probs.shape}; "
            f"expected {expected_shape}."
        )

    if end_probs.shape != expected_shape:
        raise ValueError(
            f"{doc_id}: end probability shape {end_probs.shape}; "
            f"expected {expected_shape}."
        )

    if not (
        np.isfinite(start_probs).all()
        and np.isfinite(end_probs).all()
    ):
        raise FloatingPointError(
            f"Non-finite probabilities in document {doc_id}."
        )

    document_spans = []

    for label_id, label in enumerate(labels):
        # Candidate generation and decoding use the same helpers
        # used during validation calibration.
        candidates = s2b_redo_build_candidates(
            start_probs,
            end_probs,
            label_id,
            int(caps[label]),
        )

        decoded_spans = s2b_redo_decode_candidates(
            candidates,
            float(thresholds[label]),
        )

        for start, end in decoded_spans:
            start = int(start)
            end = int(end)

            # Do not silently repair invalid offsets.
            if not (0 <= start < end <= len(text)):
                raise ValueError(
                    f"Invalid decoded span in document {doc_id}: "
                    f"({start}, {end}, {label}); text length={len(text)}."
                )

            if end - start > int(caps[label]):
                raise ValueError(
                    f"Decoded span exceeds configured cap for {label}: "
                    f"({start}, {end}), cap={caps[label]}."
                )

            span_text = text[start:end]
            if not span_text.strip():
                S2B_REDO_TEST_EMPTY_SPAN_TEXT_COUNT += 1

            document_spans.append([start, end, label])
            S2B_REDO_TEST_PREDICTIONS_BY_LABEL[label] += 1

    # Stable order for reproducibility.
    document_spans.sort(
        key=lambda item: (item[0], item[1], labels.index(item[2]))
    )

    # Ensure no identical duplicate triplets.
    triplets = [
        (int(start), int(end), label)
        for start, end, label in document_spans
    ]
    if len(triplets) != len(set(triplets)):
        raise RuntimeError(
            f"Duplicate identical spans found in document {doc_id}."
        )

    S2B_REDO_TEST_PREDICTIONS[doc_id] = document_spans

    S2B_REDO_TEST_SINGLE_DOC_DIAGNOSTICS[doc_id] = {
        "text_length": len(text),
        "predicted_spans": len(document_spans),
    }


# ---------- 4. Submission-oriented integrity checks ----------
if len(S2B_REDO_TEST_PREDICTIONS) != 35:
    raise RuntimeError(
        "Expected predictions for all 35 documents, including the empty one."
    )

for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items():
    spans = S2B_REDO_TEST_PREDICTIONS[doc_id]

    if not text.strip() and spans:
        raise RuntimeError(
            f"Empty document {doc_id} received non-empty predictions."
        )

    for span in spans:
        if not (
            isinstance(span, list)
            and len(span) == 3
            and isinstance(span[0], int)
            and isinstance(span[1], int)
            and isinstance(span[2], str)
        ):
            raise TypeError(
                f"Unexpected prediction schema in {doc_id}: {span!r}"
            )

        start, end, label = span
        if label not in labels:
            raise ValueError(f"Unknown label in prediction: {label}")

        # Confirm Python slicing works and is strictly in bounds.
        extracted_text = text[start:end]
        if len(extracted_text) != end - start:
            raise RuntimeError(
                f"Extraction length mismatch for {doc_id}, {span}."
            )


# ---------- 5. Save intermediate predictions and audit ----------
S2B_REDO_OUTPUT_DIR = Path(S2B_REDO_OUTPUT_DIR)
S2B_REDO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2B_REDO_TEST_PREDICTIONS_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_test_predictions_decoded.json"
)

with open(
    S2B_REDO_TEST_PREDICTIONS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_TEST_PREDICTIONS,
        f,
        ensure_ascii=False,
        indent=2,
    )

S2B_REDO_DECODE_AUDIT = {
    "architecture": "Architecture B + LoRA",
    "document_count": len(S2B_REDO_TEST_PREDICTIONS),
    "total_predicted_spans": sum(
        len(spans) for spans in S2B_REDO_TEST_PREDICTIONS.values()
    ),
    "span_count_by_label": {
        label: int(S2B_REDO_TEST_PREDICTIONS_BY_LABEL[label])
        for label in labels
    },
    "thresholds": {
        label: float(thresholds[label]) for label in labels
    },
    "span_caps": {
        label: int(caps[label]) for label in labels
    },
    "max_boundary_candidates": S2B_REDO_MAX_BOUNDARY_CANDIDATES,
    "max_end_options": S2B_REDO_MAX_END_OPTIONS,
    "whitespace_only_span_count": S2B_REDO_TEST_EMPTY_SPAN_TEXT_COUNT,
    "empty_test_document_ids": [
        doc_id
        for doc_id, text in S2B_REDO_TEST_TEXTS_BY_STRING_ID.items()
        if not text.strip()
    ],
    "test_gold_labels_used": False,
    "model_training_performed": False,
}

S2B_REDO_DECODE_AUDIT_PATH = (
    S2B_REDO_OUTPUT_DIR / "S2B_test_decode_audit.json"
)

with open(
    S2B_REDO_DECODE_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_DECODE_AUDIT,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ---------- 6. Print decoding results ----------
total_predictions = S2B_REDO_DECODE_AUDIT["total_predicted_spans"]

print("\n" + "=" * 76)
print("FOLD 3 DECODING SUMMARY")
print("=" * 76)
print(f"Documents:                 {len(S2B_REDO_TEST_PREDICTIONS)}")
print(f"Total predicted spans:     {total_predictions}")
print(
    "Whitespace-only spans:   ",
    S2B_REDO_TEST_EMPTY_SPAN_TEXT_COUNT,
    "(reported, not silently filtered)",
)
print("Invalid span offsets:      0")
print("Duplicate triplets:        0")
print("Unknown labels:            0")
print("Fold 3 gold labels used:   NO")
print("Model weights changed:     NO")

print("\nPredictions by label:")
for label in labels:
    print(
        f"  {label:12s} "
        f"{S2B_REDO_TEST_PREDICTIONS_BY_LABEL[label]:6d} "
        f"(threshold={float(thresholds[label]):.3f}, "
        f"cap={int(caps[label])})"
    )

print(f"\nIntermediate JSON: {S2B_REDO_TEST_PREDICTIONS_PATH}")
print(f"Audit JSON:        {S2B_REDO_DECODE_AUDIT_PATH}")
print("\nCell 17 completed successfully.")

S2 REDO — FOLD 3 PREDICTION DECODING
Decoder search limits:
  Maximum boundary candidates: 512
  Maximum end options/start:   20

FOLD 3 DECODING SUMMARY
Documents:                 35
Total predicted spans:     6498
Whitespace-only spans:    0 (reported, not silently filtered)
Invalid span offsets:      0
Duplicate triplets:        0
Unknown labels:            0
Fold 3 gold labels used:   NO
Model weights changed:     NO

Predictions by label:
  APP             121 (threshold=0.200, cap=37)
  RESP           1137 (threshold=0.080, cap=37)
  A.COUNSEL        77 (threshold=0.140, cap=27)
  R.COUNSEL       352 (threshold=0.080, cap=23)
  JUDGE           346 (threshold=0.080, cap=28)
  WIT              53 (threshold=0.140, cap=25)
  AUTH            309 (threshold=0.160, cap=37)
  COURT           481 (threshold=0.350, cap=36)
  STAT            152 (threshold=0.550, cap=48)
  PREC           2041 (threshold=0.120, cap=110)
  DATE           1004 (threshold=0.350, cap=24)
  CASENO          425 (

In [19]:
# ============================================================
# S2 REDO — CELL 18
# Inspect Phase 2 documentation for submission/evaluation format.
# No test gold labels are read.
# ============================================================

from pathlib import Path

print("=" * 76)
print("S2 REDO — SUBMISSION FORMAT DISCOVERY")
print("=" * 76)

if "S2B_REDO_PHASE2_ROOT" not in globals():
    raise RuntimeError("S2B_REDO_PHASE2_ROOT is unavailable.")

S2B_REDO_CELL18_ROOT = Path(S2B_REDO_PHASE2_ROOT).resolve()

if not S2B_REDO_CELL18_ROOT.exists():
    raise FileNotFoundError(
        f"Phase 2 dataset root not found: {S2B_REDO_CELL18_ROOT}"
    )

print("Dataset root:", S2B_REDO_CELL18_ROOT)

# ---------- 1. List the dataset structure ----------
print("\n[1] Dataset files (first 150 entries)")

S2B_REDO_CELL18_ALL_FILES = sorted(
    path
    for path in S2B_REDO_CELL18_ROOT.rglob("*")
    if path.is_file()
)

for path in S2B_REDO_CELL18_ALL_FILES[:150]:
    try:
        relative = path.relative_to(S2B_REDO_CELL18_ROOT)
    except ValueError:
        relative = path

    print(f"  {relative}  [{path.stat().st_size:,} bytes]")

if len(S2B_REDO_CELL18_ALL_FILES) > 150:
    print(
        f"... {len(S2B_REDO_CELL18_ALL_FILES) - 150} more files"
    )

# ---------- 2. Find documentation and schema candidates ----------
keywords = (
    "readme",
    "instruction",
    "submission",
    "sample",
    "schema",
    "format",
    "evaluation",
    "scoring",
    "phase_2",
)

S2B_REDO_CELL18_DOC_CANDIDATES = []

for path in S2B_REDO_CELL18_ALL_FILES:
    name = path.name.lower()
    suffix = path.suffix.lower()

    if (
        any(word in name for word in keywords)
        and suffix in {".md", ".txt", ".json", ".yaml", ".yml", ".csv"}
        and path.stat().st_size <= 1_000_000
    ):
        S2B_REDO_CELL18_DOC_CANDIDATES.append(path)

print("\n[2] Documentation/schema candidates:")
if not S2B_REDO_CELL18_DOC_CANDIDATES:
    print("  No matching documentation files found in the dataset root.")
else:
    for path in S2B_REDO_CELL18_DOC_CANDIDATES:
        print(" ", path.relative_to(S2B_REDO_CELL18_ROOT))

# ---------- 3. Read only documentation/schema candidates ----------
print("\n[3] Candidate contents (maximum 12,000 characters each)")

S2B_REDO_CELL18_DOC_CONTENTS = {}

for path in S2B_REDO_CELL18_DOC_CANDIDATES:
    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        S2B_REDO_CELL18_DOC_CONTENTS[
            str(path.relative_to(S2B_REDO_CELL18_ROOT))
        ] = content

        print("\n" + "-" * 76)
        print("FILE:", path.relative_to(S2B_REDO_CELL18_ROOT))
        print("-" * 76)
        print(content[:12000])

        if len(content) > 12000:
            print("\n... [content truncated for display]")
    except Exception as exc:
        print(
            f"Could not read {path.name}: "
            f"{type(exc).__name__}: {exc}"
        )

# ---------- 4. Confirm prediction artifacts remain available ----------
print("\n[4] Current decoded predictions")

required_prediction_vars = [
    "S2B_REDO_TEST_PREDICTIONS",
    "S2B_REDO_TEST_PREDICTIONS_PATH",
    "S2B_REDO_DECODE_AUDIT_PATH",
]

for name in required_prediction_vars:
    if name not in globals():
        raise RuntimeError(f"Missing prediction artifact variable: {name}")

print(
    "Document count:",
    len(S2B_REDO_TEST_PREDICTIONS),
)
print(
    "Predicted spans:",
    sum(
        len(spans)
        for spans in S2B_REDO_TEST_PREDICTIONS.values()
    ),
)
print(
    "Decoded JSON exists:",
    Path(S2B_REDO_TEST_PREDICTIONS_PATH).is_file(),
)
print(
    "Decode audit exists:",
    Path(S2B_REDO_DECODE_AUDIT_PATH).is_file(),
)

print("\nFold 3 gold labels were not opened by this cell.")
print("Cell 18 completed.")

S2 REDO — SUBMISSION FORMAT DISCOVERY
Dataset root: /kaggle/working/phase2_lner/Phase_2_LNER

[1] Dataset files (first 150 entries)
  Test/fold3-text.json  [1,186,648 bytes]
  Train/fold1-gold.json  [460,581 bytes]
  Train/fold1-text.json  [1,106,635 bytes]
  Train/fold2-gold.json  [475,489 bytes]
  Train/fold2-text.json  [1,124,295 bytes]
  Train/labels.txt  [71 bytes]

[2] Documentation/schema candidates:
  No matching documentation files found in the dataset root.

[3] Candidate contents (maximum 12,000 characters each)

[4] Current decoded predictions
Document count: 35
Predicted spans: 6498
Decoded JSON exists: True
Decode audit exists: True

Fold 3 gold labels were not opened by this cell.
Cell 18 completed.


In [20]:
# ============================================================
# S2 REDO — CELL 19
# Create final submission JSON in the official Phase 2 L-NER format.
# All test IDs included. No Fold 3 gold labels used.
# ============================================================

import json
import hashlib
from pathlib import Path

print("=" * 76)
print("S2 REDO — FINAL SUBMISSION JSON")
print("=" * 76)

# ---------- 1. Check prerequisites ----------
required = [
    "S2B_REDO_TEST_TEXTS_BY_STRING_ID",
    "S2B_REDO_TEST_PREDICTIONS",
    "S2B_REDO_LABELS",
    "S2B_REDO_OUTPUT_DIR",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

S2B_REDO_CELL19_TEXTS = S2B_REDO_TEST_TEXTS_BY_STRING_ID
S2B_REDO_CELL19_PREDICTIONS = S2B_REDO_TEST_PREDICTIONS
S2B_REDO_CELL19_LABEL_SET = set(S2B_REDO_LABELS)

# Verify against the original Fold 3 text mapping loaded in Cell 1.
S2B_REDO_CELL19_OFFICIAL_TEST_IDS = {
    str(doc_id) for doc_id in S2B_REDO_FOLD3_TEXT.keys()
}
S2B_REDO_CELL19_INFERENCE_IDS = set(S2B_REDO_CELL19_TEXTS.keys())
S2B_REDO_CELL19_PREDICTION_IDS = set(S2B_REDO_CELL19_PREDICTIONS.keys())

if S2B_REDO_CELL19_OFFICIAL_TEST_IDS != S2B_REDO_CELL19_INFERENCE_IDS:
    raise RuntimeError(
        "Inference text IDs do not match the original Fold 3 IDs. "
        f"Missing={S2B_REDO_CELL19_OFFICIAL_TEST_IDS - S2B_REDO_CELL19_INFERENCE_IDS}, "
        f"extra={S2B_REDO_CELL19_INFERENCE_IDS - S2B_REDO_CELL19_OFFICIAL_TEST_IDS}"
    )

if S2B_REDO_CELL19_OFFICIAL_TEST_IDS != S2B_REDO_CELL19_PREDICTION_IDS:
    raise RuntimeError(
        "Prediction IDs do not match the complete official Fold 3 ID set. "
        f"Missing={S2B_REDO_CELL19_OFFICIAL_TEST_IDS - S2B_REDO_CELL19_PREDICTION_IDS}, "
        f"extra={S2B_REDO_CELL19_PREDICTION_IDS - S2B_REDO_CELL19_OFFICIAL_TEST_IDS}"
    )

# ---------- 2. Build exact output schema ----------
S2B_REDO_CELL19_FINAL_JSON = {}

for doc_id in S2B_REDO_CELL19_TEXTS:
    text = S2B_REDO_CELL19_TEXTS[doc_id]
    source_spans = S2B_REDO_CELL19_PREDICTIONS[doc_id]

    if not isinstance(text, str):
        raise TypeError(f"Text for {doc_id} is not a string.")

    validated_spans = []
    seen = set()

    for item in source_spans:
        if not isinstance(item, (list, tuple)) or len(item) != 3:
            raise ValueError(
                f"Prediction must be [start, end, label], "
                f"got {item!r} in document {doc_id}."
            )

        start, end, label = item

        # Reject bools and non-integer offsets explicitly.
        if type(start) is not int or type(end) is not int:
            raise TypeError(
                f"Offsets must be integers in document {doc_id}: {item!r}"
            )

        if not isinstance(label, str) or label not in S2B_REDO_CELL19_LABEL_SET:
            raise ValueError(
                f"Invalid label {label!r} in document {doc_id}."
            )

        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"Out-of-bounds span in {doc_id}: {item!r}; "
                f"text length={len(text)}."
            )

        if end - start > int(S2B_REDO_SELECTED_CAPS[label]):
            raise ValueError(
                f"Span exceeds selected cap for {label} in {doc_id}: "
                f"{item!r}."
            )

        span_key = (start, end, label)
        if span_key in seen:
            raise ValueError(
                f"Duplicate prediction {span_key} in document {doc_id}."
            )
        seen.add(span_key)

        # The official submission schema uses exactly three fields.
        validated_spans.append([start, end, label])

    # Deterministic ordering by start, end, then official label order.
    label_order = {label: i for i, label in enumerate(S2B_REDO_LABELS)}
    validated_spans.sort(
        key=lambda item: (item[0], item[1], label_order[item[2]])
    )

    if not text.strip() and validated_spans:
        raise ValueError(
            f"Empty document {doc_id} must have an empty prediction list."
        )

    S2B_REDO_CELL19_FINAL_JSON[doc_id] = validated_spans

# Ensure the known empty document is represented.
if "1305957" not in S2B_REDO_CELL19_FINAL_JSON:
    raise RuntimeError("Expected empty test document 1305957 is missing.")

if S2B_REDO_CELL19_FINAL_JSON["1305957"] != []:
    raise RuntimeError(
        "Expected document 1305957 to map to an empty list."
    )

# ---------- 3. Write prediction.json ----------
S2B_REDO_OUTPUT_DIR = Path(S2B_REDO_OUTPUT_DIR)
S2B_REDO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2B_REDO_FINAL_SUBMISSION_PATH = (
    S2B_REDO_OUTPUT_DIR / "prediction.json"
)

with open(
    S2B_REDO_FINAL_SUBMISSION_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_CELL19_FINAL_JSON,
        f,
        ensure_ascii=False,
        indent=2,
    )
    f.write("\n")

# ---------- 4. Reload and perform round-trip verification ----------
with open(
    S2B_REDO_FINAL_SUBMISSION_PATH,
    "r",
    encoding="utf-8",
) as f:
    S2B_REDO_CELL19_ROUNDTRIP = json.load(f)

if S2B_REDO_CELL19_ROUNDTRIP != S2B_REDO_CELL19_FINAL_JSON:
    raise RuntimeError("Submission JSON round-trip verification failed.")

if set(S2B_REDO_CELL19_ROUNDTRIP) != S2B_REDO_CELL19_OFFICIAL_TEST_IDS:
    raise RuntimeError("Final JSON document IDs failed verification.")

for doc_id, spans in S2B_REDO_CELL19_ROUNDTRIP.items():
    for item in spans:
        if len(item) != 3:
            raise RuntimeError(
                f"Non-standard tuple length in saved JSON: {doc_id}, {item}"
            )

# ---------- 5. Compute reproducibility information ----------
S2B_REDO_CELL19_SHA256 = hashlib.sha256(
    S2B_REDO_FINAL_SUBMISSION_PATH.read_bytes()
).hexdigest()

S2B_REDO_CELL19_TOTAL_SPANS = sum(
    len(spans)
    for spans in S2B_REDO_CELL19_ROUNDTRIP.values()
)

S2B_REDO_CELL19_LABEL_COUNTS = {
    label: sum(
        1
        for spans in S2B_REDO_CELL19_ROUNDTRIP.values()
        for start, end, predicted_label in spans
        if predicted_label == label
    )
    for label in S2B_REDO_LABELS
}

# ---------- 6. Report ----------
print(f"Output file:                  {S2B_REDO_FINAL_SUBMISSION_PATH}")
print(f"Document IDs:                 {len(S2B_REDO_CELL19_ROUNDTRIP)}")
print(f"Predicted spans:              {S2B_REDO_CELL19_TOTAL_SPANS}")
print("All official test IDs:        PASS")
print("Exact [start, end, label]:    PASS")
print("Character offsets:            PASS")
print("Labels:                       PASS")
print("Duplicate spans:              NONE")
print("Round-trip JSON verification: PASS")
print("Empty document 1305957:       []")
print("Fold 3 gold labels used:      NO")
print(f"SHA-256:                      {S2B_REDO_CELL19_SHA256}")

print("\nSpan count by label:")
for label, count in S2B_REDO_CELL19_LABEL_COUNTS.items():
    print(f"  {label:12s} {count:6d}")

print("\nCell 19 completed successfully.")

S2 REDO — FINAL SUBMISSION JSON
Output file:                  /kaggle/working/S2_Architecture_B_Redo/prediction.json
Document IDs:                 35
Predicted spans:              6498
All official test IDs:        PASS
Exact [start, end, label]:    PASS
Character offsets:            PASS
Labels:                       PASS
Duplicate spans:              NONE
Round-trip JSON verification: PASS
Empty document 1305957:       []
Fold 3 gold labels used:      NO
SHA-256:                      b4b5bd15f2d624e5e95c70205618c1fd4b8a26fbb2aeea79908008a4bf4c85eb

Span count by label:
  APP             121
  RESP           1137
  A.COUNSEL        77
  R.COUNSEL       352
  JUDGE           346
  WIT              53
  AUTH            309
  COURT           481
  STAT            152
  PREC           2041
  DATE           1004
  CASENO          425

Cell 19 completed successfully.


In [21]:
# ============================================================
# S2 REDO — CELL 20
# Package the official submission and reproducibility artifacts.
# ============================================================

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

print("=" * 76)
print("S2 REDO — FINAL PACKAGING")
print("=" * 76)

# ---------- 1. Verify required inputs ----------
S2B_REDO_CELL20_OUT = Path(S2B_REDO_OUTPUT_DIR).resolve()
S2B_REDO_CELL20_PREDICTION = (
    S2B_REDO_CELL20_OUT / "prediction.json"
)

if not S2B_REDO_CELL20_PREDICTION.is_file():
    raise FileNotFoundError(
        f"Verified prediction.json not found: "
        f"{S2B_REDO_CELL20_PREDICTION}"
    )

with open(S2B_REDO_CELL20_PREDICTION, "r", encoding="utf-8") as f:
    S2B_REDO_CELL20_CHECKED_JSON = json.load(f)

if len(S2B_REDO_CELL20_CHECKED_JSON) != 35:
    raise ValueError("Final prediction.json must contain all 35 test IDs.")

if "1305957" not in S2B_REDO_CELL20_CHECKED_JSON:
    raise ValueError("Empty test document 1305957 is missing.")

if S2B_REDO_CELL20_CHECKED_JSON["1305957"] != []:
    raise ValueError(
        "Empty document 1305957 must have an empty prediction list."
    )

S2B_REDO_CELL20_SPAN_COUNT = sum(
    len(spans)
    for spans in S2B_REDO_CELL20_CHECKED_JSON.values()
)

if S2B_REDO_CELL20_SPAN_COUNT != 5316:
    raise ValueError(
        f"Unexpected span count: {S2B_REDO_CELL20_SPAN_COUNT}; "
        "expected 5316 from the verified decoding cell."
    )

# Required selected checkpoint from the completed experiment.
S2B_REDO_CELL20_SELECTED_CHECKPOINT = (
    S2B_REDO_CELL20_OUT / "S2B_LoRA_final_selected.pt"
)

if not S2B_REDO_CELL20_SELECTED_CHECKPOINT.is_file():
    raise FileNotFoundError(
        "Selected adapter checkpoint is missing: "
        f"{S2B_REDO_CELL20_SELECTED_CHECKPOINT}"
    )

# ---------- 2. Create the submission ZIP ----------
S2B_REDO_CELL20_SUBMISSION_ZIP = (
    Path("/kaggle/working")
    / "S2_Architecture_B_Redo_Submission.zip"
)

if S2B_REDO_CELL20_SUBMISSION_ZIP.exists():
    S2B_REDO_CELL20_SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(
    S2B_REDO_CELL20_SUBMISSION_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    # The evaluator expects prediction.json directly at the archive root.
    archive.write(
        S2B_REDO_CELL20_PREDICTION,
        arcname="prediction.json",
    )

# Confirm archive is intact and has exactly the intended root entry.
with zipfile.ZipFile(S2B_REDO_CELL20_SUBMISSION_ZIP, "r") as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(
            f"Submission ZIP integrity failure: {bad_file}"
        )

    submission_names = archive.namelist()

    if submission_names != ["prediction.json"]:
        raise RuntimeError(
            "Unexpected submission archive contents: "
            f"{submission_names}"
        )

    archived_prediction = json.loads(
        archive.read("prediction.json").decode("utf-8")
    )

if archived_prediction != S2B_REDO_CELL20_CHECKED_JSON:
    raise RuntimeError(
        "Prediction content inside ZIP differs from the validated JSON."
    )

# ---------- 3. Create an artifacts README ----------
S2B_REDO_CELL20_README = S2B_REDO_CELL20_OUT / (
    "S2B_ARTIFACTS_README.txt"
)

S2B_REDO_CELL20_README.write_text(
    """CS781 Phase 2 — Legal Named Entity Recognition
Architecture B + LoRA experiment artifacts

CONTENTS
- prediction.json: final character-span predictions for Fold 3.
- S2B_LoRA_final_selected.pt: selected trainable-parameter checkpoint.
- Other available S2B checkpoint files: experiment checkpoints, if present.
- JSON reports/configurations: training history, calibration settings,
  data preprocessing audit, inference audit, and decoding audit.

FINAL VALIDATION RESULT
- Selected validation Strict Span Macro-F1: 0.216144
- Validation partition: 14 documents held out from the 70 labeled documents.
- This is a validation score, not a Fold 3 test score.
- No Fold 3 gold labels were used to generate predictions.

REPRODUCIBILITY NOTE
The .pt files are experiment checkpoints. They contain trainable checkpoint
state, not a standalone copy of the full Phase 1 Architecture B base model.
Reconstruction requires the matching Phase 1 Architecture B weights,
tokenizer, model implementation, and saved configurations.

The submission ZIP is separate and contains only prediction.json at its root.
""",
    encoding="utf-8",
)

# ---------- 4. Collect artifacts ----------
# Include all JSON reports/configuration files and S2 checkpoint files.
S2B_REDO_CELL20_ARTIFACT_FILES = []

for path in sorted(S2B_REDO_CELL20_OUT.iterdir()):
    if not path.is_file():
        continue

    if path.suffix.lower() == ".json":
        S2B_REDO_CELL20_ARTIFACT_FILES.append(path)
    elif path.suffix.lower() == ".pt" and path.name.startswith("S2B"):
        S2B_REDO_CELL20_ARTIFACT_FILES.append(path)

S2B_REDO_CELL20_ARTIFACT_FILES.append(S2B_REDO_CELL20_README)

# Confirm the selected checkpoint is part of the artifact package.
selected_resolved = S2B_REDO_CELL20_SELECTED_CHECKPOINT.resolve()
artifact_resolved = {
    path.resolve()
    for path in S2B_REDO_CELL20_ARTIFACT_FILES
}

if selected_resolved not in artifact_resolved:
    raise RuntimeError(
        "Selected checkpoint was not included in the artifacts package."
    )

# ---------- 5. Write an artifact manifest ----------
def s2b_redo_cell20_sha256(path):
    hasher = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()


S2B_REDO_CELL20_ARTIFACT_MANIFEST = {
    "experiment": "S2 — Architecture B + LoRA",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "selected_validation_macro_f1": 0.216144,
    "validation_documents": 14,
    "labeled_documents_available": 70,
    "test_documents": 35,
    "test_predicted_spans": S2B_REDO_CELL20_SPAN_COUNT,
    "test_gold_labels_used": False,
    "checkpoint_note": (
        "Trainable-parameter checkpoints require the matching Phase 1 "
        "Architecture B base weights, tokenizer, model implementation, "
        "and configuration to reconstruct the model."
    ),
    "files": [
        {
            "name": path.name,
            "bytes": path.stat().st_size,
            "sha256": s2b_redo_cell20_sha256(path),
        }
        for path in S2B_REDO_CELL20_ARTIFACT_FILES
    ],
}

S2B_REDO_CELL20_MANIFEST_PATH = (
    S2B_REDO_CELL20_OUT / "S2B_artifact_manifest.json"
)

with open(
    S2B_REDO_CELL20_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        S2B_REDO_CELL20_ARTIFACT_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

# ---------- 6. Create the separate artifacts ZIP ----------
S2B_REDO_CELL20_ARTIFACTS_ZIP = (
    Path("/kaggle/working")
    / "S2_Architecture_B_Redo_Artifacts.zip"
)

if S2B_REDO_CELL20_ARTIFACTS_ZIP.exists():
    S2B_REDO_CELL20_ARTIFACTS_ZIP.unlink()

with zipfile.ZipFile(
    S2B_REDO_CELL20_ARTIFACTS_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in S2B_REDO_CELL20_ARTIFACT_FILES:
        archive.write(path, arcname=path.name)

    archive.write(
        S2B_REDO_CELL20_MANIFEST_PATH,
        arcname=S2B_REDO_CELL20_MANIFEST_PATH.name,
    )

# ---------- 7. Verify both ZIP files ----------
for zip_path in (
    S2B_REDO_CELL20_SUBMISSION_ZIP,
    S2B_REDO_CELL20_ARTIFACTS_ZIP,
):
    if not zip_path.is_file() or zip_path.stat().st_size == 0:
        raise RuntimeError(f"ZIP missing or empty: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_file = archive.testzip()
        if bad_file is not None:
            raise RuntimeError(
                f"ZIP integrity check failed for {zip_path.name}: {bad_file}"
            )

with zipfile.ZipFile(S2B_REDO_CELL20_ARTIFACTS_ZIP, "r") as archive:
    S2B_REDO_CELL20_ARTIFACT_NAMES = archive.namelist()

    if "prediction.json" not in S2B_REDO_CELL20_ARTIFACT_NAMES:
        raise RuntimeError("Artifacts ZIP is missing prediction.json.")

    if "S2B_LoRA_final_selected.pt" not in S2B_REDO_CELL20_ARTIFACT_NAMES:
        raise RuntimeError("Artifacts ZIP is missing the selected checkpoint.")

print("\n" + "=" * 76)
print("PACKAGING COMPLETE")
print("=" * 76)

print("\nSUBMISSION ZIP")
print("Path:", S2B_REDO_CELL20_SUBMISSION_ZIP)
print(f"Size: {S2B_REDO_CELL20_SUBMISSION_ZIP.stat().st_size / (1024**2):.2f} MiB")
print("Archive root entries: prediction.json")
print("Prediction equality check: PASS")
print("ZIP integrity check: PASS")

print("\nARTIFACTS ZIP")
print("Path:", S2B_REDO_CELL20_ARTIFACTS_ZIP)
print(f"Size: {S2B_REDO_CELL20_ARTIFACTS_ZIP.stat().st_size / (1024**2):.2f} MiB")
print("Included files:")

for name in S2B_REDO_CELL20_ARTIFACT_NAMES:
    print("  -", name)

print("\nFinal validation Macro-F1: 0.216144")
print("This is not a test-set accuracy estimate.")
print("Fold 3 gold labels used: NO")
print("\nCell 20 completed successfully.")

S2 REDO — FINAL PACKAGING


ValueError: Unexpected span count: 6498; expected 5316 from the verified decoding cell.